In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 15.64 GB


In [ ]:
!pip -q install transformers accelerate


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


In [ ]:
import torch

prompt = """Read this note carefully:
Project code: RG-2048
The project uses a blue database server.
The launch date is 15 December 2026.

Question: What is the project code? Answer briefly:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Model answer:", answer)

Model answer:  Project code: RG-2048. The answer is: RG-2048.


In [ ]:
import torch

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Create a longer sample document
long_text = """
Project Alpha has a budget of 450000 rupees.
Its lead engineer is Maya.
The project code is ALPHA-7319.
The launch date is 15 December 2026.
""" * 100

inputs = tokenizer(
    long_text,
    return_tensors="pt"
).to("cuda")

torch.cuda.reset_peak_memory_stats()

with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=True
    )

torch.cuda.synchronize()

print("Input tokens:", inputs["input_ids"].shape[1])
print(
    "Peak GPU memory (MB):",
    round(torch.cuda.max_memory_allocated() / 1024**2, 2)
)
print(
    "KV cache memory (MB):",
    round(sum(
        k.numel() * k.element_size() +
        v.numel() * v.element_size()
        for k, v in zip(
            outputs.past_key_values.key_cache,
            outputs.past_key_values.value_cache
        )
    ) / 1024**2, 4)
)

Input tokens: 4701
Peak GPU memory (MB): 3861.02


AttributeError: 'DynamicCache' object has no attribute 'key_cache'

In [ ]:
cache = outputs.past_key_values

print("Cache type:", type(cache).__name__)
print("Transformers version:")

import transformers
print(transformers.__version__)

print("Available cache attributes:")
print([x for x in dir(cache) if not x.startswith("__")])

if hasattr(cache, "layers"):
    print("Number of cache layers:", len(cache.layers))
    if len(cache.layers) > 0:
        print("First layer attributes:")
        print([x for x in dir(cache.layers[0])
               if not x.startswith("__")])

In [ ]:

cache = outputs.past_key_values

total_bytes = 0
layers_measured = 0

for layer in cache.layers:
    if layer.keys is not None and layer.values is not None:
        total_bytes += (
            layer.keys.numel() * layer.keys.element_size()
            + layer.values.numel() * layer.values.element_size()
        )
        layers_measured += 1

print("Cache type:", type(cache).__name__)
print("Layers measured:", layers_measured)
print("Input tokens:", inputs["input_ids"].shape[1])
print(f"KV-cache size: {total_bytes / (1024**2):.2f} MiB")
print(f"KV-cache size: {total_bytes / (1024**3):.4f} GiB")


In [ ]:

baseline = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "transformers_version": "5.18.0",
    "input_tokens": inputs["input_ids"].shape[1],
    "layers_measured": layers_measured,
    "kv_cache_bytes": total_bytes,
    "kv_cache_mib": total_bytes / (1024**2),
}

print(baseline)


In [ ]:

import inspect
import transformers

print("Transformers:", transformers.__version__)

try:
    from transformers import SinkCache
    print("SinkCache is available!")
    print("Constructor:", inspect.signature(SinkCache))
except ImportError as e:
    print("SinkCache is not directly available:", e)


In [ ]:

import transformers
import importlib.util

print("Transformers:", transformers.__version__)

for module_name in [
    "transformers.cache_utils",
    "transformers.generation",
    "transformers.models.qwen2.modeling_qwen2",
]:
    spec = importlib.util.find_spec(module_name)
    print(f"{module_name}:",
          "available" if spec is not None else "not found")

import transformers.cache_utils as cache_utils

print("\nCache classes containing 'Sink' or 'Sliding':")
print([
    name for name in dir(cache_utils)
    if "sink" in name.lower() or "sliding" in name.lower()
])


In [ ]:

import inspect
from transformers.cache_utils import SlidingWindowCache

print("SlidingWindowCache signature:")
print(inspect.signature(SlidingWindowCache))

print("\nDocumentation:")
print(inspect.getdoc(SlidingWindowCache))

print("\nQwen attention implementation:")
from transformers.models.qwen2 import modeling_qwen2

print("Module:", modeling_qwen2.__file__)
print("Has eager attention:", hasattr(modeling_qwen2, "eager_attention_forward"))


In [ ]:

import torch

# Use the existing full-context cache.
cache = outputs.past_key_values
original_bytes = baseline["kv_cache_bytes"]

SINK_TOKENS = 4
RECENT_TOKENS = 256
kept_tokens = SINK_TOKENS + RECENT_TOKENS

# Verify the cache has the expected sequence length before changing it.
assert cache.layers[0].keys.shape[-2] == baseline["input_tokens"], (
    "Cache length doesn't match the saved baseline. Stop here."
)

# Prune each layer: keep initial tokens + recent tokens.
for layer in cache.layers:
    if layer.keys is None or layer.values is None:
        continue

    assert layer.keys.shape[-2] > kept_tokens

    layer.keys = torch.cat(
        [
            layer.keys[..., :SINK_TOKENS, :],
            layer.keys[..., -RECENT_TOKENS:, :]
        ],
        dim=-2
    )

    layer.values = torch.cat(
        [
            layer.values[..., :SINK_TOKENS, :],
            layer.values[..., -RECENT_TOKENS:, :]
        ],
        dim=-2
    )

# Measure actual stored tensor bytes after pruning.
compressed_bytes = sum(
    layer.keys.numel() * layer.keys.element_size()
    + layer.values.numel() * layer.values.element_size()
    for layer in cache.layers
    if layer.keys is not None and layer.values is not None
)

reduction = (1 - compressed_bytes / original_bytes) * 100

print("Original input tokens:", baseline["input_tokens"])
print("Retained cache tokens per layer:", kept_tokens)
print(f"Original KV cache: {original_bytes / (1024**2):.2f} MiB")
print(f"Pruned KV cache: {compressed_bytes / (1024**2):.2f} MiB")
print(f"Cache storage reduction: {reduction:.2f}%")


In [ ]:

import torch

# Synthetic document: facts at different positions.
begin_fact = "The secret project code is ORCHID-7319."
middle_fact = "The laboratory access code is COBALT-4826."
end_fact = "The final delivery code is MAPLE-9054."

filler = (
    "This is background information about routine project operations, "
    "documentation, planning, and general administrative procedures. "
)

document = (
    begin_fact + "\n"
    + filler * 350
    + middle_fact + "\n"
    + filler * 350
    + end_fact + "\n"
    + "Answer questions using only the document."
)

recall_inputs = tokenizer(document, return_tensors="pt").to(model.device)

with torch.no_grad():
    recall_outputs = model(**recall_inputs, use_cache=True)

recall_cache = recall_outputs.past_key_values

print("Document tokens:", recall_inputs["input_ids"].shape[1])
print("Cache layers:", len(recall_cache.layers))
print("Expected facts: 3")
print("Cache recreated: yes")


In [ ]:

import torch

facts = {
    "beginning": "ORCHID-7319",
    "middle": "COBALT-4826",
    "end": "MAPLE-9054",
}

filler = (
    "Routine project notes describe documentation, planning, "
    "administration, and standard operational procedures. "
)

document = (
    f"The project code is {facts['beginning']}.\n"
    + filler * 80
    + f"\nThe laboratory code is {facts['middle']}.\n"
    + filler * 80
    + f"\nThe delivery code is {facts['end']}.\n"
)

questions = [
    ("What is the project code?", facts["beginning"]),
    ("What is the laboratory code?", facts["middle"]),
    ("What is the delivery code?", facts["end"]),
]

correct = 0

for question, expected in questions:
    prompt = (
        f"Document:\n{document}\n\n"
        f"Question: {question}\n"
        "Answer with only the code.\nAnswer:"
    )

    batch = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        result = model.generate(
            **batch,
            max_new_tokens=20,
            do_sample=False,
            use_cache=False
        )

    answer = tokenizer.decode(
        result[0][batch["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    passed = expected.lower() in answer.lower()
    correct += int(passed)

    print(f"Question: {question}")
    print(f"Expected: {expected}")
    print(f"Model answer: {answer}")
    print(f"Correct: {passed}\n")

print(f"Baseline recall: {correct}/3 ({correct / 3 * 100:.1f}%)")


In [ ]:
print("Expected questions: 3")
print("If only one result appeared, rerun the recall cell.")

In [ ]:

%pip -q install streaming-llm


In [ ]:

import gc
import torch

# Use a short, controlled document to limit GPU memory.
facts = {
    "beginning": "ORCHID-7319",
    "middle": "COBALT-4826",
    "end": "MAPLE-9054",
}

filler = (
    "Routine project documentation contains planning, operations, "
    "administration, and standard procedures. "
)

document = (
    f"The project code is {facts['beginning']}.\n"
    + filler * 20
    + f"\nThe laboratory code is {facts['middle']}.\n"
    + filler * 20
    + f"\nThe delivery code is {facts['end']}.\n"
)

questions = [
    ("What is the project code?", facts["beginning"]),
    ("What is the laboratory code?", facts["middle"]),
    ("What is the delivery code?", facts["end"]),
]

def ask_model(prompt):
    batch = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        result = model.generate(
            **batch,
            max_new_tokens=12,
            do_sample=False,
            use_cache=False
        )
    return tokenizer.decode(
        result[0][batch["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

print("NORMAL-CONTEXT RECALL TEST")
normal_correct = 0

for question, expected in questions:
    prompt = (
        f"Document:\n{document}\n\n"
        f"Question: {question}\nAnswer with only the code:"
    )
    answer = ask_model(prompt)
    passed = expected in answer
    normal_correct += int(passed)
    print(f"{question} | Expected: {expected} | "
          f"Answer: {answer} | Correct: {passed}")

print(f"\nNormal-context recall: {normal_correct}/3")

print("\nNOTE: This cell does not test pruned-cache recall.")
print("It establishes a small, memory-conscious recall baseline.")
print("We must not report this as a compression-quality result.")


In [ ]:

!git clone https://github.com/mit-han-lab/streaming-llm.git
%cd streaming-llm
!ls


In [ ]:
!python --version
!python -c "import torch, transformers; print('PyTorch:', torch.__version__); print('Transformers:', transformers.__version__)"
!sed -n '1,220p' setup.py

In [ ]:
%cd /content/streaming-llm

!sed -n '1,260p' streaming_llm/kv_cache.py
!sed -n '1,220p' streaming_llm/enable_streaming_llm.py

In [ ]:

%cd /content/streaming-llm

import torch
from streaming_llm.kv_cache import StartRecentKVCache

# Create a simulated KV cache for one layer.
# Shape: [batch, heads, sequence_length, head_dimension]
keys = torch.randn(1, 2, 100, 4)
values = torch.randn(1, 2, 100, 4)
cache = [(keys, values)]

# Keep the first 4 tokens and the latest 16 tokens.
compress = StartRecentKVCache(start_size=4, recent_size=16)
compressed = compress(cache)

original_tokens = keys.shape[2]
retained_tokens = compressed[0][0].shape[2]

# Count bytes in both key and value tensors.
original_bytes = (
    keys.numel() * keys.element_size()
    + values.numel() * values.element_size()
)
compressed_keys, compressed_values = compressed[0]
compressed_bytes = (
    compressed_keys.numel() * compressed_keys.element_size()
    + compressed_values.numel() * compressed_values.element_size()
)

reduction = (1 - compressed_bytes / original_bytes) * 100

print("Original tokens:", original_tokens)
print("Retained tokens:", retained_tokens)
print("Original KV storage:", original_bytes, "bytes")
print("Compressed KV storage:", compressed_bytes, "bytes")
print("Storage reduction:", round(reduction, 2), "%")

assert original_tokens == 100
assert retained_tokens == 20
assert original_bytes == 6400
assert compressed_bytes == 1280
assert abs(reduction - 80.0) < 1e-6

print("PASS: Cache compression test succeeded.")


In [ ]:

import torch
import transformers

print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("StreamingLLM cache class:", StartRecentKVCache.__name__)

# Check that the compressor preserves the beginning and end.
test_keys = torch.arange(100).reshape(1, 1, 100, 1)
test_values = test_keys.clone()

test_cache = [(test_keys, test_values)]
result = StartRecentKVCache(start_size=4, recent_size=16)(test_cache)

retained = result[0][0].flatten().tolist()

print("First retained tokens:", retained[:4])
print("Last retained tokens:", retained[-4:])
print("Retained count:", len(retained))

assert retained == list(range(4)) + list(range(84, 100))
print("PASS: Correct beginning and ending positions retained.")


In [ ]:

# Check existing variables without loading another model.
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())

if "model" in globals():
    print("Model type:", model.config.model_type)
    print("Model device:", next(model.parameters()).device)
    print("Model dtype:", next(model.parameters()).dtype)


In [ ]:

import torch

prompt = "Remember this code: ORCHID-7319. Reply briefly."
test_inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(next(model.parameters()).device)

with torch.inference_mode():
    result = model(
        **test_inputs,
        use_cache=True
    )

kv = result.past_key_values

print("Model:", model.config.model_type)
print("Cache class:", type(kv).__name__)
print("Number of layers:", len(kv.layers) if hasattr(kv, "layers") else len(kv))
print("Input tokens:", test_inputs["input_ids"].shape[1])

if hasattr(kv, "layers"):
    layer = kv.layers[0]
    print("Layer type:", type(layer).__name__)
    print("Key shape:", tuple(layer.keys.shape) if layer.keys is not None else None)
    print("Value shape:", tuple(layer.values.shape) if layer.values is not None else None)
else:
    print("Key shape:", tuple(kv[0][0].shape))
    print("Value shape:", tuple(kv[0][1].shape))

del result, kv, test_inputs
torch.cuda.empty_cache()

print("Inspection complete.")


In [ ]:

import torch
from transformers import DynamicCache

prompt = (
    "Remember these codes: "
    "project ORCHID-7319; lab COBALT-4826; "
    "delivery MAPLE-9054. Reply briefly."
)

inputs_test = tokenizer(prompt, return_tensors="pt").to("cuda:0")

with torch.inference_mode():
    out_test = model(**inputs_test, use_cache=True)

cache = out_test.past_key_values

def cache_bytes(c):
    total = 0
    for layer in c.layers:
        for tensor in (layer.keys, layer.values):
            if tensor is not None:
                total += tensor.numel() * tensor.element_size()
    return total

before_bytes = cache_bytes(cache)
original_tokens = cache.layers[0].keys.shape[2]

# Work on a copy so the original model output remains intact.
compressed_cache = DynamicCache()
compressed_cache.layers = [
    layer for layer in cache.layers
]

# Clone and compress each layer independently.
for layer in compressed_cache.layers:
    if layer.keys is not None and layer.values is not None:
        layer.keys = torch.cat(
            [layer.keys[:, :, :4, :], layer.keys[:, :, -16:, :]], dim=2
        ).contiguous()
        layer.values = torch.cat(
            [layer.values[:, :, :4, :], layer.values[:, :, -16:, :]], dim=2
        ).contiguous()

after_bytes = cache_bytes(compressed_cache)

print("Original cache tokens:", original_tokens)
print("Retained tokens per layer:", compressed_cache.layers[0].keys.shape[2])
print("Original KV storage (MiB):", round(before_bytes / 1024**2, 4))
print("Compressed KV storage (MiB):", round(after_bytes / 1024**2, 4))
print("Storage reduction (%):", round((1 - after_bytes / before_bytes) * 100, 2))

# Release temporary objects.
del out_test, cache, compressed_cache, inputs_test
torch.cuda.empty_cache()


In [ ]:

import torch
from transformers import DynamicCache

device = next(model.parameters()).device

def run_recall_test(compress_cache=False):
    prompt = (
        "Remember this project code: ORCHID-7319. "
        "Answer the question with only the project code. "
        "Project code:"
    )
    batch = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        out = model(**batch, use_cache=True)

    cache = out.past_key_values
    original_length = cache.layers[0].keys.shape[2]

    if compress_cache:
        # Build a separate cache; don't mutate the model's original cache.
        compressed = DynamicCache()
        for layer in cache.layers:
            new_layer = type(layer)()
            new_layer.keys = torch.cat(
                [layer.keys[:, :, :4, :], layer.keys[:, :, -16:, :]], dim=2
            ).contiguous()
            new_layer.values = torch.cat(
                [layer.values[:, :, :4, :], layer.values[:, :, -16:, :]], dim=2
            ).contiguous()
            compressed.layers.append(new_layer)
        cache = compressed

    print(
        "Mode:", "compressed" if compress_cache else "normal",
        "| Original tokens:", original_length,
        "| Cache tokens:", cache.layers[0].keys.shape[2]
    )

    # This is a compatibility probe, not yet a validated recall score.
    # The cache is compressed, but Qwen position and attention-mask handling
    # must be verified before generation results can be trusted.
    del out, batch, cache
    torch.cuda.empty_cache()

run_recall_test(False)
run_recall_test(True)
print("Cache preparation probe completed.")


In [ ]:

import copy
import time
import torch
from transformers import DynamicCache

device = next(model.parameters()).device

# Context contains a fact; the question is added AFTER compression.
context = (
    "Project code: ORCHID-7319. "
    "Lab code: COBALT-4826. "
    "Delivery code: MAPLE-9054. "
    "These codes must be recalled exactly."
)
question = " What is the project code? Answer with only the code."

def cache_size_bytes(cache):
    total = 0
    for layer in cache.layers:
        for tensor in (layer.keys, layer.values):
            if tensor is not None:
                total += tensor.numel() * tensor.element_size()
    return total

def compress_dynamic_cache(cache, start_size=4, recent_size=16):
    # Copy the cache so the original is not modified.
    result = copy.deepcopy(cache)
    for layer in result.layers:
        if layer.keys is not None and layer.values is not None:
            layer.keys = torch.cat(
                [layer.keys[:, :, :start_size, :],
                 layer.keys[:, :, -recent_size:, :]], dim=2
            ).contiguous()
            layer.values = torch.cat(
                [layer.values[:, :, :start_size, :],
                 layer.values[:, :, -recent_size:, :]], dim=2
            ).contiguous()
    return result

def run_case(compressed=False):
    context_inputs = tokenizer(context, return_tensors="pt").to(device)
    question_inputs = tokenizer(
        question, add_special_tokens=False, return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        prefill = model(**context_inputs, use_cache=True)
        cache = prefill.past_key_values
        original_len = cache.layers[0].keys.shape[2]
        original_bytes = cache_size_bytes(cache)

        if compressed:
            cache = compress_dynamic_cache(cache)

        retained_len = cache.layers[0].keys.shape[2]
        cache_bytes = cache_size_bytes(cache)

        # Keep the original absolute positions for RoPE, while cache_position
        # indexes the compacted cache. This is an experimental compatibility
        # probe; the result must not be treated as validated StreamingLLM.
        q_len = question_inputs["input_ids"].shape[1]
        attention_mask = torch.ones(
            (1, retained_len + q_len), device=device, dtype=torch.long
        )
        position_ids = torch.arange(
            original_len, original_len + q_len, device=device
        ).unsqueeze(0)
        cache_position = torch.arange(
            retained_len, retained_len + q_len, device=device
        )

        start = time.perf_counter()
        result = model(
            input_ids=question_inputs["input_ids"],
            attention_mask=attention_mask,
            past_key_values=cache,
            use_cache=True,
            position_ids=position_ids,
            cache_position=cache_position,
        )
        next_token = result.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        elapsed_ms = (time.perf_counter() - start) * 1000

        answer = tokenizer.decode(next_token[0], skip_special_tokens=True).strip()

    del context_inputs, question_inputs, prefill, cache, result
    torch.cuda.empty_cache()

    return {
        "mode": "compressed" if compressed else "normal",
        "original_tokens": original_len,
        "retained_tokens": retained_len,
        "cache_mib": round(cache_bytes / 1024**2, 4),
        "first_generated_token": answer,
        "step_latency_ms": round(elapsed_ms, 2),
    }

normal = run_case(False)
compressed = run_case(True)

print("NORMAL:", normal)
print("COMPRESSED:", compressed)
print("Storage reduction (%):", round(
    (1 - compressed["cache_mib"] / normal["cache_mib"]) * 100, 2
))
print("\nNOTE: This is a one-token compatibility probe, not a validated recall benchmark.")


In [ ]:

import copy
import torch

device = next(model.parameters()).device
START_SIZE = 16
RECENT_SIZE = 16
MAX_NEW_TOKENS = 12

# Put the facts far enough apart that compression removes middle content.
filler = "The project documentation contains routine technical details. " * 35

context = (
    "Project code is ORCHID-7319. "
    + filler
    + "Laboratory code is COBALT-4826. "
    + filler
    + "Delivery code is MAPLE-9054."
)

questions = [
    ("What is the project code?", "ORCHID-7319"),
    ("What is the laboratory code?", "COBALT-4826"),
    ("What is the delivery code?", "MAPLE-9054"),
]

def cache_size(cache):
    return sum(
        t.numel() * t.element_size()
        for layer in cache.layers
        for t in (layer.keys, layer.values)
        if t is not None
    )

def compress_cache(cache):
    result = copy.deepcopy(cache)
    for layer in result.layers:
        if layer.keys is not None and layer.values is not None:
            layer.keys = torch.cat([
                layer.keys[:, :, :START_SIZE, :],
                layer.keys[:, :, -RECENT_SIZE:, :]
            ], dim=2).contiguous()
            layer.values = torch.cat([
                layer.values[:, :, :START_SIZE, :],
                layer.values[:, :, -RECENT_SIZE:, :]
            ], dim=2).contiguous()
    return result

def answer_from_cache(context, question, compressed):
    context_ids = tokenizer(
        context, return_tensors="pt"
    ).input_ids.to(device)

    question_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.inference_mode():
        prefill = model(input_ids=context_ids, use_cache=True)

    original_cache = prefill.past_key_values
    original_len = original_cache.layers[0].keys.shape[2]
    original_bytes = cache_size(original_cache)

    cache = compress_cache(original_cache) if compressed else original_cache
    retained_len = cache.layers[0].keys.shape[2]

    # Append the question to the actual (possibly compressed) cache.
    q_len = question_ids.shape[1]
    attention = torch.ones(
        (1, retained_len + q_len), dtype=torch.long, device=device
    )
    positions = torch.arange(
        original_len, original_len + q_len, device=device
    ).unsqueeze(0)
    cache_positions = torch.arange(
        retained_len, retained_len + q_len, device=device
    )

    with torch.inference_mode():
        out = model(
            input_ids=question_ids,
            past_key_values=cache,
            attention_mask=attention,
            position_ids=positions,
            cache_position=cache_positions,
            use_cache=True,
        )

    generated = []
    next_logits = out.logits[:, -1, :]
    cache = out.past_key_values
    eos = tokenizer.eos_token_id

    # Greedy generation using the returned cache on every step.
    for step in range(MAX_NEW_TOKENS):
        token = next_logits.argmax(dim=-1, keepdim=True)
        token_id = token.item()
        if eos is not None and token_id == eos:
            break
        generated.append(token_id)

        if step == MAX_NEW_TOKENS - 1:
            break

        past_len = cache.layers[0].keys.shape[2]
        attention = torch.ones(
            (1, past_len + 1), dtype=torch.long, device=device
        )
        position = torch.tensor(
            [[original_len + q_len + step]],
            dtype=torch.long, device=device
        )
        cache_position = torch.tensor(
            [past_len], dtype=torch.long, device=device
        )

        with torch.inference_mode():
            out = model(
                input_ids=token,
                past_key_values=cache,
                attention_mask=attention,
                position_ids=position,
                cache_position=cache_position,
                use_cache=True,
            )
        cache = out.past_key_values
        next_logits = out.logits[:, -1, :]

    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    del prefill, out, cache, original_cache
    torch.cuda.empty_cache()

    return {
        "answer": answer,
        "correct": questions_expected(question, answer),
        "original_tokens": original_len,
        "retained_tokens": retained_len,
        "original_cache_mib": round(original_bytes / 1024**2, 4),
    }

def questions_expected(question, answer):
    expected = dict(questions)[question]
    return expected in answer

results = []
for question, expected in questions:
    normal = answer_from_cache(context, question, compressed=False)
    compact = answer_from_cache(context, question, compressed=True)

    results.append((question, expected, normal, compact))
    print("\nQUESTION:", question, "| EXPECTED:", expected)
    print("NORMAL:", normal)
    print("COMPRESSED:", compact)

print("\n=== RECALL SUMMARY ===")
print("Normal correct:", sum(r[2]["correct"] for r in results), "/ 3")
print("Compressed correct:", sum(r[3]["correct"] for r in results), "/ 3")
print("Compression keeps first", START_SIZE, "and last", RECENT_SIZE, "tokens.")
print("Interpret results cautiously: this is a prototype cache/position experiment,")
print("not yet a validated general-purpose StreamingLLM implementation.")


In [ ]:

import json
import csv
from datetime import datetime, timezone
from google.colab import files

# Preserve the results from the experiment you just ran.
records = []

for question, expected, normal, compressed in results:
    records.append({
        "question": question,
        "expected_answer": expected,
        "normal_answer": normal["answer"],
        "normal_correct": normal["correct"],
        "compressed_answer": compressed["answer"],
        "compressed_correct": compressed["correct"],
        "original_tokens": normal["original_tokens"],
        "compressed_tokens": compressed["retained_tokens"],
        "original_cache_mib": normal["original_cache_mib"],
        "compressed_cache_mib": None,  # Not measured directly yet
        "status": "preliminary",
    })

report = {
    "project": "RecallGuard-KV",
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "method": "Keep first 16 and last 16 cache positions",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "normal_recall": f'{sum(r["normal_correct"] for r in records)}/{len(records)}',
    "compressed_recall": f'{sum(r["compressed_correct"] for r in records)}/{len(records)}',
    "limitations": [
        "Only three synthetic recall questions",
        "Compressed cache storage not directly recorded in this recall run",
        "Cache position handling requires further validation",
        "Results are preliminary, not a general benchmark"
    ],
    "results": records,
}

json_path = "/content/recallguard_kv_results.json"
csv_path = "/content/recallguard_kv_results.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print("Saved JSON:", json_path)
print("Saved CSV:", csv_path)
print("Normal recall:", report["normal_recall"])
print("Compressed recall:", report["compressed_recall"])

files.download(json_path)
files.download(csv_path)



In [ ]:

# RecallGuard-KV — Experiment 2 setup
# Target: keep first 16 + most recent 32 cache positions

START_SIZE = 16
RECENT_SIZE = 32
EXPERIMENT_NAME = "recallguard_kv_exp02_recent32"

# Confirm the existing notebook state without running another test.
print("Experiment:", EXPERIMENT_NAME)
print("START_SIZE:", START_SIZE)
print("RECENT_SIZE:", RECENT_SIZE)
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Previous results available:", "results" in globals())

if "results" in globals():
    print("Previous result count:", len(results))

# Find existing recall-test functions so we can reuse the
# original context and decoding implementation safely.
print("\nExisting recall-related functions:")
for name, obj in globals().items():
    if callable(obj) and any(
        word in name.lower()
        for word in ("recall", "decode", "compress", "cache", "answer", "test")
    ):
        if getattr(obj, "__module__", None) != "builtins":
            print("-", name)


In [ ]:

# RecallGuard-KV — Experiment 2 setup (corrected)
START_SIZE = 16
RECENT_SIZE = 32
EXPERIMENT_NAME = "recallguard_kv_exp02_recent32"

print("Experiment:", EXPERIMENT_NAME)
print("START_SIZE:", START_SIZE)
print("RECENT_SIZE:", RECENT_SIZE)
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Previous results available:", "results" in globals())

if "results" in globals():
    print("Previous result count:", len(results))

print("\nExisting recall-related functions:")
for name, obj in list(globals().items()):  # snapshot prevents dictionary-size error
    if callable(obj) and any(
        word in name.lower()
        for word in ("recall", "decode", "compress", "cache", "answer", "test")
    ):
        if getattr(obj, "__module__", None) != "builtins":
            print("-", name)


In [ ]:

import inspect

print("=== Recall test function ===")
print(inspect.signature(run_recall_test))
print(inspect.getsource(run_recall_test))

print("\n=== Cache compression functions ===")
for fn in (compress, compress_dynamic_cache, compress_cache):
    try:
        print(f"\n--- {fn.__name__}{inspect.signature(fn)} ---")
        print(inspect.getsource(fn))
    except (TypeError, OSError) as e:
        print("Source unavailable:", e)


In [ ]:

import inspect

for name in ["run_recall_test", "compress_dynamic_cache",
              "compress_cache", "answer_from_cache",
              "cache_size_bytes"]:
    obj = globals().get(name)

    print(f"\n--- {name} ---")
    if obj is None:
        print("Not found")
    elif callable(obj):
        try:
            print("Signature:", inspect.signature(obj))
            print(inspect.getsource(obj))
        except (TypeError, OSError) as e:
            print("Source unavailable:", e)
    else:
        print("Type:", type(obj).__name__)
        print(repr(obj)[:500])



In [ ]:

# RecallGuard-KV — Experiment 2
# Keep first 16 + most recent 32 positions.
# Does NOT overwrite the original `results` variable.

import json
import csv
import os
from datetime import datetime, timezone
from google.colab import files

START_SIZE = 16
RECENT_SIZE = 32
MAX_NEW_TOKENS = 12

assert "model" in globals() and "tokenizer" in globals()
assert "answer_from_cache" in globals()
assert "results" in globals() and len(results) == 3
assert "context" in globals(), (
    "The original context variable was not found. Stop here; "
    "do not recreate a different test context."
)

# Preserve the original questions and expected answers.
exp2_results = []

for question, expected, _, _ in results:
    print("\nQuestion:", question)
    print("Expected:", expected)

    normal = answer_from_cache(context, question, compressed=False)
    compressed = answer_from_cache(context, question, compressed=True)

    # Check against the known expected string directly as well.
    normal_ok = str(expected).lower() in normal["answer"].lower()
    compressed_ok = str(expected).lower() in compressed["answer"].lower()

    record = {
        "question": question,
        "expected_answer": expected,
        "normal_answer": normal["answer"],
        "normal_correct": normal_ok,
        "compressed_answer": compressed["answer"],
        "compressed_correct": compressed_ok,
        "original_tokens": compressed["original_tokens"],
        "retained_tokens": compressed["retained_tokens"],
        "original_cache_mib": compressed["original_cache_mib"],
        "compressed_cache_mib": None,
        "start_size": START_SIZE,
        "recent_size": RECENT_SIZE,
        "status": "preliminary; cache positions not fully validated",
    }
    exp2_results.append(record)

    print("Normal:", normal["answer"], "| Correct:", normal_ok)
    print("Compressed:", compressed["answer"], "| Correct:", compressed_ok)
    print("Tokens:", record["original_tokens"], "->",
          record["retained_tokens"])

# Save to new filenames, never the Experiment 1 filenames.
report = {
    "project": "RecallGuard-KV",
    "experiment": "exp02_recent32",
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "start_size": START_SIZE,
    "recent_size": RECENT_SIZE,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "normal_recall": (
        f'{sum(r["normal_correct"] for r in exp2_results)}/'
        f'{len(exp2_results)}'
    ),
    "compressed_recall": (
        f'{sum(r["compressed_correct"] for r in exp2_results)}/'
        f'{len(exp2_results)}'
    ),
    "limitations": [
        "Only three synthetic recall questions",
        "Compressed cache memory was not directly measured in this run",
        "Qwen cache position and attention handling need validation",
        "Results are preliminary, not a general benchmark",
    ],
    "results": exp2_results,
}

json_path = "/content/recallguard_kv_exp02_recent32.json"
csv_path = "/content/recallguard_kv_exp02_recent32.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=exp2_results[0].keys())
    writer.writeheader()
    writer.writerows(exp2_results)

print("\n=== EXPERIMENT 2 SUMMARY ===")
print("Normal recall:", report["normal_recall"])
print("Compressed recall:", report["compressed_recall"])
print("Saved:", json_path)
print("Saved:", csv_path)

files.download(json_path)
files.download(csv_path)


In [ ]:

# RecallGuard-KV — static compression sanity check
# No model inference; no GPU usage.

import inspect

print("Compression function:")
print(inspect.getsource(compress_dynamic_cache))

print("\nMemory accounting function:")
print(inspect.getsource(cache_size_bytes))

print("\nExpected retention configurations:")
for start_size, recent_size in [(4, 16), (16, 32)]:
    print(
        f"start={start_size}, recent={recent_size}, "
        f"expected retained positions={start_size + recent_size}"
    )


In [ ]:

# RecallGuard-KV: static validation — no model inference
import inspect
import torch
import transformers

print("=== 1. SLICE / OVERLAP CHECK ===")

def retained_indices(length, start_size, recent_size):
    # Match the current implementation's slicing behavior.
    return list(range(length))[:start_size] + list(range(length))[-recent_size:]

for length, start, recent in [(100, 4, 16), (100, 16, 32),
                              (40, 16, 32), (48, 16, 32)]:
    indices = retained_indices(length, start, recent)
    duplicates = len(indices) - len(set(indices))
    print(
        f"length={length}, start={start}, recent={recent}: "
        f"retained={len(indices)}, unique={len(set(indices))}, "
        f"duplicate positions={duplicates}"
    )

print("\n=== 2. CACHE POSITION HANDLING ===")
src = inspect.getsource(answer_from_cache)
for i, line in enumerate(src.splitlines(), 1):
    if any(k in line for k in [
        "attention", "position", "cache_position", "retained_len",
        "original_len", "past_len", "model("
    ]):
        print(f"{i:03}: {line}")

print("\n=== 3. QWEN / TRANSFORMERS CONFIG ===")
print("Transformers version:", transformers.__version__)
print("Model type:", model.config.model_type)
print("Model class:", type(model).__name__)
print("Attention implementation:",
      getattr(model.config, "_attn_implementation", "not exposed"))
print("Cache class:", type(model.model.layers[0]).__name__)

print("\n=== STATIC CHECK COMPLETE ===")
print("No inference was run. Position/attention correctness is not proven.")


In [ ]:

import copy
import torch

def compress_dynamic_cache(cache, start_size=4, recent_size=16):
    if start_size < 0 or recent_size < 0:
        raise ValueError("Cache sizes must be non-negative.")

    result = copy.deepcopy(cache)

    for layer in result.layers:
        if layer.keys is None or layer.values is None:
            continue

        length = layer.keys.shape[2]

        # If the requested slices would overlap, retain the
        # entire short sequence rather than duplicate positions.
        if length <= start_size + recent_size:
            continue

        key_prefix = layer.keys[:, :, :start_size, :]
        key_recent = layer.keys[:, :, -recent_size:, :] if recent_size else layer.keys[:, :, :0, :]
        value_prefix = layer.values[:, :, :start_size, :]
        value_recent = layer.values[:, :, -recent_size:, :] if recent_size else layer.values[:, :, :0, :]

        layer.keys = torch.cat(
            [key_prefix, key_recent], dim=2
        ).contiguous()
        layer.values = torch.cat(
            [value_prefix, value_recent], dim=2
        ).contiguous()

    return result

print("Updated compression function.")
print("No model inference performed.")


In [ ]:

import inspect
import transformers
from transformers.models.qwen2.modeling_qwen2 import (
    Qwen2Model,
    Qwen2Attention,
)

print("Transformers:", transformers.__version__)

print("\n=== Qwen2Model.forward ===")
print(inspect.getsource(Qwen2Model.forward))

print("\n=== Qwen2Attention.forward ===")
print(inspect.getsource(Qwen2Attention.forward))


In [ ]:

def compress_cache(cache):
    result = copy.deepcopy(cache)

    for layer in result.layers:
        if layer.keys is None or layer.values is None:
            continue

        length = layer.keys.shape[2]
        keep = START_SIZE + RECENT_SIZE

        # Don't compress if the cache is already short enough
        if length <= keep:
            continue

        # Keep the initial tokens and the most recent tokens
        prefix_k = layer.keys[:, :, :START_SIZE, :]
        recent_k = (
            layer.keys[:, :, -RECENT_SIZE:, :]
            if RECENT_SIZE > 0
            else layer.keys[:, :, :0, :]
        )

        prefix_v = layer.values[:, :, :START_SIZE, :]
        recent_v = (
            layer.values[:, :, -RECENT_SIZE:, :]
            if RECENT_SIZE > 0
            else layer.values[:, :, :0, :]
        )

        layer.keys = torch.cat([prefix_k, recent_k], dim=2).contiguous()
        layer.values = torch.cat([prefix_v, recent_v], dim=2).contiguous()

    return result


In [ ]:
position_ids = torch.arange(original_len, original_len + q_len, ...)
cache_position = torch.arange(retained_len, retained_len + q_len, ...)

In [ ]:
import inspect

print(inspect.getsource(answer_from_cache))


In [ ]:

def answer_from_cache(context, question, compressed):
    context_ids = tokenizer(
        context, return_tensors="pt"
    ).input_ids.to(device)

    question_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.inference_mode():
        prefill = model(input_ids=context_ids, use_cache=True)

    original_cache = prefill.past_key_values
    original_len = original_cache.layers[0].keys.shape[2]
    original_bytes = cache_size(original_cache)

    cache = compress_cache(original_cache) if compressed else original_cache
    retained_len = cache.layers[0].keys.shape[2]

    q_len = question_ids.shape[1]

    attention = torch.ones(
        (1, retained_len + q_len),
        dtype=torch.long,
        device=device
    )

    # Preserve the question's original absolute token positions.
    positions = torch.arange(
        original_len,
        original_len + q_len,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # Cache positions refer to the physical cache length.
    cache_positions = torch.arange(
        retained_len,
        retained_len + q_len,
        dtype=torch.long,
        device=device
    )

    with torch.inference_mode():
        out = model(
            input_ids=question_ids,
            past_key_values=cache,
            attention_mask=attention,
            position_ids=positions,
            cache_position=cache_positions,
            use_cache=True,
        )

    generated = []
    next_logits = out.logits[:, -1, :]
    cache = out.past_key_values
    eos = tokenizer.eos_token_id

    for step in range(MAX_NEW_TOKENS):
        token = next_logits.argmax(dim=-1, keepdim=True)
        token_id = token.item()

        if eos is not None and token_id == eos:
            break

        generated.append(token_id)

        if step == MAX_NEW_TOKENS - 1:
            break

        past_len = cache.layers[0].keys.shape[2]

        attention = torch.ones(
            (1, past_len + 1),
            dtype=torch.long,
            device=device
        )

        position = torch.tensor(
            [[original_len + q_len + step]],
            dtype=torch.long,
            device=device
        )

        cache_position = torch.tensor(
            [past_len],
            dtype=torch.long,
            device=device
        )

        with torch.inference_mode():
            out = model(
                input_ids=token,
                past_key_values=cache,
                attention_mask=attention,
                position_ids=position,
                cache_position=cache_position,
                use_cache=True,
            )

        cache = out.past_key_values
        next_logits = out.logits[:, -1, :]

    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    result = {
        "answer": answer,
        "correct": questions_expected(question, answer),
        "original_tokens": original_len,
        "retained_tokens": retained_len,
        "original_cache_mib": round(
            original_bytes / 1024**2, 4
        ),
    }

    del prefill, out, cache, original_cache
    torch.cuda.empty_cache()

    return result


In [ ]:

import torch

# Test whether prefix + recent selection overlaps
START_SIZE = 16
RECENT_SIZE = 32
length = 40

dummy = torch.arange(length).reshape(1, 1, length, 1)

if length <= START_SIZE + RECENT_SIZE:
    compressed = dummy
else:
    compressed = torch.cat([
        dummy[:, :, :START_SIZE, :],
        dummy[:, :, -RECENT_SIZE:, :]
    ], dim=2)

kept = compressed.flatten().tolist()

print("Original length:", length)
print("Retained length:", len(kept))
print("Unique positions:", len(set(kept)))
print("Duplicates:", len(kept) != len(set(kept)))


In [ ]:
# Check whether compression preserves the original cache
# and retains the expected number of positions.

assert "original_cache" in globals(), (
    "original_cache isn't available. Run your existing small prefill "
    "test first, or use the cache-integrity check after creating a cache."
)

before = [
    (layer.keys.clone(), layer.values.clone())
    for layer in original_cache.layers
]

compressed_cache = compress_cache(original_cache)

# Original cache must remain unchanged.
for i, layer in enumerate(original_cache.layers):
    assert torch.equal(layer.keys, before[i][0]), (
        f"Original keys changed in layer {i}"
    )
    assert torch.equal(layer.values, before[i][1]), (
        f"Original values changed in layer {i}"
    )

# All layers should retain the same number of positions.
lengths = [
    layer.keys.shape[2]
    for layer in compressed_cache.layers
    if layer.keys is not None
]

print("Original cache unchanged: PASS")
print("Compressed layer lengths:", sorted(set(lengths)))
print("Number of layers checked:", len(lengths))
print("Cache integrity: PASS")

In [ ]:

import torch
from types import SimpleNamespace

# Save your actual compression settings
print("START_SIZE:", START_SIZE)
print("RECENT_SIZE:", RECENT_SIZE)

# Create a tiny synthetic cache with two layers
dummy_cache = SimpleNamespace(layers=[])

for _ in range(2):
    dummy_cache.layers.append(SimpleNamespace(
        keys=torch.arange(100, dtype=torch.float32).reshape(1, 1, 10, 1),
        values=torch.arange(100, 200, dtype=torch.float32).reshape(1, 1, 10, 1)
    ))

# Snapshot the original tensors
before = [
    (layer.keys.clone(), layer.values.clone())
    for layer in dummy_cache.layers
]

compressed = compress_cache(dummy_cache)

# Verify original cache was not modified
unchanged = all(
    torch.equal(layer.keys, before[i][0])
    and torch.equal(layer.values, before[i][1])
    for i, layer in enumerate(dummy_cache.layers)
)

print("Original cache unchanged:", unchanged)
print("Original lengths:", [x[0].shape[2] for x in before])
print("Compressed lengths:", [x.keys.shape[2] for x in compressed.layers])

assert unchanged, "Original cache was modified!"
print("Synthetic cache integrity: PASS")


In [ ]:
import torch
from types import SimpleNamespace

# Save your actual compression settings
print("START_SIZE:", START_SIZE)
print("RECENT_SIZE:", RECENT_SIZE)

# Create a tiny synthetic cache with two layers
seq_len = 10
dummy_cache = SimpleNamespace(layers=[])

for layer_idx in range(2):
    keys = torch.arange(seq_len, dtype=torch.float32).reshape(1, 1, seq_len, 1)
    values = torch.arange(100, 100 + seq_len, dtype=torch.float32).reshape(1, 1, seq_len, 1)
    dummy_cache.layers.append(SimpleNamespace(keys=keys, values=values))

# Snapshot the original tensors
before = [
    (layer.keys.clone(), layer.values.clone())
    for layer in dummy_cache.layers
]

compressed = compress_cache(dummy_cache)

# Verify original cache was not modified
unchanged = all(
    torch.equal(layer.keys, before[i][0])
    and torch.equal(layer.values, before[i][1])
    for i, layer in enumerate(dummy_cache.layers)
)

print("Original cache unchanged:", unchanged)
print("Original lengths:", [x[0].shape[2] for x in before])
print("Compressed lengths:", [x.keys.shape[2] for x in compressed.layers])

assert unchanged, "Original cache was modified!"
print("Synthetic cache integrity: PASS")

In [ ]:

import torch
from types import SimpleNamespace

length = 100
dummy = SimpleNamespace(layers=[])

for _ in range(2):
    dummy.layers.append(SimpleNamespace(
        keys=torch.arange(length).reshape(1, 1, length, 1),
        values=torch.arange(length).reshape(1, 1, length, 1)
    ))

compressed = compress_cache(dummy)

for i, layer in enumerate(compressed.layers):
    kept = layer.keys.flatten().tolist()
    print(
        f"Layer {i}: retained={len(kept)}, "
        f"unique={len(set(kept))}, "
        f"first={kept[0]}, last={kept[-1]}"
    )
    assert len(kept) == START_SIZE + RECENT_SIZE
    assert len(set(kept)) == len(kept)

print("Long-cache compression: PASS")


In [ ]:

import torch

# 1. Confirm required variables exist
assert "model" in globals(), "Model not loaded"
assert "tokenizer" in globals(), "Tokenizer not loaded"
assert "compress_cache" in globals(), "compress_cache() not defined"

model.eval()

# 2. Use a short context
context = "Project code is ORCHID-7319. Lab code is COBALT-4826."
context_ids = tokenizer(
    context, return_tensors="pt"
).input_ids.to(device)

# 3. Create the original KV cache
with torch.inference_mode():
    prefill = model(input_ids=context_ids, use_cache=True)

original_cache = prefill.past_key_values
original_len = original_cache.layers[0].keys.shape[2]

# 4. Compress it
compressed_cache = compress_cache(original_cache)
retained_len = compressed_cache.layers[0].keys.shape[2]

print("Original positions:", original_len)
print("Retained positions:", retained_len)

# 5. Check position IDs and cache positions for one new token
probe_ids = tokenizer(
    "\nQuestion:",
    add_special_tokens=False,
    return_tensors="pt"
).input_ids[:, :1].to(device)

probe_len = probe_ids.shape[1]

attention = torch.ones(
    (1, retained_len + probe_len),
    dtype=torch.long,
    device=device
)

position_ids = torch.tensor(
    [[original_len]],
    dtype=torch.long,
    device=device
)

cache_position = torch.tensor(
    [retained_len],
    dtype=torch.long,
    device=device
)

# 6. Attempt one bounded decoding step
with torch.inference_mode():
    result = model(
        input_ids=probe_ids,
        past_key_values=compressed_cache,
        attention_mask=attention,
        position_ids=position_ids,
        cache_position=cache_position,
        use_cache=True,
    )

# 7. Structural assertions
assert result.logits.shape[0] == 1
assert result.logits.shape[1] == probe_len
assert result.past_key_values is not None

new_len = result.past_key_values.layers[0].keys.shape[2]

assert new_len == retained_len + probe_len, (
    f"Unexpected cache length: {new_len}"
)

assert torch.isfinite(result.logits).all(), "Non-finite logits detected"

print("Forward pass: PASS")
print("Logits finite: PASS")
print("Cache length after probe:", new_len)
print("Structural decoding check: PASS")

# Cleanup
del prefill, original_cache, compressed_cache, result
torch.cuda.empty_cache()


In [ ]:

import torch

# Build a small context longer than the 48-token retention budget.
context = (
    "Project code is ORCHID-7319. "
    "Lab code is COBALT-4826. "
    "Delivery code is MAPLE-9054. "
    "Remember these three codes exactly. "
) * 8

context_ids = tokenizer(
    context, return_tensors="pt"
).input_ids.to(device)

assert context_ids.shape[1] > START_SIZE + RECENT_SIZE, (
    f"Context has only {context_ids.shape[1]} tokens; "
    "increase the repetition count slightly."
)

with torch.inference_mode():
    prefill = model(input_ids=context_ids, use_cache=True)

original_cache = prefill.past_key_values
compressed_cache = compress_cache(original_cache)

original_len = original_cache.layers[0].keys.shape[2]
retained_len = compressed_cache.layers[0].keys.shape[2]

print("Original cache positions:", original_len)
print("Compressed cache positions:", retained_len)
print("Positions removed:", original_len - retained_len)

assert retained_len < original_len, "Compression did not remove positions!"

probe_ids = tokenizer(
    "\nQuestion:",
    add_special_tokens=False,
    return_tensors="pt"
).input_ids[:, :1].to(device)

with torch.inference_mode():
    result = model(
        input_ids=probe_ids,
        past_key_values=compressed_cache,
        attention_mask=torch.ones(
            (1, retained_len + 1), dtype=torch.long, device=device
        ),
        position_ids=torch.tensor(
            [[original_len]], dtype=torch.long, device=device
        ),
        cache_position=torch.tensor(
            [retained_len], dtype=torch.long, device=device
        ),
        use_cache=True,
    )

assert torch.isfinite(result.logits).all()
assert result.past_key_values.layers[0].keys.shape[2] == retained_len + 1

print("Actual compression: PASS")
print("Compressed-cache forward pass: PASS")

del prefill, original_cache, compressed_cache, result
torch.cuda.empty_cache()


In [ ]:

# Measure KV-cache memory directly from tensor storage.
# Use the original and compressed cache variables if they still exist.

def measure_kv_cache_bytes(cache):
    total = 0
    for layer in cache.layers:
        if layer.keys is not None:
            total += layer.keys.numel() * layer.keys.element_size()
        if layer.values is not None:
            total += layer.values.numel() * layer.values.element_size()
    return total

if "original_cache" not in globals() or "compressed_cache" not in globals():
    print("Cache variables are no longer available.")
    print("We can measure them during the next bounded test.")
else:
    original_bytes = measure_kv_cache_bytes(original_cache)
    compressed_bytes = measure_kv_cache_bytes(compressed_cache)

    reduction = (
        100 * (original_bytes - compressed_bytes) / original_bytes
        if original_bytes else 0
    )

    print(f"Original KV cache:   {original_bytes / 1024**2:.4f} MiB")
    print(f"Compressed KV cache: {compressed_bytes / 1024**2:.4f} MiB")
    print(f"Memory reduction:    {reduction:.2f}%")


In [ ]:

import torch

# Short context: enough tokens to exceed the 48-position budget
context = (
    "Project code is ORCHID-7319. "
    "Lab code is COBALT-4826. "
    "Delivery code is MAPLE-9054. "
    "Remember these codes exactly. "
) * 8

context_ids = tokenizer(
    context, return_tensors="pt"
).input_ids.to(device)

def kv_cache_bytes(cache):
    total = 0
    for layer in cache.layers:
        if layer.keys is not None:
            total += layer.keys.numel() * layer.keys.element_size()
        if layer.values is not None:
            total += layer.values.numel() * layer.values.element_size()
    return total

original_cache = None
compressed_cache = None

try:
    with torch.inference_mode():
        output = model(input_ids=context_ids, use_cache=True)

    original_cache = output.past_key_values
    compressed_cache = compress_cache(original_cache)

    original_len = original_cache.layers[0].keys.shape[2]
    retained_len = compressed_cache.layers[0].keys.shape[2]

    original_bytes = kv_cache_bytes(original_cache)
    compressed_bytes = kv_cache_bytes(compressed_cache)

    reduction = (
        100 * (original_bytes - compressed_bytes) / original_bytes
        if original_bytes else 0
    )

    print("Original positions:", original_len)
    print("Compressed positions:", retained_len)
    print(f"Original KV cache: {original_bytes / 1024**2:.4f} MiB")
    print(f"Compressed KV cache: {compressed_bytes / 1024**2:.4f} MiB")
    print(f"KV-cache memory reduction: {reduction:.2f}%")

    assert retained_len < original_len, "Compression did not reduce positions!"
    assert compressed_bytes < original_bytes, "Memory did not decrease!"

    print("Memory measurement: PASS")

finally:
    del output
    del original_cache
    del compressed_cache
    del context_ids
    torch.cuda.empty_cache()


In [ ]:

import torch
from transformers.masking_utils import create_causal_mask

assert "model" in globals(), "Load your existing Qwen2 model first"
assert "tokenizer" in globals(), "Tokenizer is not available"
assert "compress_cache" in globals(), "compress_cache() is not defined"

model.eval()

# Build a small context that exceeds the 48-token retention budget.
context = (
    "Project code is ORCHID-7319. "
    "Lab code is COBALT-4826. "
    "Delivery code is MAPLE-9054. "
) * 6

input_ids = tokenizer(
    context, return_tensors="pt"
).input_ids.to(device)

with torch.inference_mode():
    prefill = model(input_ids=input_ids, use_cache=True)

original_cache = prefill.past_key_values
compressed_cache = compress_cache(original_cache)

original_len = original_cache.layers[0].keys.shape[2]
retained_len = compressed_cache.layers[0].keys.shape[2]

print("Original positions:", original_len)
print("Retained positions:", retained_len)

assert retained_len < original_len, "Compression did not remove positions"

# Probe one new token after the compressed cache.
probe_ids = tokenizer(
    "\nQuestion:",
    add_special_tokens=False,
    return_tensors="pt"
).input_ids[:, :1].to(device)

embeds = model.get_input_embeddings()(probe_ids)

# Standard mask creation using the compressed cache.
mask = create_causal_mask(
    config=model.config,
    inputs_embeds=embeds,
    attention_mask=torch.ones(
        (1, retained_len + 1), dtype=torch.long, device=device
    ),
    past_key_values=compressed_cache,
    position_ids=torch.tensor(
        [[original_len]], dtype=torch.long, device=device
    ),
)

print("Mask type:", type(mask).__name__)
if isinstance(mask, torch.Tensor):
    print("Mask shape:", tuple(mask.shape))
    print("Mask finite:", bool(torch.isfinite(mask).all()))

print("\nIMPORTANT:")
print("This tests standard mask construction, not whether it")
print("represents the original positions of retained tokens correctly.")

del prefill, original_cache, compressed_cache, embeds, mask
torch.cuda.empty_cache()


In [ ]:

import inspect
from transformers.masking_utils import (
    create_causal_mask,
    sdpa_mask,
)

print("=== create_causal_mask ===")
print(inspect.getsource(create_causal_mask))

print("\n=== sdpa_mask ===")
print(inspect.getsource(sdpa_mask))


In [ ]:

import torch
from transformers.masking_utils import create_causal_mask

# Simulate a compressed cache:
# original positions retained = first 4 and last 4
original_positions = torch.tensor([0, 1, 2, 3, 96, 97, 98, 99])

retained_len = len(original_positions)
original_next_position = 100

# Use the model's embedding dtype and device
dummy_ids = tokenizer(
    " X", add_special_tokens=False, return_tensors="pt"
).input_ids[:, :1].to(device)

embeds = model.get_input_embeddings()(dummy_ids)

# Build an explicit mask instead of allowing SDPA's None shortcut
mask = create_causal_mask(
    config=model.config,
    inputs_embeds=embeds,
    attention_mask=torch.ones(
        (1, retained_len + 1), dtype=torch.long, device=device
    ),
    past_key_values=None,
    position_ids=torch.tensor(
        [[original_next_position]], dtype=torch.long, device=device
    ),
    allow_is_causal_skip=False,
)

print("Mask type:", type(mask).__name__)
print("Original retained positions:", original_positions.tolist())
print("Expected allowed keys for the next token:", retained_len)

if isinstance(mask, torch.Tensor):
    print("Mask shape:", tuple(mask.shape))
    print("Mask values:", mask.detach().cpu().tolist())
else:
    print("No explicit tensor mask was returned.")

print(
    "\nNote: this is a diagnostic of mask construction, "
    "not proof that the model uses original positions."
)


In [ ]:

import inspect
from transformers.models.qwen2 import modeling_qwen2

print("=== Qwen2Model.forward ===")
print(inspect.getsource(modeling_qwen2.Qwen2Model.forward))

print("\n=== Qwen2Attention.forward ===")
print(inspect.getsource(modeling_qwen2.Qwen2Attention.forward))


In [ ]:

import torch
from transformers.masking_utils import create_causal_mask

# Keep this test small.
test_context = (
    "Project code: ORCHID-7319. "
    "Lab code: COBALT-4826. "
) * 8

context_ids = tokenizer(
    test_context, return_tensors="pt"
).input_ids.to(device)

with torch.inference_mode():
    prefill = model(
        input_ids=context_ids,
        use_cache=True
    )

original_cache = prefill.past_key_values
compressed_cache = compress_cache(original_cache)

original_len = original_cache.layers[0].keys.shape[2]
retained_len = compressed_cache.layers[0].keys.shape[2]

# One new token, using its original absolute position.
probe_ids = tokenizer(
    " X", add_special_tokens=False, return_tensors="pt"
).input_ids[:, :1].to(device)

probe_embeds = model.get_input_embeddings()(probe_ids)
position_ids = torch.tensor(
    [[original_len]], device=device
)
attention_mask = torch.ones(
    (1, retained_len + 1),
    dtype=torch.long,
    device=device
)

mask = create_causal_mask(
    config=model.config,
    inputs_embeds=probe_embeds,
    attention_mask=attention_mask,
    past_key_values=compressed_cache,
    position_ids=position_ids,
    allow_is_causal_skip=False,
)

print("Original cache length:", original_len)
print("Compressed cache length:", retained_len)
print("Model sliding layers:", model.model.has_sliding_layers)
print("Mask type:", type(mask).__name__)
print("Mask shape:", None if mask is None else tuple(mask.shape))

if mask is not None:
    print("Mask dtype:", mask.dtype)
    print("Mask values:", mask.detach().cpu())

del prefill, original_cache, compressed_cache, probe_embeds
torch.cuda.empty_cache()


In [ ]:

import torch
import gc

# Small controlled test: three facts, three questions.
test_context = """
Project code: ORCHID-7319.
Lab code: COBALT-4826.
Delivery code: MAPLE-9054.
"""

test_questions = [
    ("What is the project code?", "ORCHID-7319"),
    ("What is the lab code?", "COBALT-4826"),
    ("What is the delivery code?", "MAPLE-9054"),
]

MAX_TEST_TOKENS = 12

def run_recall_test(context, question, compressed):
    context_ids = tokenizer(
        context, return_tensors="pt"
    ).input_ids.to(device)

    question_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    # Prefill the context once.
    with torch.inference_mode():
        prefill = model(input_ids=context_ids, use_cache=True)

    original_cache = prefill.past_key_values
    original_len = original_cache.layers[0].keys.shape[2]

    # Keep the original cache intact for the baseline.
    cache = compress_cache(original_cache) if compressed else original_cache
    retained_len = cache.layers[0].keys.shape[2]

    # The question is appended at its original absolute positions.
    q_len = question_ids.shape[1]
    attention_mask = torch.ones(
        (1, retained_len + q_len),
        dtype=torch.long,
        device=device
    )
    position_ids = torch.arange(
        original_len, original_len + q_len, device=device
    ).unsqueeze(0)
    cache_position = torch.arange(
        retained_len, retained_len + q_len, device=device
    )

    with torch.inference_mode():
        out = model(
            input_ids=question_ids,
            past_key_values=cache,
            attention_mask=attention_mask,
            position_ids=position_ids,
            cache_position=cache_position,
            use_cache=True
        )

    generated = []
    next_logits = out.logits[:, -1, :]
    cache = out.past_key_values
    eos = tokenizer.eos_token_id

    for step in range(MAX_TEST_TOKENS):
        token = next_logits.argmax(dim=-1, keepdim=True)
        token_id = token.item()

        if eos is not None and token_id == eos:
            break

        generated.append(token_id)

        if step == MAX_TEST_TOKENS - 1:
            break

        past_len = cache.layers[0].keys.shape[2]
        attention_mask = torch.ones(
            (1, past_len + 1),
            dtype=torch.long,
            device=device
        )
        position = torch.tensor(
            [[original_len + q_len + step]], device=device
        )
        cache_position = torch.tensor(
            [past_len], device=device
        )

        with torch.inference_mode():
            out = model(
                input_ids=token,
                past_key_values=cache,
                attention_mask=attention_mask,
                position_ids=position,
                cache_position=cache_position,
                use_cache=True
            )

        cache = out.past_key_values
        next_logits = out.logits[:, -1, :]

    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    # Clean up GPU memory after each run.
    del prefill, out, cache, original_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    expected = next(
        expected for q, expected in test_questions if q == question
    )
    correct = expected.lower() in answer.lower()

    return {
        "question": question,
        "expected": expected,
        "answer": answer,
        "correct": correct,
        "original_tokens": original_len,
        "retained_tokens": retained_len,
    }


results = []

for question, expected in test_questions:
    for compressed in (False, True):
        result = run_recall_test(
            test_context, question, compressed
        )
        result["mode"] = (
            "compressed" if compressed else "normal"
        )
        results.append(result)

        print(
            f'{result["mode"]:10} | '
            f'{question} | '
            f'Expected: {expected} | '
            f'Answer: {result["answer"]!r} | '
            f'Correct: {result["correct"]}'
        )

print("\n=== SUMMARY ===")
for mode in ("normal", "compressed"):
    subset = [r for r in results if r["mode"] == mode]
    correct = sum(r["correct"] for r in subset)
    print(f"{mode}: {correct}/{len(subset)} correct")

print(
    "Retained cache positions:",
    results[0]["original_tokens"], "->",
    results[1]["retained_tokens"]
)


In [ ]:
import torch
import gc

# Facts go in the MIDDLE of a long context so compression has to evict them.
# START_SIZE and RECENT_SIZE are kept by compression; everything between is removed.
FILLER_BEFORE = " ".join(
    ["The quarterly summary notes that routine operations continued as planned."] * 6
)
FILLER_AFTER = " ".join(
    ["Staff members reviewed general procedures and no action items were recorded."] * 14
)

FACTS = """
Project code: ORCHID-7319.
Lab code: COBALT-4826.
Delivery code: MAPLE-9054.
"""

test_context = FILLER_BEFORE + "\n" + FACTS + "\n" + FILLER_AFTER

# Sanity check: the context must be longer than what compression keeps.
ctx_len = len(tokenizer(test_context).input_ids)
print(f"Test context length: {ctx_len} tokens (START_SIZE + RECENT_SIZE = {START_SIZE + RECENT_SIZE})")
assert ctx_len > START_SIZE + RECENT_SIZE + 20, \
    "Context is still too short. Add more filler text."

test_questions = [
    ("What is the project code?", "ORCHID-7319"),
    ("What is the lab code?", "COBALT-4826"),
    ("What is the delivery code?", "MAPLE-9054"),
]

MAX_TEST_TOKENS = 12


def run_recall_test(context, question, compressed):
    context_ids = tokenizer(
        context, return_tensors="pt"
    ).input_ids.to(device)

    question_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.inference_mode():
        prefill = model(input_ids=context_ids, use_cache=True)

    original_cache = prefill.past_key_values
    original_len = original_cache.layers[0].keys.shape[2]

    cache = compress_cache(original_cache) if compressed else original_cache
    retained_len = cache.layers[0].keys.shape[2]

    # Guard: a "compressed" run must actually have removed tokens.
    if compressed:
        assert retained_len < original_len, (
            f"Compression did not remove any tokens "
            f"({original_len} -> {retained_len}). Check compress_cache."
        )

    q_len = question_ids.shape[1]
    attention_mask = torch.ones(
        (1, retained_len + q_len), dtype=torch.long, device=device
    )
    # Question keeps its original absolute positions.
    position_ids = torch.arange(
        original_len, original_len + q_len, device=device
    ).unsqueeze(0)
    cache_position = torch.arange(
        retained_len, retained_len + q_len, device=device
    )

    with torch.inference_mode():
        out = model(
            input_ids=question_ids,
            past_key_values=cache,
            attention_mask=attention_mask,
            position_ids=position_ids,
            cache_position=cache_position,
            use_cache=True
        )

    generated = []
    next_logits = out.logits[:, -1, :]
    cache = out.past_key_values
    eos = tokenizer.eos_token_id

    for step in range(MAX_TEST_TOKENS):
        token = next_logits.argmax(dim=-1, keepdim=True)
        token_id = token.item()

        if eos is not None and token_id == eos:
            break

        generated.append(token_id)

        if step == MAX_TEST_TOKENS - 1:
            break

        past_len = cache.layers[0].keys.shape[2]
        attention_mask = torch.ones(
            (1, past_len + 1), dtype=torch.long, device=device
        )
        position = torch.tensor(
            [[original_len + q_len + step]], device=device
        )
        cache_position = torch.tensor([past_len], device=device)

        with torch.inference_mode():
            out = model(
                input_ids=token,
                past_key_values=cache,
                attention_mask=attention_mask,
                position_ids=position,
                cache_position=cache_position,
                use_cache=True
            )

        cache = out.past_key_values
        next_logits = out.logits[:, -1, :]

    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    del prefill, out, cache, original_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    expected = next(e for q, e in test_questions if q == question)
    correct = expected.lower() in answer.lower()

    return {
        "question": question,
        "expected": expected,
        "answer": answer,
        "correct": correct,
        "original_tokens": original_len,
        "retained_tokens": retained_len,
    }


results = []

for question, expected in test_questions:
    for compressed in (False, True):
        result = run_recall_test(test_context, question, compressed)
        result["mode"] = "compressed" if compressed else "normal"
        results.append(result)

        print(
            f'{result["mode"]:10} | {question} | '
            f'Expected: {expected} | '
            f'Answer: {result["answer"]!r} | '
            f'Correct: {result["correct"]}'
        )

print("\n=== SUMMARY ===")
for mode in ("normal", "compressed"):
    subset = [r for r in results if r["mode"] == mode]
    correct = sum(r["correct"] for r in subset)
    print(f"{mode}: {correct}/{len(subset)} correct")

comp = [r for r in results if r["mode"] == "compressed"][0]
print(f"Cache size: {comp['original_tokens']} -> {comp['retained_tokens']} tokens")

In [ ]:
FILLER = "The quarterly summary notes that routine operations continued as planned. "
FILL_N = 20  # ~270 tokens of filler total, well above START_SIZE + RECENT_SIZE

fact = "Project code: ORCHID-7319."
question = "What is the project code?"
expected = "ORCHID-7319"

contexts = {
    # fact within the first 16 tokens (kept by the sink)
    "start":  fact + " " + FILLER * FILL_N,
    # fact buried in the middle (evicted)
    "middle": FILLER * (FILL_N // 2) + fact + " " + FILLER * (FILL_N // 2),
    # fact within the last 32 tokens (kept by the recent window)
    "end":    FILLER * FILL_N + fact + " ",
}

# run_recall_test looks up the expected answer from test_questions
test_questions = [(question, expected)]

print(f"{'position':8} | {'mode':10} | correct | answer")
rows = []
for pos, ctx in contexts.items():
    n = len(tokenizer(ctx).input_ids)
    for compressed in (False, True):
        r = run_recall_test(ctx, question, compressed)
        rows.append((pos, compressed, r))
        print(f"{pos:8} | {'compressed' if compressed else 'normal':10} | "
              f"{str(r['correct']):7} | {r['answer']!r}  "
              f"[{r['original_tokens']}->{r['retained_tokens']}]")

In [ ]:
# Make run_recall_test use the new compression (it reads this global)
compress_cache = compress_cache_position_aware

FILLER = "The quarterly summary notes that routine operations continued as planned. "
fact = "Project code: ORCHID-7319."
question = "What is the project code?"
expected = "ORCHID-7319"
test_questions = [(question, expected)]  # run_recall_test looks this up


def fact_token_span(ctx):
    """Token indices covered by the fact (needs a fast tokenizer)."""
    enc = tokenizer(ctx, return_offsets_mapping=True)
    c0 = ctx.index(fact)
    c1 = c0 + len(fact)
    toks = [i for i, (s, e) in enumerate(enc.offset_mapping)
            if e > s and s < c1 and e > c0]
    return toks, len(enc.input_ids)


def fact_is_kept(ctx):
    toks, n = fact_token_span(ctx)
    kept = set(select_position_aware_indices(n))
    return all(t in kept for t in toks), toks, n


def build_context(position):
    if position == "start":
        return fact + " " + FILLER * 20
    if position == "end":
        return FILLER * 20 + fact          # no trailing space
    # middle: scan padding until the fact lands inside the middle window
    for pad in range(0, 40):
        ctx = "Note " * pad + FILLER * 10 + fact + " " + FILLER * 10
        kept, _, _ = fact_is_kept(ctx)
        if kept:
            return ctx
    raise RuntimeError("Could not align fact with middle window; widen the pad range.")


# A 4th case: fact placed in the middle but OUTSIDE the kept window
def build_evicted_context():
    return FILLER * 8 + fact + " " + FILLER * 14   # off-center on purpose


cases = {
    "start": build_context("start"),
    "middle (in window)": build_context("middle"),
    "end": build_context("end"),
    "evicted": build_evicted_context(),
}

print(f"{'case':20} | {'kept?':5} | {'mode':10} | correct | answer")
all_ok = True
for name, ctx in cases.items():
    kept, toks, n = fact_is_kept(ctx)
    for compressed in (False, True):
        r = run_recall_test(ctx, question, compressed)
        mode = "compressed" if compressed else "normal"
        print(f"{name:20} | {str(kept):5} | {mode:10} | {str(r['correct']):7} | "
              f"{r['answer']!r} [{r['original_tokens']}->{r['retained_tokens']}]")
        if not compressed:
            all_ok &= r["correct"]                 # baseline must always pass
        else:
            all_ok &= (r["correct"] == kept)       # compressed passes iff fact was kept

print("\nALL CHECKS PASS" if all_ok else "\nSOMETHING UNEXPECTED, see rows above")

In [ ]:

import copy
import torch

START_KEEP = 16
MIDDLE_KEEP = 16
RECENT_KEEP = 16
TOTAL_KEEP = START_KEEP + MIDDLE_KEEP + RECENT_KEEP

def select_position_aware_indices(length):
    """Select start, middle, and recent token positions."""
    if length <= TOTAL_KEEP:
        return list(range(length))

    middle_start = (length - MIDDLE_KEEP) // 2

    indices = (
        list(range(START_KEEP))
        + list(range(middle_start, middle_start + MIDDLE_KEEP))
        + list(range(length - RECENT_KEEP, length))
    )

    assert len(indices) == TOTAL_KEEP
    assert len(set(indices)) == TOTAL_KEEP
    return indices


def compress_cache_position_aware(cache):
    """Keep start, middle, and recent KV positions."""
    result = copy.deepcopy(cache)

    for layer in result.layers:
        if layer.keys is None or layer.values is None:
            continue

        length = layer.keys.shape[2]
        if length <= TOTAL_KEEP:
            continue

        indices = select_position_aware_indices(length)
        index_tensor = torch.tensor(
            indices, device=layer.keys.device, dtype=torch.long
        )

        layer.keys = layer.keys.index_select(
            2, index_tensor
        ).contiguous()
        layer.values = layer.values.index_select(
            2, index_tensor.to(layer.values.device)
        ).contiguous()

    return result

print("Position-aware compression function defined.")


In [ ]:

length = 233
indices = select_position_aware_indices(length)

print("Original positions:", length)
print("Retained positions:", len(indices))
print("Unique positions:", len(set(indices)))
print("First window:", indices[:START_KEEP])
print(
    "Middle window:",
    indices[START_KEEP:START_KEEP + MIDDLE_KEEP]
)
print("Recent window:", indices[-RECENT_KEEP:])
print("First position:", indices[0])
print("Last position:", indices[-1])

# Put a simulated fact near the middle of the context.
fact_position = length // 2
print("Simulated middle fact position:", fact_position)
print("Middle fact retained:", fact_position in indices)

assert len(indices) == 48
assert len(set(indices)) == 48
assert indices[0] == 0
assert indices[-1] == length - 1
assert fact_position in indices

print("\nPASS: synthetic position-selection test")


In [ ]:

def run_recall_with_strategy(context, question, strategy):
    context_ids = tokenizer(
        context, return_tensors="pt"
    ).input_ids.to(device)

    question_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.inference_mode():
        prefill = model(input_ids=context_ids, use_cache=True)

    original_cache = prefill.past_key_values
    original_len = original_cache.layers[0].keys.shape[2]

    if strategy == "normal":
        cache = original_cache
    elif strategy == "old":
        cache = compress_cache(original_cache)
    elif strategy == "position_aware":
        cache = compress_cache_position_aware(original_cache)
    else:
        raise ValueError("Unknown strategy")

    retained_len = cache.layers[0].keys.shape[2]
    q_len = question_ids.shape[1]

    attention_mask = torch.ones(
        (1, retained_len + q_len),
        dtype=torch.long, device=device
    )
    position_ids = torch.arange(
        original_len, original_len + q_len, device=device
    ).unsqueeze(0)
    cache_position = torch.arange(
        retained_len, retained_len + q_len, device=device
    )

    with torch.inference_mode():
        out = model(
            input_ids=question_ids,
            past_key_values=cache,
            attention_mask=attention_mask,
            position_ids=position_ids,
            cache_position=cache_position,
            use_cache=True
        )

    generated = []
    next_logits = out.logits[:, -1, :]
    cache = out.past_key_values
    eos = tokenizer.eos_token_id

    for step in range(12):
        token = next_logits.argmax(dim=-1, keepdim=True)
        token_id = token.item()

        if eos is not None and token_id == eos:
            break

        generated.append(token_id)
        if step == 11:
            break

        past_len = cache.layers[0].keys.shape[2]
        attention_mask = torch.ones(
            (1, past_len + 1), dtype=torch.long, device=device
        )
        position = torch.tensor(
            [[original_len + q_len + step]], device=device
        )
        cache_position = torch.tensor(
            [past_len], device=device
        )

        with torch.inference_mode():
            out = model(
                input_ids=token,
                past_key_values=cache,
                attention_mask=attention_mask,
                position_ids=position,
                cache_position=cache_position,
                use_cache=True
            )

        cache = out.past_key_values
        next_logits = out.logits[:, -1, :]

    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    del prefill, out, cache, original_cache
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    expected = "ORCHID-7319"
    return {
        "strategy": strategy,
        "answer": answer,
        "correct": expected.lower() in answer.lower(),
        "original_tokens": original_len,
        "retained_tokens": retained_len,
    }


FILLER = (
    "The quarterly summary notes that routine operations "
    "continued as planned. "
)
FILL_N = 20

fact = "Project code: ORCHID-7319."
middle_context = (
    FILLER * (FILL_N // 2)
    + fact + " "
    + FILLER * (FILL_N // 2)
)
question = "What is the project code?"

for strategy in ("normal", "old", "position_aware"):
    r = run_recall_with_strategy(
        middle_context, question, strategy
    )
    print(
        f"{r['strategy']:15} | "
        f"{r['correct']} | "
        f"{r['answer']!r} | "
        f"{r['original_tokens']} -> {r['retained_tokens']}"
    )


In [ ]:

print(f"{'position':8} | {'strategy':15} | correct | answer")

position_results = []

for pos, ctx in contexts.items():
    for strategy in ("normal", "old", "position_aware"):
        r = run_recall_with_strategy(ctx, question, strategy)
        position_results.append((pos, strategy, r))

        print(
            f"{pos:8} | {strategy:15} | "
            f"{str(r['correct']):7} | {r['answer']!r}"
        )

print("\n=== SUMMARY ===")
for strategy in ("normal", "old", "position_aware"):
    subset = [
        r for _, s, r in position_results if s == strategy
    ]
    print(
        f"{strategy:15}: "
        f"{sum(r['correct'] for r in subset)}/{len(subset)}"
    )


In [ ]:
import copy
import gc
import math
import time
import torch
import torch.nn.functional as F
from transformers.cache_utils import DynamicCache

START_KEEP = 16
MIDDLE_KEEP = 16
RECENT_KEEP = 16
TOTAL_KEEP = START_KEEP + MIDDLE_KEEP + RECENT_KEEP  # Strictly 48


def _sync_cuda(device=None):
    if torch.cuda.is_available():
        if device is None or (isinstance(device, torch.device) and device.type == "cuda"):
            torch.cuda.synchronize(device)


def _get_cache_length_and_device(cache):
    if cache is None or not hasattr(cache, "layers"):
        return 0, torch.device("cpu")
    for layer in cache.layers:
        if getattr(layer, "keys", None) is not None:
            return int(layer.keys.shape[2]), layer.keys.device
    return 0, torch.device("cpu")


def select_first_plus_recent_indices(length, first_keep=16, recent_keep=32):
    """Baseline policy: keep first 16 and most recent 32 positions (budget = 48)."""
    total = first_keep + recent_keep
    assert total == TOTAL_KEEP, f"Budget must be {TOTAL_KEEP}, got {total}"
    if length <= 0:
        return []
    if length <= total:
        return list(range(length))
    indices = list(range(first_keep)) + list(range(length - recent_keep, length))
    assert len(indices) == total and len(set(indices)) == total and indices == sorted(indices)
    return indices


def select_position_aware_indices(length):
    """Existing start-middle-recent policy: 16 start + 16 midpoint + 16 recent (budget = 48)."""
    if length <= 0:
        return []
    if length <= TOTAL_KEEP:
        return list(range(length))

    middle_start = (length - MIDDLE_KEEP) // 2
    indices = (
        list(range(START_KEEP))
        + list(range(middle_start, middle_start + MIDDLE_KEEP))
        + list(range(length - RECENT_KEEP, length))
    )
    assert len(indices) == TOTAL_KEEP and len(set(indices)) == TOTAL_KEEP and indices == sorted(indices)
    return indices


def select_multi_window_position_indices(
    length, start_keep=8, num_mid_windows=3, mid_window_size=8, recent_keep=16
):
    """
    Divides interior [start_keep, length - recent_keep) into 3 equal partitions and selects
    a centered 8-position contiguous window from each partition (8 + 3*8 + 16 = 48).
    """
    total_budget = start_keep + (num_mid_windows * mid_window_size) + recent_keep
    assert total_budget == TOTAL_KEEP, f"Budget must equal {TOTAL_KEEP}"
    if length <= 0:
        return []
    if length <= total_budget:
        return list(range(length))

    interior_start = start_keep
    interior_end = length - recent_keep
    interior_len = interior_end - interior_start  # >= 25 when length >= 49

    mid_indices = []
    for w in range(num_mid_windows):
        part_start = interior_start + (w * interior_len) // num_mid_windows
        part_end = interior_start + ((w + 1) * interior_len) // num_mid_windows
        part_len = part_end - part_start
        assert part_len >= mid_window_size, f"Partition too small ({part_len} < {mid_window_size})"
        win_start = part_start + (part_len - mid_window_size) // 2
        win_start = min(win_start, part_end - mid_window_size)
        mid_indices.extend(range(win_start, win_start + mid_window_size))

    indices = (
        list(range(start_keep))
        + mid_indices
        + list(range(length - recent_keep, length))
    )
    assert len(indices) == TOTAL_KEEP and len(set(indices)) == TOTAL_KEEP and indices == sorted(indices)
    return indices


def _select_k_non_overlapping_spans_dp(span_scores_list, k_spans, span_size):
    """
    Exact 1D Dynamic Programming to select `k_spans` strictly non-overlapping spans of length `span_size`
    that maximize total score. Guaranteed to succeed whenever num_offsets >= (k_spans - 1) * span_size + 1.
    """
    num_offsets = len(span_scores_list)
    min_required = (k_spans - 1) * span_size + 1
    assert num_offsets >= min_required, f"Need at least {min_required} offsets, got {num_offsets}"

    # Clean any non-finite scores so degenerate layer values never corrupt selection
    clean_scores = [float(x) if math.isfinite(float(x)) else 0.0 for x in span_scores_list]

    # dp_val[s][i]: best score for picking (s+1) spans where span `s` starts at offset `i`
    dp_val = [[-float("inf")] * num_offsets for _ in range(k_spans)]
    dp_prev = [[-1] * num_offsets for _ in range(k_spans)]

    for i in range(0, num_offsets - (k_spans - 1) * span_size):
        dp_val[0][i] = clean_scores[i]

    for s in range(1, k_spans):
        min_i = s * span_size
        max_i = num_offsets - (k_spans - 1 - s) * span_size
        best_prev_score = -float("inf")
        best_prev_idx = -1
        for i in range(min_i, max_i):
            cand_prev = i - span_size
            if dp_val[s - 1][cand_prev] > best_prev_score:
                best_prev_score = dp_val[s - 1][cand_prev]
                best_prev_idx = cand_prev
            dp_val[s][i] = best_prev_score + clean_scores[i]
            dp_prev[s][i] = best_prev_idx

    # Backtrack from the best final span start
    last_s = k_spans - 1
    best_last_idx = max(
        range(last_s * span_size, num_offsets),
        key=lambda idx: dp_val[last_s][idx],
    )

    chosen = [0] * k_spans
    cur = best_last_idx
    for s in range(last_s, -1, -1):
        chosen[s] = cur
        cur = dp_prev[s][cur]

    return chosen


@torch.no_grad()
def select_hybrid_span_kv_indices(
    cache, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16
):
    """
    Experimental heuristic selector: retains `start_keep` (8) and `recent_keep` (16), and selects
    `num_mid_spans` (3) non-overlapping 8-position interior spans using Value L2 norms and
    Key cosine similarity to the mean recent Key via exact non-overlapping DP.
    """
    total_budget = start_keep + (num_mid_spans * span_size) + recent_keep
    assert total_budget == TOTAL_KEEP

    length, score_device = _get_cache_length_and_device(cache)
    if length <= 0:
        return []
    if length <= total_budget:
        return list(range(length))

    interior_start = start_keep
    interior_end = length - recent_keep
    interior_len = interior_end - interior_start  # Guaranteed >= 25 when length >= 49

    token_scores = torch.zeros(interior_len, device=score_device, dtype=torch.float32)
    valid_layers = 0

    for layer in cache.layers:
        if getattr(layer, "keys", None) is None or getattr(layer, "values", None) is None:
            continue
        if layer.keys.shape[2] != length or layer.values.shape[2] != length:
            raise ValueError("Mismatched sequence length across cache layers.")

        k_int = layer.keys[0, :, interior_start:interior_end, :].to(score_device, dtype=torch.float32)
        v_int = layer.values[0, :, interior_start:interior_end, :].to(score_device, dtype=torch.float32)
        k_rec = layer.keys[0, :, interior_end:length, :].to(score_device, dtype=torch.float32)

        v_norm = torch.norm(v_int, p=2, dim=-1).mean(dim=0)
        v_max = v_norm.max()
        v_norm = torch.where(v_max > 1e-6, v_norm / (v_max + 1e-6), torch.zeros_like(v_norm))

        k_int_unit = F.normalize(k_int, p=2, dim=-1, eps=1e-6)
        k_rec_mean = F.normalize(k_rec.mean(dim=1, keepdim=True), p=2, dim=-1, eps=1e-6)
        k_sim = (k_int_unit * k_rec_mean).sum(dim=-1).clamp(min=0.0).mean(dim=0)
        k_max = k_sim.max()
        k_sim = torch.where(k_max > 1e-6, k_sim / (k_max + 1e-6), torch.zeros_like(k_sim))

        layer_score = torch.nan_to_num(0.65 * v_norm + 0.35 * k_sim, nan=0.0, posinf=0.0, neginf=0.0)
        token_scores += layer_score
        valid_layers += 1

    if valid_layers == 0:
        return select_multi_window_position_indices(
            length, start_keep=start_keep, num_mid_windows=num_mid_spans,
            mid_window_size=span_size, recent_keep=recent_keep
        )

    span_scores = F.avg_pool1d(
        token_scores.view(1, 1, -1), kernel_size=span_size, stride=1
    ).view(-1)

    selected_offsets = _select_k_non_overlapping_spans_dp(
        span_scores.detach().cpu().tolist(), k_spans=num_mid_spans, span_size=span_size
    )

    mid_indices = []
    for offset in selected_offsets:
        abs_start = interior_start + offset
        mid_indices.extend(range(abs_start, abs_start + span_size))

    indices = (
        list(range(start_keep))
        + mid_indices
        + list(range(length - recent_keep, length))
    )

    assert len(indices) == TOTAL_KEEP and len(set(indices)) == TOTAL_KEEP and indices == sorted(indices)
    return indices


@torch.no_grad()
def compress_cache_with_policy(
    cache, policy="start_middle_recent", use_deepcopy=False, return_timing=False
):
    """
    Compresses a DynamicCache to TOTAL_KEEP (48) positions without mutating `cache`.
    If `return_timing=True`, returns `(result_cache, indices, timing_dict)`.
    Otherwise returns `(result_cache, indices)` for backward compatibility.
    """
    length, device = _get_cache_length_and_device(cache)
    _sync_cuda(device)
    t_start = time.perf_counter()

    if policy == "first_plus_recent":
        indices = select_first_plus_recent_indices(length)
    elif policy == "start_middle_recent":
        indices = select_position_aware_indices(length)
    elif policy == "multi_window_pos":
        indices = select_multi_window_position_indices(length)
    elif policy == "hybrid_span_kv":
        indices = select_hybrid_span_kv_indices(cache)
    else:
        raise ValueError(f"Unknown policy: {policy}")

    _sync_cuda(device)
    t_select = time.perf_counter()

    if use_deepcopy:
        result = copy.deepcopy(cache)
        if length > TOTAL_KEEP:
            idx_cache = {}
            for layer in result.layers:
                if getattr(layer, "keys", None) is None or getattr(layer, "values", None) is None:
                    continue
                k_dev = layer.keys.device
                if k_dev not in idx_cache:
                    idx_cache[k_dev] = torch.tensor(indices, device=k_dev, dtype=torch.long)
                v_dev = layer.values.device
                if v_dev not in idx_cache:
                    idx_cache[v_dev] = torch.tensor(indices, device=v_dev, dtype=torch.long)
                layer.keys = layer.keys.index_select(2, idx_cache[k_dev]).contiguous()
                layer.values = layer.values.index_select(2, idx_cache[v_dev]).contiguous()
    else:
        result = DynamicCache()
        idx_cache = {}
        for layer_idx, layer in enumerate(cache.layers):
            if getattr(layer, "keys", None) is None or getattr(layer, "values", None) is None:
                continue
            if length <= TOTAL_KEEP:
                k_sliced = layer.keys.clone()
                v_sliced = layer.values.clone()
            else:
                k_dev = layer.keys.device
                if k_dev not in idx_cache:
                    idx_cache[k_dev] = torch.tensor(indices, device=k_dev, dtype=torch.long)
                v_dev = layer.values.device
                if v_dev not in idx_cache:
                    idx_cache[v_dev] = torch.tensor(indices, device=v_dev, dtype=torch.long)
                k_sliced = layer.keys.index_select(2, idx_cache[k_dev]).contiguous()
                v_sliced = layer.values.index_select(2, idx_cache[v_dev]).contiguous()
            result.update(k_sliced, v_sliced, layer_idx)

    _sync_cuda(device)
    t_end = time.perf_counter()

    timing_info = {
        "selector_ms": (t_select - t_start) * 1000.0,
        "compression_total_ms": (t_end - t_start) * 1000.0,
    }
    if return_timing:
        return result, indices, timing_info
    return result, indices


def measure_kv_cache_mib(cache):
    """Measures KV-cache tensor storage in MiB (strictly K/V tensors, not total GPU VRAM)."""
    total_bytes = 0
    if cache is None or not hasattr(cache, "layers"):
        return 0.0
    for layer in cache.layers:
        if getattr(layer, "keys", None) is not None:
            total_bytes += layer.keys.numel() * layer.keys.element_size()
        if getattr(layer, "values", None) is not None:
            total_bytes += layer.values.numel() * layer.values.element_size()
    return total_bytes / (1024 ** 2)

In [ ]:
@torch.no_grad()
def run_structural_invariant_tests(model, tokenizer):
    print("=== Stage 1A: Fast CPU & Boundary Index-Math Checks ===")
    policies = ["first_plus_recent", "start_middle_recent", "multi_window_pos", "hybrid_span_kv"]

    # 1. Empty / unpopulated cache safety check
    empty_cache = DynamicCache()
    assert select_hybrid_span_kv_indices(empty_cache) == []

    # 2. Sweep critical boundary sequence lengths (including 48, 49, 50..62 where greedy masking used to fail)
    test_lengths = [1, 16, 47, 48, 49, 50, 51, 55, 60, 61, 64, 128, 329]
    for L in test_lengths:
        dummy_cache = DynamicCache()
        # Small 2-layer synthetic CPU cache for fast boundary validation
        for l_idx in range(2):
            torch.manual_seed(42 + l_idx)
            dummy_cache.update(torch.randn(1, 2, L, 16), torch.randn(1, 2, L, 16), l_idx)

        for pol in policies:
            comp_c, idx = compress_cache_with_policy(dummy_cache, policy=pol)
            expected_len = min(L, TOTAL_KEEP)
            assert len(idx) == expected_len, f"[{pol}] L={L}: expected {expected_len}, got {len(idx)}"
            assert len(set(idx)) == expected_len, f"[{pol}] L={L}: duplicate indices!"
            assert idx == sorted(idx), f"[{pol}] L={L}: unsorted indices!"
            assert 0 <= idx[0] and idx[-1] < L, f"[{pol}] L={L}: out of bounds!"

            # Policy-specific anchor checks when L > 48
            if L > TOTAL_KEEP:
                if pol == "first_plus_recent":
                    assert idx[:16] == list(range(16)) and idx[16:] == list(range(L - 32, L))
                elif pol == "start_middle_recent":
                    mid = (L - 16) // 2
                    assert idx[:16] == list(range(16))
                    assert idx[16:32] == list(range(mid, mid + 16))
                    assert idx[32:] == list(range(L - 16, L))
                elif pol in ("multi_window_pos", "hybrid_span_kv"):
                    assert idx[:8] == list(range(8)) and idx[32:] == list(range(L - 16, L))
                    # Verify 3 contiguous 8-token spans in the interior
                    mid_part = idx[8:32]
                    for s in range(3):
                        span = mid_part[s * 8 : (s + 1) * 8]
                        assert span == list(range(span[0], span[0] + 8)), f"[{pol}] Non-contiguous 8-token span!"
    print("[PASS] CPU & boundary length sweep (L=1..329, including L=49..61) verified for all 4 policies.")

    print("\n=== Stage 1B: GPU Model & Cache Structural Invariants ===")
    device = next(model.parameters()).device

    # 3. Short cache (<= 48 positions) must remain uncompressed
    short_inputs = tokenizer("RecallGuard-KV short cache test.", return_tensors="pt").to(device)
    short_len = short_inputs["input_ids"].shape[1]
    assert short_len <= TOTAL_KEEP
    short_cache = model(**short_inputs, use_cache=True).past_key_values
    short_k0_snap = short_cache.layers[0].keys.clone()

    for pol in policies:
        comp_s, idx_s = compress_cache_with_policy(short_cache, policy=pol)
        assert idx_s == list(range(short_len))
        for l_idx, layer in enumerate(comp_s.layers):
            assert torch.equal(layer.keys, short_cache.layers[l_idx].keys)
            assert torch.equal(layer.values, short_cache.layers[l_idx].values)
    assert torch.equal(short_cache.layers[0].keys, short_k0_snap), "Short input cache was mutated!"
    print(f"[PASS] Short-cache no-op & immutability verified (length={short_len} <= 48).")

    # 4. Small-cache equivalence check between use_deepcopy=False and use_deepcopy=True (~68 tokens)
    small_long_text = ("Telemetry check nominal on node alpha. " * 8) + "End."
    small_long_inputs = tokenizer(small_long_text, return_tensors="pt").to(device)
    small_long_len = small_long_inputs["input_ids"].shape[1]
    assert TOTAL_KEEP < small_long_len < 120, f"Expected 49..119 tokens, got {small_long_len}"
    small_long_cache = model(**small_long_inputs, use_cache=True).past_key_values

    for pol in policies:
        c_fast, idx_f = compress_cache_with_policy(small_long_cache, policy=pol, use_deepcopy=False)
        c_deep, idx_d = compress_cache_with_policy(small_long_cache, policy=pol, use_deepcopy=True)
        assert idx_f == idx_d
        for l_idx in range(len(c_fast.layers)):
            assert torch.equal(c_fast.layers[l_idx].keys, c_deep.layers[l_idx].keys)
            assert torch.equal(c_fast.layers[l_idx].values, c_deep.layers[l_idx].values)
        del c_fast, c_deep
    del small_long_cache
    print(f"[PASS] Memory-safe vs. deepcopy equivalence verified on small cache (length={small_long_len}).")

    # 5. Long cache (> 200 positions): exact slice match, input immutability, and BOTH continuation modes
    long_text = ("System log entry: normal routine telemetry check completed. " * 25) + "End of log."
    long_inputs = tokenizer(long_text, return_tensors="pt").to(device)
    long_len = long_inputs["input_ids"].shape[1]
    assert long_len > 200
    long_cache = model(**long_inputs, use_cache=True).past_key_values
    orig_mib = measure_kv_cache_mib(long_cache)

    # Snapshot original layer 0 & layer 23 tensors to prove `long_cache` is never mutated
    orig_k0_snap = long_cache.layers[0].keys.clone()
    orig_v23_snap = long_cache.layers[-1].values.clone()
    next_token = torch.tensor([[tokenizer.eos_token_id or 151643]], device=device)

    for pol in policies:
        comp_cache, indices, t_info = compress_cache_with_policy(
            long_cache, policy=pol, use_deepcopy=False, return_timing=True
        )
        comp_mib = measure_kv_cache_mib(comp_cache)
        idx_t = torch.tensor(indices, device=device, dtype=torch.long)

        # Invariant: all 24 layers exist, match shape [1, H, 48, D], and equal exact indexed slices of `long_cache`
        assert len(comp_cache.layers) == len(long_cache.layers) == 24
        for l_idx, layer in enumerate(comp_cache.layers):
            orig_layer = long_cache.layers[l_idx]
            assert layer.keys.shape == layer.values.shape
            assert layer.keys.shape[2] == TOTAL_KEEP
            assert torch.equal(layer.keys, orig_layer.keys.index_select(2, idx_t))
            assert torch.equal(layer.values, orig_layer.values.index_select(2, idx_t))

        # Invariant: input `long_cache` was not modified
        assert long_cache.layers[0].keys.shape[2] == long_len
        assert torch.equal(long_cache.layers[0].keys, orig_k0_snap)
        assert torch.equal(long_cache.layers[-1].values, orig_v23_snap)

        # Continuation Mode 1: Default cache continuation (next position assumed = 48)
        # Note: mutates `comp_cache` in-place to 49 positions.
        out_default = model(input_ids=next_token, past_key_values=comp_cache, use_cache=True)
        assert torch.isfinite(out_default.logits).all(), f"[{pol}] Non-finite logits in Mode 1!"
        assert comp_cache.layers[0].keys.shape[2] == TOTAL_KEEP + 1

        # Continuation Mode 2: Explicit original absolute position_ids continuation (next position = long_len)
        # Uses a fresh compressed cache so Mode 1's mutation does not contaminate Mode 2.
        comp_cache_mode2, _ = compress_cache_with_policy(long_cache, policy=pol, use_deepcopy=False)
        abs_pos_ids = torch.tensor([[long_len]], device=device, dtype=torch.long)
        out_abs_pos = model(
            input_ids=next_token,
            past_key_values=comp_cache_mode2,
            position_ids=abs_pos_ids,
            use_cache=True,
        )
        assert torch.isfinite(out_abs_pos.logits).all(), f"[{pol}] Non-finite logits in Mode 2!"
        assert comp_cache_mode2.layers[0].keys.shape[2] == TOTAL_KEEP + 1

        reduction_pct = (1.0 - (comp_mib / orig_mib)) * 100.0
        print(
            f"[PASS] Policy={pol:<20} | Orig={long_len} pos ({orig_mib:.4f} MiB) -> "
            f"Retained={TOTAL_KEEP} pos ({comp_mib:.4f} MiB, -{reduction_pct:.2f}% KV tensor storage) | "
            f"Select={t_info['selector_ms']:.2f} ms, TotalComp={t_info['compression_total_ms']:.2f} ms | "
            f"Mode1 & Mode2 Forward OK"
        )
        del comp_cache, comp_cache_mode2, out_default, out_abs_pos

    del long_cache, orig_k0_snap, orig_v23_snap
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run_structural_invariant_tests(model, tokenizer)

In [ ]:
@torch.no_grad()
def run_structural_invariant_tests(model, tokenizer):
    print("=== Stage 1A: Fast CPU & Boundary Index-Math Checks ===")
    policies = ["first_plus_recent", "start_middle_recent", "multi_window_pos", "hybrid_span_kv"]

    # 1. Empty / unpopulated cache safety check
    empty_cache = DynamicCache()
    assert select_hybrid_span_kv_indices(empty_cache) == []

    # 2. Sweep critical boundary sequence lengths (including 48, 49, 50..62 where greedy masking used to fail)
    test_lengths = [1, 16, 47, 48, 49, 50, 51, 55, 60, 61, 64, 128, 329]
    for L in test_lengths:
        dummy_cache = DynamicCache()
        # Small 2-layer synthetic CPU cache for fast boundary validation
        for l_idx in range(2):
            torch.manual_seed(42 + l_idx)
            dummy_cache.update(torch.randn(1, 2, L, 16), torch.randn(1, 2, L, 16), l_idx)

        for pol in policies:
            comp_c, idx = compress_cache_with_policy(dummy_cache, policy=pol)
            expected_len = min(L, TOTAL_KEEP)
            assert len(idx) == expected_len, f"[{pol}] L={L}: expected {expected_len}, got {len(idx)}"
            assert len(set(idx)) == expected_len, f"[{pol}] L={L}: duplicate indices!"
            assert idx == sorted(idx), f"[{pol}] L={L}: unsorted indices!"
            assert 0 <= idx[0] and idx[-1] < L, f"[{pol}] L={L}: out of bounds!"

            # Policy-specific anchor checks when L > 48
            if L > TOTAL_KEEP:
                if pol == "first_plus_recent":
                    assert idx[:16] == list(range(16)) and idx[16:] == list(range(L - 32, L))
                elif pol == "start_middle_recent":
                    mid = (L - 16) // 2
                    assert idx[:16] == list(range(16))
                    assert idx[16:32] == list(range(mid, mid + 16))
                    assert idx[32:] == list(range(L - 16, L))
                elif pol in ("multi_window_pos", "hybrid_span_kv"):
                    assert idx[:8] == list(range(8)) and idx[32:] == list(range(L - 16, L))
                    # Verify 3 contiguous 8-token spans in the interior
                    mid_part = idx[8:32]
                    for s in range(3):
                        span = mid_part[s * 8 : (s + 1) * 8]
                        assert span == list(range(span[0], span[0] + 8)), f"[{pol}] Non-contiguous 8-token span!"
    print("[PASS] CPU & boundary length sweep (L=1..329, including L=49..61) verified for all 4 policies.")

    print("\n=== Stage 1B: GPU Model & Cache Structural Invariants ===")
    device = next(model.parameters()).device

    # 3. Short cache (<= 48 positions) must remain uncompressed
    short_inputs = tokenizer("RecallGuard-KV short cache test.", return_tensors="pt").to(device)
    short_len = short_inputs["input_ids"].shape[1]
    assert short_len <= TOTAL_KEEP
    short_cache = model(**short_inputs, use_cache=True).past_key_values
    short_k0_snap = short_cache.layers[0].keys.clone()

    for pol in policies:
        comp_s, idx_s = compress_cache_with_policy(short_cache, policy=pol)
        assert idx_s == list(range(short_len))
        for l_idx, layer in enumerate(comp_s.layers):
            assert torch.equal(layer.keys, short_cache.layers[l_idx].keys)
            assert torch.equal(layer.values, short_cache.layers[l_idx].values)
    assert torch.equal(short_cache.layers[0].keys, short_k0_snap), "Short input cache was mutated!"
    print(f"[PASS] Short-cache no-op & immutability verified (length={short_len} <= 48).")

    # 4. Small-cache equivalence check between use_deepcopy=False and use_deepcopy=True (~68 tokens)
    small_long_text = ("Telemetry check nominal on node alpha. " * 8) + "End."
    small_long_inputs = tokenizer(small_long_text, return_tensors="pt").to(device)
    small_long_len = small_long_inputs["input_ids"].shape[1]
    assert TOTAL_KEEP < small_long_len < 120, f"Expected 49..119 tokens, got {small_long_len}"
    small_long_cache = model(**small_long_inputs, use_cache=True).past_key_values

    for pol in policies:
        c_fast, idx_f = compress_cache_with_policy(small_long_cache, policy=pol, use_deepcopy=False)
        c_deep, idx_d = compress_cache_with_policy(small_long_cache, policy=pol, use_deepcopy=True)
        assert idx_f == idx_d
        for l_idx in range(len(c_fast.layers)):
            assert torch.equal(c_fast.layers[l_idx].keys, c_deep.layers[l_idx].keys)
            assert torch.equal(c_fast.layers[l_idx].values, c_deep.layers[l_idx].values)
        del c_fast, c_deep
    del small_long_cache
    print(f"[PASS] Memory-safe vs. deepcopy equivalence verified on small cache (length={small_long_len}).")

    # 5. Long cache (> 200 positions): exact slice match, input immutability, and BOTH continuation modes
    long_text = ("System log entry: normal routine telemetry check completed. " * 25) + "End of log."
    long_inputs = tokenizer(long_text, return_tensors="pt").to(device)
    long_len = long_inputs["input_ids"].shape[1]
    assert long_len > 200
    long_cache = model(**long_inputs, use_cache=True).past_key_values
    orig_mib = measure_kv_cache_mib(long_cache)

    # Snapshot original layer 0 & layer 23 tensors to prove `long_cache` is never mutated
    orig_k0_snap = long_cache.layers[0].keys.clone()
    orig_v23_snap = long_cache.layers[-1].values.clone()
    next_token = torch.tensor([[tokenizer.eos_token_id or 151643]], device=device)

    for pol in policies:
        comp_cache, indices, t_info = compress_cache_with_policy(
            long_cache, policy=pol, use_deepcopy=False, return_timing=True
        )
        comp_mib = measure_kv_cache_mib(comp_cache)
        idx_t = torch.tensor(indices, device=device, dtype=torch.long)

        # Invariant: all 24 layers exist, match shape [1, H, 48, D], and equal exact indexed slices of `long_cache`
        assert len(comp_cache.layers) == len(long_cache.layers) == 24
        for l_idx, layer in enumerate(comp_cache.layers):
            orig_layer = long_cache.layers[l_idx]
            assert layer.keys.shape == layer.values.shape
            assert layer.keys.shape[2] == TOTAL_KEEP
            assert torch.equal(layer.keys, orig_layer.keys.index_select(2, idx_t))
            assert torch.equal(layer.values, orig_layer.values.index_select(2, idx_t))

        # Invariant: input `long_cache` was not modified
        assert long_cache.layers[0].keys.shape[2] == long_len
        assert torch.equal(long_cache.layers[0].keys, orig_k0_snap)
        assert torch.equal(long_cache.layers[-1].values, orig_v23_snap)

        # Continuation Mode 1: Default cache continuation (next position assumed = 48)
        # Note: mutates `comp_cache` in-place to 49 positions.
        out_default = model(input_ids=next_token, past_key_values=comp_cache, use_cache=True)
        assert torch.isfinite(out_default.logits).all(), f"[{pol}] Non-finite logits in Mode 1!"
        assert comp_cache.layers[0].keys.shape[2] == TOTAL_KEEP + 1

        # Continuation Mode 2: Explicit original absolute position_ids continuation (next position = long_len)
        # Uses a fresh compressed cache so Mode 1's mutation does not contaminate Mode 2.
        comp_cache_mode2, _ = compress_cache_with_policy(long_cache, policy=pol, use_deepcopy=False)
        abs_pos_ids = torch.tensor([[long_len]], device=device, dtype=torch.long)
        out_abs_pos = model(
            input_ids=next_token,
            past_key_values=comp_cache_mode2,
            position_ids=abs_pos_ids,
            use_cache=True,
        )
        assert torch.isfinite(out_abs_pos.logits).all(), f"[{pol}] Non-finite logits in Mode 2!"
        assert comp_cache_mode2.layers[0].keys.shape[2] == TOTAL_KEEP + 1

        reduction_pct = (1.0 - (comp_mib / orig_mib)) * 100.0
        print(
            f"[PASS] Policy={pol:<20} | Orig={long_len} pos ({orig_mib:.4f} MiB) -> "
            f"Retained={TOTAL_KEEP} pos ({comp_mib:.4f} MiB, -{reduction_pct:.2f}% KV tensor storage) | "
            f"Select={t_info['selector_ms']:.2f} ms, TotalComp={t_info['compression_total_ms']:.2f} ms | "
            f"Mode1 & Mode2 Forward OK"
        )
        del comp_cache, comp_cache_mode2, out_default, out_abs_pos

    del long_cache, orig_k0_snap, orig_v23_snap
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run_structural_invariant_tests(model, tokenizer)

In [ ]:
import re
import numpy as np
import pandas as pd

FILLER_TEMPLATES = {
    "telemetry": (
        "Node cluster status report: CPU utilization remains steady at 42 percent with nominal thermal margins. "
        "Network packet routing across internal switches shows zero dropped frames during the last monitoring interval. "
    ),
    "operations": (
        "Routine maintenance schedule: facility ventilation systems and backup power relays passed weekly inspection. "
        "Inventory logs confirm that standard replacement components are stocked in warehouse sector four. "
    ),
    "research": (
        "Background literature notes: standard calibration curves were verified against reference laboratory samples. "
        "Ambient humidity and barometric pressure stayed within acceptable experimental tolerances throughout the trial. "
    ),
}


def build_prompt_with_fact_at_depth(
    tokenizer,
    fact_sentence,
    question_text,
    target_depth_ratio,
    approx_context_tokens=240,
    template_key="telemetry",
    expected_identifier=None,
    return_metadata=False,
):
    """
    Constructs a chat-templated prompt with `fact_sentence` inserted near `target_depth_ratio` of the filler.
    When `return_metadata=True`, uses tokenizer char-to-token `offset_mapping` to measure the exact token
    indices and actual normalized depth of `expected_identifier` (and `fact_sentence`) in the tokenized prompt.
    """
    filler_unit = FILLER_TEMPLATES[template_key]
    unit_tokens = len(tokenizer.encode(filler_unit, add_special_tokens=False))
    num_units = max(6, approx_context_tokens // unit_tokens)

    insert_idx = max(0, min(num_units, int(round(target_depth_ratio * num_units))))
    chunks = [filler_unit for _ in range(num_units)]
    chunks.insert(insert_idx, f" {fact_sentence} ")

    context_str = "".join(chunks)
    messages = [
        {"role": "system", "content": "You are a precise assistant. Answer the question using only the exact identifier from the text."},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {question_text}\nAnswer with the exact identifier only:"},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    if not return_metadata:
        return prompt_text

    # Unambiguous character span location
    assert prompt_text.count(fact_sentence) == 1, "fact_sentence must appear exactly once in prompt_text."
    fact_char_start = prompt_text.index(fact_sentence)
    fact_char_end = fact_char_start + len(fact_sentence)

    target_str = expected_identifier if expected_identifier is not None else fact_sentence
    assert prompt_text.count(target_str) == 1, f"Identifier '{target_str}' must appear exactly once in prompt_text."
    id_char_start = prompt_text.index(target_str)
    id_char_end = id_char_start + len(target_str)

    # Map character spans to exact token indices via offset_mapping
    enc = tokenizer(prompt_text, return_offsets_mapping=True, add_special_tokens=False)
    input_ids_list = enc["input_ids"]
    offsets = enc["offset_mapping"]
    total_tokens = len(input_ids_list)
    prefill_tokens = total_tokens - 1

    id_token_indices = [
        tok_i for tok_i, (c_start, c_end) in enumerate(offsets)
        if c_end > id_char_start and c_start < id_char_end
    ]
    fact_token_indices = [
        tok_i for tok_i, (c_start, c_end) in enumerate(offsets)
        if c_end > fact_char_start and c_start < fact_char_end
    ]
    assert len(id_token_indices) > 0, f"Failed to map identifier '{target_str}' to token indices."

    decoded_span = tokenizer.decode([input_ids_list[i] for i in id_token_indices])
    assert target_str in decoded_span, f"Token span decode '{decoded_span}' does not contain '{target_str}'"

    id_center_tok = 0.5 * (id_token_indices[0] + id_token_indices[-1])
    actual_depth_pct = round(100.0 * (id_center_tok / max(1, prefill_tokens)), 2)

    meta = {
        "prompt_text": prompt_text,
        "target_depth_pct": round(100.0 * target_depth_ratio, 1),
        "actual_depth_pct": actual_depth_pct,
        "total_prompt_tokens": total_tokens,
        "prefill_tokens": prefill_tokens,
        "id_token_indices": id_token_indices,
        "id_token_span": f"{id_token_indices[0]}..{id_token_indices[-1]}",
        "fact_token_span": f"{fact_token_indices[0]}..{fact_token_indices[-1]}",
    }
    return prompt_text, meta


def check_exact_fact_match(generated_answer, expected_identifier):
    """
    Documented Matcher Rule:
    Case-insensitive regex match requiring non-alphanumeric boundaries around `expected_identifier`
    so 'RG-101-A' matches 'RG-101-A' or 'Answer: RG-101-A.', but never matches 'RG-101-A9'.
    """
    norm_ans = generated_answer.strip().upper()
    norm_exp = expected_identifier.strip().upper()
    pattern = r"(?<![A-Z0-9])" + re.escape(norm_exp) + r"(?![A-Z0-9])"
    return bool(re.search(pattern, norm_ans))


@torch.no_grad()
def generate_from_compressed_cache(
    model,
    tokenizer,
    prompt_text,
    policy="full_cache",
    max_new_tokens=16,
    preserve_true_position_ids=False,
    id_token_indices=None,
    num_runs=3,
    warmup_runs=1,
):
    """
    Executes prefill -> cache compression -> greedy decoding with CUDA synchronization,
    warm-up + repeated timing trials (reporting median and IQR), and peak GPU memory tracking.
    """
    device = next(model.parameters()).device
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(device)
    input_ids = inputs["input_ids"]
    prefill_ids = input_ids[:, :-1]
    last_token_id = input_ids[:, -1:]
    prefill_len = int(prefill_ids.shape[1])

    prefill_times, select_times, compress_times, decode_times, e2e_times = [], [], [], [], []
    peak_delta_mibs = []
    final_record = None

    total_iters = warmup_runs + max(1, num_runs)
    for run_idx in range(total_iters):
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
            _sync_cuda(device)
            base_mem_bytes = torch.cuda.memory_allocated(device)
            torch.cuda.reset_peak_memory_stats(device)
        else:
            base_mem_bytes = 0

        # 1. Timed Prefill
        _sync_cuda(device)
        t0 = time.perf_counter()
        out = model(input_ids=prefill_ids, use_cache=True)
        orig_cache = out.past_key_values
        _sync_cuda(device)
        t1 = time.perf_counter()

        orig_mib = measure_kv_cache_mib(orig_cache)

        # 2. Timed Compression (and Selector Scoring Overhead)
        if policy == "full_cache":
            active_cache = orig_cache
            indices = list(range(prefill_len))
            sel_ms = 0.0
            comp_ms = 0.0
        else:
            active_cache, indices, t_info = compress_cache_with_policy(
                orig_cache, policy=policy, use_deepcopy=False, return_timing=True
            )
            sel_ms = t_info["selector_ms"]
            comp_ms = t_info["compression_total_ms"]
            # Release original uncompressed cache before decoding
            del orig_cache

        del out
        comp_mib = measure_kv_cache_mib(active_cache)
        retained_pos = int(active_cache.layers[0].keys.shape[2])
        removed_pos = prefill_len - retained_pos

        # 3. Timed Greedy Decoding
        _sync_cuda(device)
        t2 = time.perf_counter()
        generated_ids = []
        cur_token = last_token_id
        true_pos = prefill_len

        for step in range(max_new_tokens):
            if preserve_true_position_ids and policy != "full_cache":
                pos_ids = torch.tensor([[true_pos + step]], device=device, dtype=torch.long)
                step_out = model(
                    input_ids=cur_token, past_key_values=active_cache, position_ids=pos_ids, use_cache=True
                )
            else:
                step_out = model(input_ids=cur_token, past_key_values=active_cache, use_cache=True)
            active_cache = step_out.past_key_values
            next_token = torch.argmax(step_out.logits[:, -1, :], dim=-1, keepdim=True)
            token_id = int(next_token.item())
            del step_out
            if token_id == tokenizer.eos_token_id:
                break
            generated_ids.append(token_id)
            cur_token = next_token

        _sync_cuda(device)
        t3 = time.perf_counter()

        if device.type == "cuda":
            peak_bytes = torch.cuda.max_memory_allocated(device)
            peak_delta_mib = max(0.0, (peak_bytes - base_mem_bytes) / (1024 ** 2))
        else:
            peak_delta_mib = 0.0

        p_ms = (t1 - t0) * 1000.0
        d_ms = (t3 - t2) * 1000.0
        total_ms = p_ms + comp_ms + d_ms

        if run_idx >= warmup_runs:
            prefill_times.append(p_ms)
            select_times.append(sel_ms)
            compress_times.append(comp_ms)
            decode_times.append(d_ms)
            e2e_times.append(total_ms)
            peak_delta_mibs.append(peak_delta_mib)

        if final_record is None:
            answer_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            retained_set = set(indices)
            if id_token_indices:
                kept_id_toks = sum(1 for t in id_token_indices if t in retained_set)
                id_ret_ratio = round(kept_id_toks / len(id_token_indices), 2)
            else:
                id_ret_ratio = np.nan

            storage_red_pct = (1.0 - (comp_mib / orig_mib)) * 100.0 if orig_mib > 0 else 0.0
            final_record = {
                "answer": answer_text,
                "orig_positions": prefill_len,
                "retained_positions": retained_pos,
                "removed_positions": removed_pos,
                "id_tokens_retained_ratio": id_ret_ratio,
                "orig_kv_mib": round(orig_mib, 4),
                "comp_kv_mib": round(comp_mib, 4),
                "kv_reduction_pct": round(storage_red_pct, 2),
                "retained_indices": indices,
            }

        del active_cache

    def _med_iqr(arr):
        q25, q50, q75 = np.percentile(arr, [25, 50, 75])
        return round(float(q50), 2), round(float(q75 - q25), 2)

    e2e_med, e2e_iqr = _med_iqr(e2e_times)
    comp_med, comp_iqr = _med_iqr(compress_times)
    sel_med, _ = _med_iqr(select_times)
    dec_med, _ = _med_iqr(decode_times)
    pref_med, _ = _med_iqr(prefill_times)
    peak_med, _ = _med_iqr(peak_delta_mibs)

    final_record.update({
        "prefill_ms": pref_med,
        "selector_ms": sel_med,
        "compress_ms": comp_med,
        "compress_iqr_ms": comp_iqr,
        "decode_ms": dec_med,
        "latency_ms": e2e_med,
        "latency_iqr_ms": e2e_iqr,
        "peak_gpu_delta_mib": peak_med,
    })
    return final_record


def run_five_depth_smoke_test(
    model, tokenizer, approx_tokens=240, template_key="telemetry", num_runs=3, warmup_runs=1
):
    depths = [
        ("Start (~5%)", 0.05, "RG-101-A"),
        ("Early-Mid (~25%)", 0.25, "RG-205-B"),
        ("Exact-Mid (~50%)", 0.50, "RG-500-M"),
        ("Late-Mid (~75%)", 0.75, "RG-750-L"),
        ("Recent (~92%)", 0.92, "RG-920-Z"),
    ]
    policies = [
        "full_cache",
        "first_plus_recent",
        "start_middle_recent",
        "multi_window_pos",
        "hybrid_span_kv",
    ]

    rows = []
    for depth_label, ratio, expected_id in depths:
        fact = f"IMPORTANT RECORD: The assigned project identifier is {expected_id}."
        question = "What is the assigned project identifier?"
        prompt, meta = build_prompt_with_fact_at_depth(
            tokenizer,
            fact,
            question,
            ratio,
            approx_context_tokens=approx_tokens,
            template_key=template_key,
            expected_identifier=expected_id,
            return_metadata=True,
        )

        # Run full_cache baseline first on the exact prompt
        base_res = generate_from_compressed_cache(
            model, tokenizer, prompt, policy="full_cache",
            id_token_indices=meta["id_token_indices"], num_runs=num_runs, warmup_runs=warmup_runs
        )
        base_passed = check_exact_fact_match(base_res["answer"], expected_id)

        for pol in policies:
            res = (
                base_res
                if pol == "full_cache"
                else generate_from_compressed_cache(
                    model, tokenizer, prompt, policy=pol,
                    id_token_indices=meta["id_token_indices"], num_runs=num_runs, warmup_runs=warmup_runs
                )
            )
            passed = check_exact_fact_match(res["answer"], expected_id)
            rows.append({
                "Target_Depth": depth_label,
                "Actual_Depth_%": meta["actual_depth_pct"],
                "ID_Tok_Span": meta["id_token_span"],
                "Policy": pol,
                "Expected": expected_id,
                "Raw_Answer": res["answer"],
                "Passed": int(passed),
                "Baseline_Valid": int(base_passed),
                "ID_Tok_Kept_Ratio": res["id_tokens_retained_ratio"],
                "Orig_Pos": res["orig_positions"],
                "Retained_Pos": res["retained_positions"],
                "Removed_Pos": res["removed_positions"],
                "Orig_KV_MiB": res["orig_kv_mib"],
                "Comp_KV_MiB": res["comp_kv_mib"],
                "KV_Red_%": res["kv_reduction_pct"],
                "Peak_Delta_MiB": res["peak_gpu_delta_mib"],
                "Select_ms": res["selector_ms"],
                "Compress_ms": res["compress_ms"],
                "Decode_ms": res["decode_ms"],
                "E2E_Med_ms": res["latency_ms"],
                "E2E_IQR_ms": res["latency_iqr_ms"],
            })

    df = pd.DataFrame(rows)
    print("\n=== Stage 2A: 5-Depth Smoke Test Results (Measured Token Depths & Split Timings) ===")
    display_cols = [
        "Target_Depth", "Actual_Depth_%", "ID_Tok_Span", "Policy", "Expected", "Raw_Answer",
        "Passed", "Baseline_Valid", "ID_Tok_Kept_Ratio", "Orig_Pos", "Retained_Pos",
        "Comp_KV_MiB", "KV_Red_%", "Select_ms", "Compress_ms", "Decode_ms", "E2E_Med_ms"
    ]
    print(df[display_cols].to_string(index=False))

    total_cases_per_pol = len(depths)
    base_failures = int((df[df["Policy"] == "full_cache"]["Baseline_Valid"] == 0).sum())
    eligible_cases = total_cases_per_pol - base_failures
    print(f"\nBaseline Eligibility: {eligible_cases}/{total_cases_per_pol} cases passed full_cache ({base_failures} baseline failures).")

    valid_df = df[df["Baseline_Valid"] == 1]
    summary = df.groupby("Policy", sort=False).agg(
        Eligible_Cases=("Baseline_Valid", "sum"),
        Baseline_Failures=("Baseline_Valid", lambda s: int((s == 0).sum())),
        Valid_Passed=("Passed", lambda s: int(s[df.loc[s.index, "Baseline_Valid"] == 1].sum())),
        Mean_ID_Kept_Ratio=("ID_Tok_Kept_Ratio", "mean"),
        Comp_KV_MiB=("Comp_KV_MiB", "mean"),
        Peak_Delta_MiB=("Peak_Delta_MiB", "mean"),
        Med_Select_ms=("Select_ms", "median"),
        Med_Compress_ms=("Compress_ms", "median"),
        Med_Decode_ms=("Decode_ms", "median"),
        Med_E2E_ms=("E2E_Med_ms", "median"),
    )
    summary["Valid_Recall_%"] = np.where(
        summary["Eligible_Cases"] > 0,
        (100.0 * summary["Valid_Passed"] / summary["Eligible_Cases"]).round(1),
        np.nan,
    )
    print("\n=== Smoke Test Summary by Policy ===")
    print(summary.to_string())
    return df

# Execute after Cell 2 passes:
smoke_df = run_five_depth_smoke_test(model, tokenizer)

In [ ]:
import os

def run_expanded_evaluation(
    model,
    tokenizer,
    checkpoint_csv="recallguard_expanded_checkpoint.csv",
    num_runs=1,
    warmup_runs=0,
):
    """
    Guarded, resumable 30-condition x 5-policy expanded evaluation.
    Saves results incrementally after each prompt condition so Colab disconnects never lose completed runs.
    """
    # Pre-flight GPU memory gate check
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_mib = free_bytes / (1024 ** 2)
        assert free_mib > 500.0, f"Insufficient free GPU memory ({free_mib:.1f} MiB < 500 MiB). Restart/clear VRAM first."
        print(f"[Pre-flight OK] Available GPU memory: {free_mib:.1f} MiB / {total_bytes / (1024**2):.1f} MiB")

    lengths = [240, 480]
    templates = ["telemetry", "operations", "research"]
    depths = [
        ("Start_5%", 0.05, "PRJ-110-S"),
        ("EarlyMid_25%", 0.25, "PRJ-250-E"),
        ("ExactMid_50%", 0.50, "PRJ-500-C"),
        ("LateMid_75%", 0.75, "PRJ-750-L"),
        ("Recent_92%", 0.92, "PRJ-920-R"),
    ]
    policies = [
        "full_cache",
        "first_plus_recent",
        "start_middle_recent",
        "multi_window_pos",
        "hybrid_span_kv",
    ]

    if os.path.exists(checkpoint_csv):
        existing_df = pd.read_csv(checkpoint_csv)
        completed_keys = set(
            zip(existing_df["Target_Len"], existing_df["Template"], existing_df["Depth"], existing_df["Policy"])
        )
        records = existing_df.to_dict("records")
        print(f"[Resume] Loaded {len(records)} completed policy evaluations from {checkpoint_csv}.")
    else:
        completed_keys = set()
        records = []

    for ctx_len in lengths:
        for tmpl in templates:
            for depth_name, ratio, code in depths:
                if all((ctx_len, tmpl, depth_name, pol) in completed_keys for pol in policies):
                    continue

                fact = f"CRITICAL NOTE: The active project identifier is {code}."
                question = "What is the active project identifier?"
                prompt, meta = build_prompt_with_fact_at_depth(
                    tokenizer, fact, question, ratio,
                    approx_context_tokens=ctx_len, template_key=tmpl,
                    expected_identifier=code, return_metadata=True
                )
                assert meta["prefill_tokens"] > TOTAL_KEEP, "Expanded evaluation case must exceed 48 tokens!"

                base_res = generate_from_compressed_cache(
                    model, tokenizer, prompt, policy="full_cache",
                    id_token_indices=meta["id_token_indices"], num_runs=num_runs, warmup_runs=warmup_runs
                )
                base_ok = check_exact_fact_match(base_res["answer"], code)

                for pol in policies:
                    if (ctx_len, tmpl, depth_name, pol) in completed_keys:
                        continue
                    res = (
                        base_res
                        if pol == "full_cache"
                        else generate_from_compressed_cache(
                            model, tokenizer, prompt, policy=pol,
                            id_token_indices=meta["id_token_indices"], num_runs=num_runs, warmup_runs=warmup_runs
                        )
                    )
                    if pol != "full_cache":
                        assert res["removed_positions"] > 0, f"[{pol}] Expected compression to remove >0 positions!"
                    ok = check_exact_fact_match(res["answer"], code)
                    records.append({
                        "Target_Len": ctx_len,
                        "Template": tmpl,
                        "Depth": depth_name,
                        "Actual_Depth_%": meta["actual_depth_pct"],
                        "ID_Tok_Span": meta["id_token_span"],
                        "Policy": pol,
                        "Expected": code,
                        "Raw_Answer": res["answer"],
                        "Pass": int(ok),
                        "Base_OK": int(base_ok),
                        "ID_Tok_Kept_Ratio": res["id_tokens_retained_ratio"],
                        "Orig_Pos": res["orig_positions"],
                        "Retained_Pos": res["retained_positions"],
                        "Removed_Pos": res["removed_positions"],
                        "Orig_KV_MiB": res["orig_kv_mib"],
                        "Comp_KV_MiB": res["comp_kv_mib"],
                        "Peak_Delta_MiB": res["peak_gpu_delta_mib"],
                        "Select_ms": res["selector_ms"],
                        "Compress_ms": res["compress_ms"],
                        "Decode_ms": res["decode_ms"],
                        "E2E_Med_ms": res["latency_ms"],
                    })
                    completed_keys.add((ctx_len, tmpl, depth_name, pol))

                # Save incrementally after each 5-policy condition
                pd.DataFrame(records).to_csv(checkpoint_csv, index=False)

    full_df = pd.DataFrame(records)
    base_rows = full_df[full_df["Policy"] == "full_cache"]
    base_fail_count = int((base_rows["Base_OK"] == 0).sum())
    print(f"\nFull-cache baseline: {len(base_rows) - base_fail_count}/{len(base_rows)} passed ({base_fail_count} failures separated).")

    valid = full_df[full_df["Base_OK"] == 1]
    if len(valid) > 0:
        pivot = valid.pivot_table(
            index="Policy", columns="Depth", values="Pass", aggfunc="mean", sort=False
        ) * 100.0
        pivot["Valid_Recall_%"] = valid.groupby("Policy", sort=False)["Pass"].mean() * 100.0
        pivot["Mean_ID_Kept_%"] = valid.groupby("Policy", sort=False)["ID_Tok_Kept_Ratio"].mean() * 100.0
        pivot["Comp_KV_MiB"] = valid.groupby("Policy", sort=False)["Comp_KV_MiB"].mean()
        pivot["Med_Compress_ms"] = valid.groupby("Policy", sort=False)["Compress_ms"].median()
        pivot["Med_E2E_ms"] = valid.groupby("Policy", sort=False)["E2E_Med_ms"].median()
        print("\n=== Expanded Evaluation Summary (Baseline-Valid Cases Only) ===")
        print(pivot.round(2).to_string())
    return full_df

# DO NOT UNCOMMENT until Cell 2 and Cell 3 pass and are reviewed:
# expanded_df = run_expanded_evaluation(model, tokenizer)

In [ ]:
# ============================================================================
# Diagnostic & Fix Cell:
# 1. Compare preserve_true_position_ids=False vs True when ID + Question are kept
# 2. Patch select_hybrid_span_kv_indices (4 sink + 3x8 novelty spans + 20 recent = 48)
# ============================================================================

@torch.no_grad()
def select_hybrid_span_kv_indices(
    cache, start_keep=4, num_mid_spans=3, span_size=8, recent_keep=20
):
    """
    Patched Hybrid Selector (Budget = 4 + 3*8 + 20 = 48 positions):
    - Keeps first 4 positions (attention sinks) and last 20 positions (preserves trailing question).
    - Excludes the system-prompt prefix boundary from overwhelming interior span scores by
      computing Key Novelty (1 - cos(K_int, mean(K_int))) + Value L2 norm vectorized across layers.
      Repeated filler tokens cluster near mean(K_int) (low novelty), while unique facts stand out.
    """
    total_budget = start_keep + (num_mid_spans * span_size) + recent_keep
    assert total_budget == TOTAL_KEEP, f"Budget must be {TOTAL_KEEP}, got {total_budget}"

    length, score_device = _get_cache_length_and_device(cache)
    if length <= 0:
        return []
    if length <= total_budget:
        return list(range(length))

    interior_start = start_keep
    interior_end = length - recent_keep
    interior_len = interior_end - interior_start

    # Stack keys and values across populated layers in one batched tensor: [num_layers, H, L_int, D]
    k_list, v_list, k_rec_list = [], [], []
    for layer in cache.layers:
        if getattr(layer, "keys", None) is None or getattr(layer, "values", None) is None:
            continue
        if layer.keys.shape[2] != length or layer.values.shape[2] != length:
            raise ValueError("Mismatched sequence length across cache layers.")
        k_list.append(layer.keys[0, :, interior_start:interior_end, :])
        v_list.append(layer.values[0, :, interior_start:interior_end, :])
        k_rec_list.append(layer.keys[0, :, interior_end:length, :])

    if len(k_list) == 0:
        return select_multi_window_position_indices(
            length, start_keep=start_keep, num_mid_windows=num_mid_spans,
            mid_window_size=span_size, recent_keep=recent_keep
        )

    # Use upper-middle layers (layers 6..21) where entity/semantic representations peak, or all if <24
    layer_slice = slice(6, 22) if len(k_list) >= 22 else slice(None)
    K_int = torch.stack(k_list[layer_slice], dim=0).to(score_device, dtype=torch.float32)     # [L_used, H, L_int, D]
    V_int = torch.stack(v_list[layer_slice], dim=0).to(score_device, dtype=torch.float32)     # [L_used, H, L_int, D]
    K_rec = torch.stack(k_rec_list[layer_slice], dim=0).to(score_device, dtype=torch.float32) # [L_used, H, L_rec, D]

    # Signal 1: Key Novelty = 1 - cos(K_int, mean(K_int)) -> suppresses repeated filler tokens
    K_int_unit = F.normalize(K_int, p=2, dim=-1, eps=1e-6)
    K_int_mean = F.normalize(K_int_unit.mean(dim=2, keepdim=True), p=2, dim=-1, eps=1e-6)
    novelty = (1.0 - (K_int_unit * K_int_mean).sum(dim=-1)).clamp(min=0.0).mean(dim=(0, 1))  # [L_int]
    novelty = novelty / (novelty.max() + 1e-6)

    # Signal 2: Value L2-norm across heads and layers -> [L_int]
    v_norm = torch.norm(V_int, p=2, dim=-1).mean(dim=(0, 1))
    v_norm = v_norm / (v_norm.max() + 1e-6)

    # Signal 3: Alignment with question tokens in the recent window -> [L_int]
    K_rec_mean = F.normalize(K_rec.mean(dim=2, keepdim=True), p=2, dim=-1, eps=1e-6)
    k_sim = (K_int_unit * K_rec_mean).sum(dim=-1).clamp(min=0.0).mean(dim=(0, 1))
    k_sim = k_sim / (k_sim.max() + 1e-6)

    token_scores = torch.nan_to_num(0.55 * novelty + 0.25 * v_norm + 0.20 * k_sim, nan=0.0, posinf=0.0, neginf=0.0)

    # If the chat system header (tokens 4..28) is inside the interior when length > 100,
    # damp the static system-prompt header tokens (indices 4..27) so they don't steal middle spans
    if length > 100 and interior_len > 32:
        sys_header_end = min(24, interior_len // 4)
        token_scores[:sys_header_end] *= 0.25

    span_scores = F.avg_pool1d(
        token_scores.view(1, 1, -1), kernel_size=span_size, stride=1
    ).view(-1)

    selected_offsets = _select_k_non_overlapping_spans_dp(
        span_scores.detach().cpu().tolist(), k_spans=num_mid_spans, span_size=span_size
    )

    mid_indices = []
    for offset in selected_offsets:
        abs_start = interior_start + offset
        mid_indices.extend(range(abs_start, abs_start + span_size))

    indices = (
        list(range(start_keep))
        + mid_indices
        + list(range(length - recent_keep, length))
    )
    assert len(indices) == TOTAL_KEEP and len(set(indices)) == TOTAL_KEEP and indices == sorted(indices)
    return indices


# 1. Verify structural invariants still pass with the updated hybrid selector
run_structural_invariant_tests(model, tokenizer)

# 2. Re-run the 5-depth smoke test with default continuation vs. preserve_true_position_ids=True
print("\n=== Running 5-Depth Smoke Test (preserve_true_position_ids=False) ===")
smoke_df_default = run_five_depth_smoke_test(model, tokenizer, num_runs=2, warmup_runs=1)

print("\n=== Diagnostic: Hybrid Policy with preserve_true_position_ids=True ===")
for depth_label, ratio, expected_id in [
    ("Start (~5%)", 0.05, "RG-101-A"),
    ("Early-Mid (~25%)", 0.25, "RG-205-B"),
    ("Exact-Mid (~50%)", 0.50, "RG-500-M"),
    ("Late-Mid (~75%)", 0.75, "RG-750-L"),
    ("Recent (~92%)", 0.92, "RG-920-Z"),
]:
    fact = f"IMPORTANT RECORD: The assigned project identifier is {expected_id}."
    prompt, meta = build_prompt_with_fact_at_depth(
        tokenizer, fact, "What is the assigned project identifier?",
        ratio, approx_context_tokens=240, template_key="telemetry",
        expected_identifier=expected_id, return_metadata=True
    )
    res_true_pos = generate_from_compressed_cache(
        model, tokenizer, prompt, policy="hybrid_span_kv",
        preserve_true_position_ids=True, id_token_indices=meta["id_token_indices"],
        num_runs=1, warmup_runs=0
    )
    print(
        f"{depth_label:<17} | ID_Span={meta['id_token_span']:<8} | "
        f"ID_Kept={res_true_pos['id_tokens_retained_ratio']:<4} | "
        f"Pass={int(check_exact_fact_match(res_true_pos['answer'], expected_id))} | "
        f"Select_ms={res_true_pos['selector_ms']:.2f} | Ans={res_true_pos['answer']}"
    )

In [ ]:
# ============================================================================
# Fixed & Verified Diagnostic Cell:
# - Strictly preserves start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16 (8 + 24 + 16 = 48)
# - Passes all Stage 1A & Stage 1B assertions (idx[:8], idx[8:32] 3x8 contiguous, idx[32:])
# - Uses vectorized Key Novelty + Value L2-norm so repeated filler is suppressed
# ============================================================================

@torch.no_grad()
def select_hybrid_span_kv_indices(
    cache, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16
):
    """
    Strictly adheres to spec:
    - 8 starting positions (0..7)
    - 3 non-overlapping contiguous 8-position interior spans (indices 8..31 in output)
    - 16 recent positions (length-16..length-1)
    Total budget = 8 + (3 * 8) + 16 = 48 positions.
    """
    total_budget = start_keep + (num_mid_spans * span_size) + recent_keep
    assert total_budget == TOTAL_KEEP, f"Budget must be {TOTAL_KEEP}, got {total_budget}"

    length, score_device = _get_cache_length_and_device(cache)
    if length <= 0:
        return []
    if length <= total_budget:
        return list(range(length))

    interior_start = start_keep
    interior_end = length - recent_keep
    interior_len = interior_end - interior_start  # >= 25 for all length >= 49

    k_list, v_list, k_rec_list = [], [], []
    for layer in cache.layers:
        if getattr(layer, "keys", None) is None or getattr(layer, "values", None) is None:
            continue
        if layer.keys.shape[2] != length or layer.values.shape[2] != length:
            raise ValueError("Mismatched sequence length across cache layers.")
        k_list.append(layer.keys[0, :, interior_start:interior_end, :])
        v_list.append(layer.values[0, :, interior_start:interior_end, :])
        k_rec_list.append(layer.keys[0, :, interior_end:length, :])

    if len(k_list) == 0:
        return select_multi_window_position_indices(
            length, start_keep=start_keep, num_mid_windows=num_mid_spans,
            mid_window_size=span_size, recent_keep=recent_keep
        )

    # Stack across layers in one tensor: [L_used, H, L_int, D]
    layer_slice = slice(6, 22) if len(k_list) >= 22 else slice(None)
    K_int = torch.stack(k_list[layer_slice], dim=0).to(score_device, dtype=torch.float32)
    V_int = torch.stack(v_list[layer_slice], dim=0).to(score_device, dtype=torch.float32)
    K_rec = torch.stack(k_rec_list[layer_slice], dim=0).to(score_device, dtype=torch.float32)

    # Signal 1: Key Novelty = 1 - cos(K_int, mean(K_int)) -> suppresses repeated filler
    K_int_unit = F.normalize(K_int, p=2, dim=-1, eps=1e-6)
    K_int_mean = F.normalize(K_int_unit.mean(dim=2, keepdim=True), p=2, dim=-1, eps=1e-6)
    novelty = (1.0 - (K_int_unit * K_int_mean).sum(dim=-1)).clamp(min=0.0).mean(dim=(0, 1))
    n_max = novelty.max()
    novelty = torch.where(n_max > 1e-6, novelty / (n_max + 1e-6), torch.zeros_like(novelty))

    # Signal 2: Value L2-norm across heads and layers
    v_norm = torch.norm(V_int, p=2, dim=-1).mean(dim=(0, 1))
    v_max = v_norm.max()
    v_norm = torch.where(v_max > 1e-6, v_norm / (v_max + 1e-6), torch.zeros_like(v_norm))

    # Signal 3: Cosine similarity to recent window keys
    K_rec_mean = F.normalize(K_rec.mean(dim=2, keepdim=True), p=2, dim=-1, eps=1e-6)
    k_sim = (K_int_unit * K_rec_mean).sum(dim=-1).clamp(min=0.0).mean(dim=(0, 1))
    s_max = k_sim.max()
    k_sim = torch.where(s_max > 1e-6, k_sim / (s_max + 1e-6), torch.zeros_like(k_sim))

    token_scores = torch.nan_to_num(
        0.60 * novelty + 0.25 * v_norm + 0.15 * k_sim, nan=0.0, posinf=0.0, neginf=0.0
    )

    # Damp chat-template system header (indices 8..28) and trailing question prefix (last 10 interior tokens)
    # when context is long (> 120 tokens), so static chat markers don't steal the 3 middle spans
    if length > 120 and interior_len > 40:
        token_scores[:21] *= 0.20
        token_scores[-10:] *= 0.20

    span_scores = F.avg_pool1d(
        token_scores.view(1, 1, -1), kernel_size=span_size, stride=1
    ).view(-1)

    selected_offsets = _select_k_non_overlapping_spans_dp(
        span_scores.detach().cpu().tolist(), k_spans=num_mid_spans, span_size=span_size
    )

    mid_indices = []
    for offset in selected_offsets:
        abs_start = interior_start + offset
        mid_indices.extend(range(abs_start, abs_start + span_size))

    indices = (
        list(range(start_keep))
        + mid_indices
        + list(range(length - recent_keep, length))
    )

    # Double-check all structural invariants inside the selector itself
    assert len(indices) == TOTAL_KEEP
    assert len(set(indices)) == TOTAL_KEEP
    assert indices == sorted(indices)
    assert indices[:8] == list(range(8))
    assert indices[32:] == list(range(length - 16, length))
    for s in range(3):
        sp = indices[8 + s * 8 : 8 + (s + 1) * 8]
        assert sp == list(range(sp[0], sp[0] + 8))

    return indices


# 1. Re-verify Stage 1A & Stage 1B structural invariants
run_structural_invariant_tests(model, tokenizer)

# 2. Re-run 5-Depth Smoke Test
smoke_df = run_five_depth_smoke_test(model, tokenizer, num_runs=2, warmup_runs=1)

# 3. Inspect exact middle spans selected by hybrid_span_kv and test preserve_true_position_ids=True
print("\n=== Diagnostic: Hybrid Selected Spans & True Position-ID Continuation ===")
for depth_label, ratio, expected_id in [
    ("Start (~5%)", 0.05, "RG-101-A"),
    ("Early-Mid (~25%)", 0.25, "RG-205-B"),
    ("Exact-Mid (~50%)", 0.50, "RG-500-M"),
    ("Late-Mid (~75%)", 0.75, "RG-750-L"),
    ("Recent (~92%)", 0.92, "RG-920-Z"),
]:
    fact = f"IMPORTANT RECORD: The assigned project identifier is {expected_id}."
    prompt, meta = build_prompt_with_fact_at_depth(
        tokenizer, fact, "What is the assigned project identifier?",
        ratio, approx_context_tokens=240, template_key="telemetry",
        expected_identifier=expected_id, return_metadata=True
    )
    res_def = generate_from_compressed_cache(
        model, tokenizer, prompt, policy="hybrid_span_kv",
        preserve_true_position_ids=False, id_token_indices=meta["id_token_indices"],
        num_runs=1, warmup_runs=0
    )
    res_pos = generate_from_compressed_cache(
        model, tokenizer, prompt, policy="hybrid_span_kv",
        preserve_true_position_ids=True, id_token_indices=meta["id_token_indices"],
        num_runs=1, warmup_runs=0
    )
    idx = res_def["retained_indices"]
    spans_str = f"[{idx[8]}..{idx[15]}], [{idx[16]}..{idx[23]}], [{idx[24]}..{idx[31]}]"
    print(
        f"{depth_label:<16} | Fact={meta['id_token_span']:<8} | "
        f"ChosenMidSpans={spans_str:<32} | ID_Kept={res_def['id_tokens_retained_ratio']:<4} | "
        f"Def_Ans='{res_def['answer'][:25]}' | TruePos_Ans='{res_pos['answer'][:25]}'"
    )

In [ ]:
def _select_k_non_overlapping_spans_dp(span_scores, k_spans, span_size):
    """span_scores[i] = score of the span STARTING at offset i. Returns ascending start offsets."""
    n = len(span_scores)
    if k_spans == 0:
        return []
    if span_size < 1 or k_spans < 0:
        raise ValueError("bad args")
    if n < (k_spans - 1) * span_size + 1:
        raise ValueError(f"infeasible: {n} span starts cannot fit {k_spans} spans of {span_size}")
    NEG = float("-inf")
    best = [[NEG] * (n + 1) for _ in range(k_spans + 1)]
    for i in range(n + 1):
        best[0][i] = 0.0
    for j in range(1, k_spans + 1):
        for i in range(n - 1, -1, -1):
            take = span_scores[i] + best[j - 1][min(i + span_size, n)]
            best[j][i] = max(best[j][i + 1], take)
    out, i, j = [], 0, k_spans
    while j > 0:
        take = span_scores[i] + best[j - 1][min(i + span_size, n)]
        if take >= best[j][i + 1]:
            out.append(i); j -= 1; i = min(i + span_size, n)
        else:
            i += 1
    return out

In [ ]:
import time
import torch
import torch.nn.functional as F

def _norm01(x):
    m = x.max()
    return x / m if m > 1e-6 else torch.zeros_like(x)

@torch.no_grad()
def select_hybrid_span_kv_indices(
    cache, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16,
    damp_ranges=None, damp_factor=0.2, timing_out=None,
):
    """damp_ranges: optional list of ABSOLUTE (start, end) token ranges to down-weight
    (e.g. chat header). Default None = no damping."""
    budget = start_keep + num_mid_spans * span_size + recent_keep
    length, dev = _get_cache_length_and_device(cache)
    if length <= 0:
        return []
    if length <= budget:
        return list(range(length))

    lo_i, hi_i = start_keep, length - recent_keep      # interior [lo_i, hi_i)
    interior_len = hi_i - lo_i
    if interior_len < num_mid_spans * span_size:
        raise ValueError("interior too short for requested spans")

    layers = [l for l in cache.layers
              if getattr(l, "keys", None) is not None and getattr(l, "values", None) is not None]
    if not layers:
        raise ValueError("cache has no populated layers")
    for l in layers:
        if l.keys.shape[2] != length or l.values.shape[2] != length:
            raise ValueError("mismatched sequence length across layers")
        if l.values.shape[0] != 1:
            raise ValueError("batch size must be 1")

    dev = torch.device(dev)
    q = len(layers) // 4
    use = layers[q: max(q + 1, 3 * len(layers) // 4)]

    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    # Values are NOT rotated by RoPE, so content scoring uses values (keys are position-rotated).
    novelty = torch.zeros(interior_len, device=dev, dtype=torch.float32)
    vnorm = torch.zeros_like(novelty)
    for l in use:                                   # one layer at a time: small temporaries
        V = l.values[0, :, lo_i:hi_i, :].to(dev, torch.float32)   # [H, Li, D]
        Vu = F.normalize(V, dim=-1, eps=1e-6)
        mean = F.normalize(Vu.mean(1, keepdim=True), dim=-1, eps=1e-6)
        novelty += (1.0 - (Vu * mean).sum(-1)).clamp(min=0).mean(0)
        vnorm += V.norm(dim=-1).mean(0)
    scores = torch.nan_to_num(0.7 * _norm01(novelty / len(use)) + 0.3 * _norm01(vnorm / len(use)),
                              nan=0.0, posinf=0.0, neginf=0.0)

    for a, b in (damp_ranges or []):
        a2, b2 = max(a, lo_i) - lo_i, min(b, hi_i) - lo_i
        if b2 > a2:
            scores[a2:b2] *= damp_factor

    span_scores = F.avg_pool1d(scores.view(1, 1, -1), kernel_size=span_size, stride=1).view(-1)
    offsets = _select_k_non_overlapping_spans_dp(
        span_scores.cpu().tolist(), num_mid_spans, span_size)

    if dev.type == "cuda":
        torch.cuda.synchronize()
    if timing_out is not None:
        timing_out["selector_s"] = time.perf_counter() - t0

    mid = [lo_i + o + t for o in offsets for t in range(span_size)]
    indices = list(range(start_keep)) + mid + list(range(length - recent_keep, length))

    assert len(indices) == budget == len(set(indices))
    assert indices == sorted(indices) and indices[-1] == length - 1
    assert indices[:start_keep] == list(range(start_keep))
    assert indices[-recent_keep:] == list(range(length - recent_keep, length))
    for s in range(num_mid_spans):
        sp = indices[start_keep + s * span_size: start_keep + (s + 1) * span_size]
        assert sp == list(range(sp[0], sp[0] + span_size))
    return indices

In [ ]:
def template_damp_ranges(tokenizer, prompt, question):
    """Header = tokens before the user turn; question = tokens from the last occurrence of `question`."""
    offs = tokenizer(prompt, return_offsets_mapping=True).offset_mapping
    marker = "<|im_start|>user\n"
    h = prompt.find(marker)
    header_end = sum(1 for s, e in offs if e <= h + len(marker)) if h >= 0 else 0
    qc = prompt.rfind(question)
    q_start = next((i for i, (s, e) in enumerate(offs) if s >= qc), len(offs)) if qc >= 0 else len(offs)
    return [(0, header_end), (q_start, len(offs))]

In [ ]:
import copy, torch, torch.nn.functional as F

@torch.no_grad()
def _compress_by_indices(cache, idx):
    out = copy.deepcopy(cache)
    t = torch.tensor(idx, dtype=torch.long)
    for l in out.layers:
        if getattr(l, "keys", None) is None:
            continue
        ti = t.to(l.keys.device)
        l.keys = l.keys.index_select(2, ti).contiguous()
        l.values = l.values.index_select(2, ti).contiguous()
    return out

@torch.no_grad()
def _continue(model, tokenizer, cache, orig_len, q_ids, mode, n_new=12):
    """mode='default': let HF derive positions. mode='absolute': original absolute position_ids.
    cache_position is never passed; HF derives it from the cache's stored length."""
    dev = q_ids.device
    def step(ids, pos0):
        r = cache.layers[0].keys.shape[2]
        kw = dict(input_ids=ids, past_key_values=cache, use_cache=True,
                  attention_mask=torch.ones(1, r + ids.shape[1], dtype=torch.long, device=dev))
        if mode == "absolute":
            kw["position_ids"] = torch.arange(pos0, pos0 + ids.shape[1], device=dev).unsqueeze(0)
        return model(**kw)
    q = q_ids.shape[1]
    out = step(q_ids, orig_len)
    cache = out.past_key_values
    first = out.logits[0, -1].float().cpu()
    logits, toks = first, []
    for s in range(n_new):
        t = logits.argmax().item()
        if t == tokenizer.eos_token_id:
            break
        toks.append(t)
        out = step(torch.tensor([[t]], device=dev), orig_len + q + s)
        cache = out.past_key_values
        logits = out.logits[0, -1].float().cpu()
    return first, tokenizer.decode(toks, skip_special_tokens=True)

def position_diagnostic(model, tokenizer, device, context, question, expected, selector):
    ctx_ids = tokenizer(context, return_tensors="pt").input_ids.to(device)
    q_ids = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                      return_tensors="pt").input_ids.to(device)
    with torch.inference_mode():
        base = model(input_ids=ctx_ids, use_cache=True).past_key_values
    L = base.layers[0].keys.shape[2]
    idx = selector(base)
    ident = list(range(L))
    runs = {
        "full":                (ident, "default"),
        "identity+default":    (ident, "default"),
        "identity+absolute":   (ident, "absolute"),
        "compressed+default":  (idx,   "default"),
        "compressed+absolute": (idx,   "absolute"),
    }
    res = {}
    for name, (ix, mode) in runs.items():
        with torch.inference_mode():
            res[name] = _continue(model, tokenizer, _compress_by_indices(base, ix), L, q_ids, mode)

    ref_logits, ref_ans = res["full"]
    ref_lp = F.log_softmax(ref_logits, -1)
    print(f"context={L} tokens, retained={len(idx)}, expected={expected!r}")
    print(f"{'run':20} | max|dlogit| | KL(full||run) | top1 same | exact | answer")
    for name, (lg, ans) in res.items():
        d = (lg - ref_logits).abs().max().item()
        kl = F.kl_div(F.log_softmax(lg, -1), ref_lp, log_target=True, reduction="sum").item()
        print(f"{name:20} | {d:11.4f} | {kl:13.5f} | "
              f"{str(lg.argmax().item() == ref_logits.argmax().item()):9} | "
              f"{str(expected.lower() in ans.lower()):5} | {ans!r}")

    # Implementation-error checks: no tokens removed, so results must match the full run.
    for name in ("identity+default", "identity+absolute"):
        d = (res[name][0] - ref_logits).abs().max().item()
        assert d < 1e-3, f"{name} differs from full by {d}: mask/position/cache handling bug"
    print("Identity controls PASS (pipeline reproduces full-cache logits).")

In [ ]:
context = """
IMPORTANT RECORD: The assigned project identifier is RG-500-M.
""" * 15

question = "What is the assigned project identifier?"
expected = "RG-500-M"

position_diagnostic(
    model=model,
    tokenizer=tokenizer,
    device=device,
    context=context,
    question=question,
    expected=expected,
    selector=lambda cache: select_hybrid_span_kv_indices(cache),
)

In [ ]:
# STAGE 2: POSITION-HANDLING DIAGNOSTIC
# Uses the existing model, tokenizer, device, selector, and diagnostic function.

import gc
import torch

test_cases = [
    ("Start",       0.05, "RG-101-A"),
    ("Early-Mid",   0.25, "RG-205-B"),
    ("Exact-Mid",   0.50, "RG-500-M"),
    ("Late-Mid",    0.75, "RG-750-L"),
    ("Recent",      0.92, "RG-920-Z"),
]

filler = (
    "The system records routine telemetry and project status information. "
    "The service monitors operational metrics and maintains project records. "
)

for label, ratio, expected in test_cases:
    # Place the fact among repeated filler blocks.
    blocks = [filler] * 45
    insert_at = round(ratio * len(blocks))
    fact = (
        f"IMPORTANT RECORD: The assigned project identifier is {expected}. "
    )
    blocks.insert(insert_at, fact)
    context = "".join(blocks)

    print(f"\n{'=' * 24} {label} {'=' * 24}")
    position_diagnostic(
        model=model,
        tokenizer=tokenizer,
        device=device,
        context=context,
        question="What is the assigned project identifier?",
        expected=expected,
        selector=lambda cache: select_hybrid_span_kv_indices(cache),
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# STAGE 3: RECALL SMOKE TEST
smoke_df = run_five_depth_smoke_test(
    model,
    tokenizer,
    num_runs=2,
    warmup_runs=1,
)

display(smoke_df)

In [ ]:
# STAGE 4: EXPANDED EVALUATION
# Run only after Stage 2 and Stage 3 have been inspected.

expanded_df = run_expanded_evaluation(
    model,
    tokenizer,
)

display(expanded_df)

In [ ]:
# STAGE 5: STORAGE ARITHMETIC CHECK

def kv_storage_bytes(cache):
    total = 0
    for layer in cache.layers:
        k = getattr(layer, "keys", None)
        v = getattr(layer, "values", None)
        if k is not None and v is not None:
            total += k.numel() * k.element_size()
            total += v.numel() * v.element_size()
    return total

# Use a populated cache from your existing notebook.
# Example: original_cache and compressed_cache must already exist.
original_bytes = kv_storage_bytes(original_cache)
compressed_bytes = kv_storage_bytes(compressed_cache)

assert original_bytes > 0
assert compressed_bytes > 0
assert compressed_bytes <= original_bytes

reduction_pct = 100 * (
    1 - compressed_bytes / original_bytes
)

print(f"Original KV storage:   {original_bytes / 2**20:.4f} MiB")
print(f"Compressed KV storage: {compressed_bytes / 2**20:.4f} MiB")
print(f"KV storage reduction:  {reduction_pct:.2f}%")

In [ ]:
# STAGE 5: STORAGE ARITHMETIC CHECK
import torch

def kv_storage_bytes(cache):
    total = 0
    for layer in cache.layers:
        k = getattr(layer, "keys", None)
        v = getattr(layer, "values", None)
        if k is not None and v is not None:
            total += k.numel() * k.element_size()
            total += v.numel() * v.element_size()
    return total

# Build a populated cache (needs model, tokenizer, device from earlier cells)
FILLER = "The quarterly summary notes that routine operations continued as planned. "
prompt = FILLER * 20 + "Project code: ORCHID-7319. " + FILLER * 20
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.inference_mode():
    original_cache = model(input_ids=ids, use_cache=True).past_key_values

# Any policy works here; this uses the position-aware function you defined earlier
compressed_cache = compress_cache_position_aware(original_cache)

orig_len = original_cache.layers[0].keys.shape[2]
comp_len = compressed_cache.layers[0].keys.shape[2]
assert comp_len < orig_len, f"No compression happened ({orig_len} -> {comp_len})"

original_bytes = kv_storage_bytes(original_cache)
compressed_bytes = kv_storage_bytes(compressed_cache)

assert original_bytes > 0
assert compressed_bytes > 0
assert compressed_bytes < original_bytes

# Bytes must scale with retained length (same layers, heads, dtype)
expected_ratio = comp_len / orig_len
actual_ratio = compressed_bytes / original_bytes
assert abs(expected_ratio - actual_ratio) < 1e-6, "Storage doesn't match token ratio"

reduction_pct = 100 * (1 - compressed_bytes / original_bytes)

print(f"Tokens:                {orig_len} -> {comp_len}")
print(f"Original KV storage:   {original_bytes / 2**20:.4f} MiB")
print(f"Compressed KV storage: {compressed_bytes / 2**20:.4f} MiB")
print(f"KV storage reduction:  {reduction_pct:.2f}%")

In [ ]:
# STAGE 5: STORAGE ARITHMETIC CHECK

def kv_storage_bytes(cache):
    total = 0
    for layer in cache.layers:
        k = getattr(layer, "keys", None)
        v = getattr(layer, "values", None)
        if k is not None and v is not None:
            total += k.numel() * k.element_size()
            total += v.numel() * v.element_size()
    return total

# --- Setup: create the populated caches this check needs ---
import torch
FILLER = "The quarterly summary notes that routine operations continued as planned. "
prompt = FILLER * 20 + "Project code: ORCHID-7319. " + FILLER * 20
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.inference_mode():
    original_cache = model(input_ids=ids, use_cache=True).past_key_values
compressed_cache = compress_cache_position_aware(original_cache)
# -----------------------------------------------------------

original_bytes = kv_storage_bytes(original_cache)
compressed_bytes = kv_storage_bytes(compressed_cache)

assert original_bytes > 0
assert compressed_bytes > 0
assert compressed_bytes <= original_bytes

reduction_pct = 100 * (
    1 - compressed_bytes / original_bytes
)

print(f"Original KV storage:   {original_bytes / 2**20:.4f} MiB")
print(f"Compressed KV storage: {compressed_bytes / 2**20:.4f} MiB")
print(f"KV storage reduction:  {reduction_pct:.2f}%")

In [ ]:
# Controlled diagnostic: same context, question, and selector.
# position_diagnostic compares full-cache, identity-cache,
# compressed-default, and compressed-absolute continuations.

filler = (
    "The system records routine telemetry and project status "
    "information. The service monitors operational metrics "
    "and maintains project records. "
)

blocks = [filler] * 45
blocks.insert(
    22,
    "IMPORTANT RECORD: The assigned project identifier is RG-500-M. "
)

context = "".join(blocks)

position_diagnostic(
    model=model,
    tokenizer=tokenizer,
    device=device,
    context=context,
    question="What is the assigned project identifier?",
    expected="RG-500-M",
    selector=lambda cache: select_hybrid_span_kv_indices(cache),
)

In [ ]:
import time, torch

FILLER = "The quarterly summary notes that routine operations continued as planned. "

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def id_token_span(tokenizer, ctx, ident):
    enc = tokenizer(ctx, return_offsets_mapping=True)
    c0 = ctx.index(ident); c1 = c0 + len(ident)
    toks = [i for i, (s, e) in enumerate(enc["offset_mapping"]) if e > s and s < c1 and e > c0]
    return toks, len(enc["input_ids"])

def build_context_at_depth(tokenizer, ident, target, n_tokens=240):
    """Search padding so the MEASURED identifier-token midpoint / length is closest to target."""
    fact = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    ft = len(tokenizer(FILLER).input_ids)
    best = None
    for a in range(0, n_tokens // ft + 1):
        for b in range(0, 12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((n_tokens - used) / ft))
            ctx = before + fact + " " + FILLER * c
            toks, L = id_token_span(tokenizer, ctx, ident)
            depth = (toks[0] + toks[-1]) / 2 / L
            key = (abs(depth - target), abs(L - n_tokens))
            if best is None or key < best[0]:
                best = (key, ctx, depth, L)
    return best[1], best[2], best[3]

def kv_bytes(cache):
    return sum(l.keys.numel() * l.keys.element_size() + l.values.numel() * l.values.element_size()
               for l in cache.layers if getattr(l, "keys", None) is not None)

@torch.no_grad()
def eval_case(model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16):
    """policy_fn=None -> full cache. mode in {'default','absolute'}."""
    toks, L = id_token_span(tokenizer, ctx, ident)
    ids = tokenizer(ctx, return_tensors="pt").input_ids.to(device)
    q_ids = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                      return_tensors="pt").input_ids.to(device)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        base_mem = torch.cuda.memory_allocated()

    _sync(); t = time.perf_counter()
    cache = model(input_ids=ids, use_cache=True).past_key_values
    _sync(); prefill_s = time.perf_counter() - t

    t = time.perf_counter()
    idx = list(range(L)) if policy_fn is None else policy_fn(cache)
    _sync(); selector_s = time.perf_counter() - t

    t = time.perf_counter()
    work = _compress_by_indices(cache, idx)     # NOTE: includes deepcopy; eval artifact
    _sync(); compress_s = time.perf_counter() - t

    t = time.perf_counter()
    _, ans = _continue(model, tokenizer, work, L, q_ids, mode, n_new=n_new)
    _sync(); continue_s = time.perf_counter() - t   # question prefill + decode

    kept = set(idx)
    peak = (torch.cuda.max_memory_allocated() - base_mem) / 2**20 if torch.cuda.is_available() else float("nan")
    return dict(
        L=L, depth=(toks[0] + toks[-1]) / 2 / L, id_kept=sum(i in kept for i in toks) / len(toks),
        exact=ident in ans, answer=ans, kv_mib=kv_bytes(work) / 2**20,
        prefill_s=prefill_s, selector_s=selector_s, compress_s=compress_s, continue_s=continue_s,
        peak_delta_mib=peak, idx=idx, id_toks=toks,
    )

# ---- Driver ----
policies = {
    "full": None,
    "position_aware": lambda c: select_position_aware_indices(c.layers[0].keys.shape[2]),
    "hybrid_span_kv": lambda c: select_hybrid_span_kv_indices(c),
    # add your other policies here as callables: cache -> list of indices
}
question = "What is the assigned project identifier?"
cases = [(0.05, "RG-101-A"), (0.25, "RG-205-B"), (0.50, "RG-500-M"), (0.75, "RG-750-L"), (0.92, "RG-920-Z")]

_ = eval_case(model, tokenizer, device, *[build_context_at_depth(tokenizer, "RG-500-M", 0.5)[0]],
              question, "RG-500-M", None, "default")          # warm-up, discarded

print(f"{'target':>6} {'actual':>6} {'policy':15} {'mode':8} {'idKept':>6} {'exact':>5} "
      f"{'prefill':>7} {'select':>6} {'compr':>6} {'cont':>6} {'KV MiB':>7} {'peakΔ':>7} | answer")
for target, ident in cases:
    ctx, depth, L = build_context_at_depth(tokenizer, ident, target)
   for pname, fn in policies.items():
    # Test both modes for full cache too, as a control.
    for mode in ("default", "absolute"):
            r = eval_case(model, tokenizer, device, ctx, question, ident, fn, mode)
            print(f"{target:6.2f} {r['depth']:6.2f} {pname:15} {mode:8} {r['id_kept']:6.2f} "
                  f"{str(r['exact']):>5} {r['prefill_s']:7.3f} {r['selector_s']:6.3f} "
                  f"{r['compress_s']:6.3f} {r['continue_s']:6.3f} {r['kv_mib']:7.4f} "
                  f"{r['peak_delta_mib']:7.1f} | {r['answer'][:28]!r}")

In [ ]:
import time, torch

FILLER = "The quarterly summary notes that routine operations continued as planned. "

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def id_token_span(tokenizer, ctx, ident):
    enc = tokenizer(ctx, return_offsets_mapping=True)
    c0 = ctx.index(ident); c1 = c0 + len(ident)
    toks = [i for i, (s, e) in enumerate(enc["offset_mapping"]) if e > s and s < c1 and e > c0]
    return toks, len(enc["input_ids"])

def build_context_at_depth(tokenizer, ident, target, n_tokens=240):
    """Search padding so the MEASURED identifier-token midpoint / length is closest to target."""
    fact = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    ft = len(tokenizer(FILLER).input_ids)
    best = None
    for a in range(0, n_tokens // ft + 1):
        for b in range(0, 12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((n_tokens - used) / ft))
            ctx = before + fact + " " + FILLER * c
            toks, L = id_token_span(tokenizer, ctx, ident)
            depth = (toks[0] + toks[-1]) / 2 / L
            key = (abs(depth - target), abs(L - n_tokens))
            if best is None or key < best[0]:
                best = (key, ctx, depth, L)
    return best[1], best[2], best[3]

def kv_bytes(cache):
    return sum(l.keys.numel() * l.keys.element_size() + l.values.numel() * l.values.element_size()
               for l in cache.layers if getattr(l, "keys", None) is not None)

@torch.no_grad()
def eval_case(model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16):
    """policy_fn=None -> full cache. mode in {'default','absolute'}."""
    toks, L = id_token_span(tokenizer, ctx, ident)
    ids = tokenizer(ctx, return_tensors="pt").input_ids.to(device)
    q_ids = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                      return_tensors="pt").input_ids.to(device)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        base_mem = torch.cuda.memory_allocated()

    _sync(); t = time.perf_counter()
    cache = model(input_ids=ids, use_cache=True).past_key_values
    _sync(); prefill_s = time.perf_counter() - t

    t = time.perf_counter()
    idx = list(range(L)) if policy_fn is None else policy_fn(cache)
    _sync(); selector_s = time.perf_counter() - t

    t = time.perf_counter()
    work = _compress_by_indices(cache, idx)     # NOTE: includes deepcopy; eval artifact
    _sync(); compress_s = time.perf_counter() - t

    t = time.perf_counter()
    _, ans = _continue(model, tokenizer, work, L, q_ids, mode, n_new=n_new)
    _sync(); continue_s = time.perf_counter() - t   # question prefill + decode

    kept = set(idx)
    peak = (torch.cuda.max_memory_allocated() - base_mem) / 2**20 if torch.cuda.is_available() else float("nan")
    return dict(
        L=L, depth=(toks[0] + toks[-1]) / 2 / L, id_kept=sum(i in kept for i in toks) / len(toks),
        exact=ident in ans, answer=ans, kv_mib=kv_bytes(work) / 2**20,
        prefill_s=prefill_s, selector_s=selector_s, compress_s=compress_s, continue_s=continue_s,
        peak_delta_mib=peak, idx=idx, id_toks=toks,
    )

# ---- Driver ----
policies = {
    "full": None,
    "position_aware": lambda c: select_position_aware_indices(c.layers[0].keys.shape[2]),
    "hybrid_span_kv": lambda c: select_hybrid_span_kv_indices(c),
    # add your other policies here as callables: cache -> list of indices
}
question = "What is the assigned project identifier?"
cases = [(0.05, "RG-101-A"), (0.25, "RG-205-B"), (0.50, "RG-500-M"), (0.75, "RG-750-L"), (0.92, "RG-920-Z")]

_ = eval_case(model, tokenizer, device, *[build_context_at_depth(tokenizer, "RG-500-M", 0.5)[0]],
              question, "RG-500-M", None, "default")          # warm-up, discarded

print(f"{'target':>6} {'actual':>6} {'policy':15} {'mode':8} {'idKept':>6} {'exact':>5} "
      f"{'prefill':>7} {'select':>6} {'compr':>6} {'cont':>6} {'KV MiB':>7} {'peakΔ':>7} | answer")
for target, ident in cases:
    ctx, depth, L = build_context_at_depth(tokenizer, ident, target)
    for pname, fn in policies.items():
        for mode in (("default",) if fn is None else ("default", "absolute")):
            r = eval_case(model, tokenizer, device, ctx, question, ident, fn, mode)
            print(f"{target:6.2f} {r['depth']:6.2f} {pname:15} {mode:8} {r['id_kept']:6.2f} "
                  f"{str(r['exact']):>5} {r['prefill_s']:7.3f} {r['selector_s']:6.3f} "
                  f"{r['compress_s']:6.3f} {r['continue_s']:6.3f} {r['kv_mib']:7.4f} "
                  f"{r['peak_delta_mib']:7.1f} | {r['answer'][:28]!r}")

In [ ]:
import random

WORDS = "alpha harbor copper lantern meadow quartz ridge summit tundra violet willow zenith".split()

def _sent(rng):
    return (f"Report {rng.randint(100,999)}: unit {rng.choice(WORDS)} logged value "
            f"{rng.randint(10,9999)} at site {rng.choice(WORDS)}-{rng.randint(1,99)}.")

def build_hard_context(tokenizer, ident, target, seed, n_sent=14, n_decoys=3):
    rng = random.Random(seed)
    sents = [_sent(rng) for _ in range(n_sent)]
    fpos = min(n_sent - 1, int(target * n_sent))
    others = [i for i in range(n_sent) if i != fpos]
    for i in rng.sample(others, n_decoys):
        d = f"RG-{rng.randint(100,999)}-{rng.choice('ABCDEFGHJKLMNPQRSTUVWXYZ')}"
        sents[i] = f"DECOY RECORD: The archived project identifier is {d}."
    sents[fpos] = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    return " ".join(sents)

results = {}
for seed in range(4):
    for target, ident in cases:                       # same five depths/identifiers as before
        ctx = build_hard_context(tokenizer, ident, target, seed)
        for pname, fn in policies.items():
            r = eval_case(model, tokenizer, device, ctx, question, ident, fn,
                          "default" if fn is None else "absolute")
            results.setdefault(pname, []).append((r["exact"], r["id_kept"], r["depth"], r["L"]))

print(f"{'policy':15} | exact | mean idKept | n")
for p, rows in results.items():
    n = len(rows)
    print(f"{p:15} | {sum(r[0] for r in rows):2d}/{n} | {sum(r[1] for r in rows)/n:10.2f} | {n}")

In [ ]:
import copy, contextlib, torch
import torch.nn.functional as F

@contextlib.contextmanager
def eager_attn(model):
    old = model.config._attn_implementation
    try:
        try: model.set_attn_implementation("eager")
        except Exception: model.config._attn_implementation = "eager"
        yield
    finally:
        try: model.set_attn_implementation(old)
        except Exception: model.config._attn_implementation = old

@torch.no_grad()
def select_snap_span_indices(cache, model, q_ids, start_keep=8, num_mid_spans=3,
                             span_size=8, recent_keep=16, pool=5):
    L = cache.layers[0].keys.shape[2]
    budget = start_keep + num_mid_spans * span_size + recent_keep
    if L <= budget:
        return list(range(L))
    lo, hi = start_keep, L - recent_keep
    q, dev = q_ids.shape[1], q_ids.device

    work = copy.deepcopy(cache)          # scoring pass appends the question; keep original clean
    with eager_attn(model):
        out = model(input_ids=q_ids, past_key_values=work, use_cache=True,
                    attention_mask=torch.ones(1, L + q, dtype=torch.long, device=dev),
                    position_ids=torch.arange(L, L + q, device=dev).unsqueeze(0),
                    output_attentions=True)
    atts = out.attentions
    if atts is None or atts[0] is None:
        raise RuntimeError("No attentions returned: eager attention switch failed.")

    n = len(atts)
    score = sum(a[0, :, :, lo:hi].float().sum(dim=(0, 1)) for a in atts[n // 3:])  # [interior]
    score = F.avg_pool1d(score.view(1, 1, -1), kernel_size=pool, stride=1,
                         padding=pool // 2).view(-1)
    span_scores = F.avg_pool1d(score.view(1, 1, -1), kernel_size=span_size, stride=1).view(-1)
    offs = _select_k_non_overlapping_spans_dp(span_scores.cpu().tolist(), num_mid_spans, span_size)

    idx = (list(range(start_keep))
           + [lo + o + t for o in offs for t in range(span_size)]
           + list(range(L - recent_keep, L)))
    assert len(idx) == budget == len(set(idx)) and idx == sorted(idx)
    return idx

# Register and rerun the SAME hard benchmark
CURRENT_Q_IDS = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                          return_tensors="pt").input_ids.to(device)
policies["snap_span"] = lambda c: select_snap_span_indices(c, model, CURRENT_Q_IDS)

results = {}
for seed in range(4):
    for target, ident in cases:
        ctx = build_hard_context(tokenizer, ident, target, seed)
        for pname, fn in policies.items():
            r = eval_case(model, tokenizer, device, ctx, question, ident, fn,
                          "default" if fn is None else "absolute")
            results.setdefault(pname, []).append((r["exact"], r["id_kept"], r["depth"], r["L"]))

print(f"{'policy':15} | exact | mean idKept | n")
for p, rows in results.items():
    n = len(rows)
    print(f"{p:15} | {sum(r[0] for r in rows):2d}/{n} | {sum(r[1] for r in rows)/n:10.2f} | {n}")

In [ ]:
rows = []
for seed in range(4):
    for target, ident in cases:
        ctx = build_hard_context(tokenizer, ident, target, seed)
        r = eval_case(model, tokenizer, device, ctx, question, ident,
                      policies["snap_span"], "absolute")
        rows.append((seed, target, ident, r))

print(f"{'seed':>4} {'depth':>5} {'idKept':>6} {'exact':>5} | answer")
for seed, target, ident, r in rows:
    flag = "" if r["exact"] else "  <-- FAIL"
    print(f"{seed:4d} {r['depth']:5.2f} {r['id_kept']:6.2f} {str(r['exact']):>5} | "
          f"{r['answer'][:30]!r}{flag}")

# Mean exact by target depth
for t in sorted({c[0] for c in cases}):
    sub = [r for s, tg, i, r in rows if tg == t]
    print(f"target {t:.2f}: {sum(r['exact'] for r in sub)}/{len(sub)}")

In [ ]:
configs = [  # (span_size, num_spans, pool, layer_frac); start 8 + spans*size + recent 16 = 48
    (8, 3, 5, 0.33), (4, 6, 5, 0.33), (6, 4, 5, 0.33),
    (8, 3, 1, 0.33), (8, 3, 5, 0.50), (4, 6, 1, 0.50),
]
print(f"{'size x n':>8} {'pool':>4} {'lf':>4} | exact | idKept")
for sz, ns, pool, lf in configs:
    assert 8 + sz * ns + 16 == 48
    fn = lambda c, sz=sz, ns=ns, pool=pool, lf=lf: select_snap_span_indices(
        c, model, CURRENT_Q_IDS, num_mid_spans=ns, span_size=sz, pool=pool, layer_frac=lf)
    ex, ik, n = 0, 0.0, 0
    for seed in range(4):
        for target, ident in cases:
            ctx = build_hard_context(tokenizer, ident, target, seed)
            r = eval_case(model, tokenizer, device, ctx, question, ident, fn, "absolute")
            ex += r["exact"]; ik += r["id_kept"]; n += 1
    print(f"{sz:>3}x{ns:<4} {pool:4d} {lf:4.2f} | {ex:2d}/{n} | {ik/n:.2f}")

In [ ]:

import gc
import torch

# (span_size, num_spans, pool, layer_frac)
# Budget: 8 start + span_size*num_spans + 16 recent = 48
configs = [
    (8, 3, 5, 0.33),
    (4, 6, 5, 0.33),
    (6, 4, 5, 0.33),
    (8, 3, 1, 0.33),
    (8, 3, 5, 0.50),
    (4, 6, 1, 0.50),
]

assert all(8 + sz * ns + 16 == 48 for sz, ns, _, _ in configs)

results = []

print(
    f"{'size x n':>9} {'pool':>4} {'lf':>5} | "
    f"{'exact':>8} {'idKept':>8} {'errors':>6}"
)
print("-" * 53)

for sz, ns, pool, lf in configs:
    exact_count = 0
    id_kept_sum = 0.0
    n_success = 0
    n_errors = 0

    # Capture this configuration's parameters in the function defaults.
    def policy_fn(
        cache,
        sz=sz,
        ns=ns,
        pool=pool,
        lf=lf,
    ):
        return select_snap_span_indices(
            cache,
            model,
            CURRENT_Q_IDS,
            num_mid_spans=ns,
            span_size=sz,
            pool=pool,
            layer_frac=lf,
        )

    for seed in range(4):
        for target, ident in cases:
            try:
                ctx = build_hard_context(
                    tokenizer, ident, target, seed
                )

                r = eval_case(
                    model,
                    tokenizer,
                    device,
                    ctx,
                    question,
                    ident,
                    policy_fn,
                    "absolute",
                )

                exact_count += int(bool(r["exact"]))
                id_kept_sum += float(r["id_kept"])
                n_success += 1

            except Exception as e:
                n_errors += 1
                print(
                    f"ERROR config={(sz, ns, pool, lf)}, "
                    f"seed={seed}, id={ident}: "
                    f"{type(e).__name__}: {e}"
                )

            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    recall = exact_count / n_success if n_success else 0.0
    avg_kept = id_kept_sum / n_success if n_success else 0.0

    row = {
        "span_size": sz,
        "num_spans": ns,
        "pool": pool,
        "layer_frac": lf,
        "exact": exact_count,
        "n": n_success,
        "recall": recall,
        "id_kept": avg_kept,
        "errors": n_errors,
    }
    results.append(row)

    print(
        f"{sz:>3}x{ns:<5} {pool:>4} {lf:>5.2f} | "
        f"{exact_count:>3}/{n_success:<4} "
        f"{avg_kept:>7.2f} {n_errors:>6}"
    )

print("\nSweep complete.")


In [ ]:
!nvidia-smi

In [ ]:
import inspect

print(inspect.signature(select_snap_span_indices))
print(inspect.getsource(select_snap_span_indices))

In [ ]:

import inspect
import torch

print("Function:", inspect.signature(select_snap_span_indices))

# Use the current cache and question IDs from your notebook.
idx = select_snap_span_indices(
    cache=CURRENT_CACHE,  # replace if your cache variable has another name
    model=model,
    q_ids=CURRENT_Q_IDS,
    start_keep=8,
    num_mid_spans=3,
    span_size=8,
    recent_keep=16,
    pool=5,
)

print("Selected positions:", len(idx))
print("First positions:", idx[:8])
print("Last positions:", idx[-16:])
print("Unique positions:", len(set(idx)))


In [ ]:

configs = [
    (8, 3, 5),
    (4, 6, 5),
    (6, 4, 5),
    (8, 3, 1),
    (8, 3, 7),
    (4, 6, 1),
]

for span_size, num_mid_spans, pool in configs:
    assert 8 + span_size * num_mid_spans + 16 == 48
    print(
        f"span_size={span_size}, "
        f"num_mid_spans={num_mid_spans}, pool={pool}"
    )


In [ ]:

# Find likely cache and question-token variables
for name, obj in list(globals().items()):
    try:
        if "cache" in name.lower():
            print(name, type(obj).__name__)

        elif "Q_IDS" in name.upper() or "q_ids" in name.lower():
            print(name, type(obj).__name__)
    except Exception:
        pass


In [ ]:

# Smoke test: select 48 positions from the original KV cache

print("Original cache type:", type(original_cache).__name__)
print("Original cache length:", original_cache.get_seq_length())

idx = select_snap_span_indices(
    cache=original_cache,
    model=model,
    q_ids=CURRENT_Q_IDS,
    start_keep=8,
    num_mid_spans=3,
    span_size=8,
    recent_keep=16,
    pool=5,
)

print("Selected positions:", len(idx))
print("Unique positions:", len(set(idx)))
print("First 8:", idx[:8])
print("Last 16:", idx[-16:])
print("Sorted:", idx == sorted(idx))


In [ ]:
import inspect

print(inspect.signature(eval_case))
print(inspect.getsource(eval_case))

In [ ]:
print("Test cases:", len(test_cache))
print("First test case type:", type(test_cache[0]).__name__ if test_cache else "empty")

if test_cache:
    print("First test case:", repr(test_cache[0])[:1000])

In [ ]:
policy_fn = lambda cache: select_snap_span_indices(
    cache, model, q_ids, start_keep=8,
    num_mid_spans=3, span_size=8,
    recent_keep=16, pool=5
)

In [ ]:

import inspect

# Find existing evaluation helpers and test-case definitions
for name in [
    "eval_case",
    "id_token_span",
    "select_hybrid_span_kv",
    "select_snap_span_indices",
    "CURRENT_Q_IDS",
]:
    obj = globals().get(name)
    if obj is not None:
        print(f"\n--- {name} ---")
        if callable(obj) and name != "CURRENT_Q_IDS":
            try:
                print(inspect.signature(obj))
                print(inspect.getsource(obj)[:3500])
            except (TypeError, OSError):
                print(obj)
        else:
            print(type(obj).__name__, getattr(obj, "shape", ""))


In [ ]:

# Inspect notebook variables that may contain benchmark inputs.
# This does not run the model or consume extra GPU compute for inference.

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, (list, tuple)) and len(obj) > 0:
        first = obj[0]
        if isinstance(first, dict):
            keys = list(first.keys())
            if any(k in keys for k in ["ctx", "context", "question", "ident", "answer"]):
                print(f"\n{name}: {len(obj)} items")
                print("First item keys:", keys)
                print("First item preview:", repr(first)[:500])

    if isinstance(obj, dict):
        keys = list(obj.keys())
        if any(k in keys for k in ["ctx", "context", "question", "ident", "answer"]):
            print(f"\n{name}: dict keys = {keys[:30]}")


In [ ]:

# Find likely context and question variables without printing large tensors.
for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, str) and len(obj) > 100:
        print(f"String: {name}, length={len(obj)}")

    elif isinstance(obj, (list, tuple)) and obj:
        if all(isinstance(x, str) for x in obj):
            print(f"String list: {name}, count={len(obj)}")
            print("  First item:", repr(obj[0])[:180])

print("\nExisting benchmark function:", eval_case.__name__)
print("Question IDs shape:", tuple(CURRENT_Q_IDS.shape))


In [ ]:
print("ctx:", repr(ctx[:300]))
print("\ncontext:", repr(context[:300]))
print("\nlong_text:", repr(long_text[:200]))
print("\nExisting records:", len(records))
print("Existing exp2 results:", len(exp2_results))

In [ ]:
print("ctx length:", len(ctx))
print("test_context length:", len(test_context))
print("long_text length:", len(long_text))

print("\nCurrent question IDs decode to:")
print(tokenizer.decode(CURRENT_Q_IDS[0]))

print("\nLatest evaluation result:")
print({k: v for k, v in r.items() if k not in ("idx", "id_toks")})

In [ ]:
print("Question:", tokenizer.decode(CURRENT_Q_IDS[0]))

print("\nCandidate contexts:")
for name in ["ctx", "test_context", "long_text"]:
    value = globals().get(name)
    if isinstance(value, str):
        print(f"{name}: {len(value)} characters")

print("\nExisting result keys:")
print(list(r.keys()))

In [ ]:

# Find which existing context contains the expected identifier.
target = "RG-920-Z"

for name, value in list(globals().items()):
    if isinstance(value, str) and target in value:
        print(f"\nFound in: {name}")
        print(f"Characters: {len(value)}")
        pos = value.find(target)
        print("Nearby text:", repr(value[max(0, pos-180):pos+180]))


In [ ]:

question = "What is the assigned project identifier?"
expected = "RG-920-Z"

# Confirm that this is the correct test context.
assert expected in ctx, "Expected identifier is missing from ctx!"

# Build the question tokens used by the SNAP selector.
q_ids = tokenizer(
    "\nQuestion: " + question + "\nAnswer:",
    add_special_tokens=False,
    return_tensors="pt"
).input_ids.to(device)

# Warm up the model before timing.
_ = eval_case(
    model, tokenizer, device,
    ctx, question, expected,
    None, "absolute", n_new=16
)

# Full-cache baseline.
baseline = eval_case(
    model, tokenizer, device,
    ctx, question, expected,
    None, "absolute", n_new=16
)

# SNAP compressed cache: same context and question.
def snap_policy(cache):
    return select_snap_span_indices(
        cache, model, q_ids,
        start_keep=8,
        num_mid_spans=3,
        span_size=8,
        recent_keep=16,
        pool=5
    )

snap = eval_case(
    model, tokenizer, device,
    ctx, question, expected,
    snap_policy, "absolute", n_new=16
)

# Compact comparison.
for name, result in [("FULL", baseline), ("SNAP", snap)]:
    print(f"\n--- {name} ---")
    print("Context tokens:", result["L"])
    print("Expected identifier retained:", result["id_kept"])
    print("Expected identifier found in answer:", result["exact"])
    print("Generated answer:", repr(result["answer"]))
    print("Retained KV cache (MiB):", round(result["kv_mib"], 4))
    print("Prefill (s):", round(result["prefill_s"], 4))
    print("Selection (s):", round(result["selector_s"], 4))
    print("Compression/copy (s):", round(result["compress_s"], 4))
    print("Continuation (s):", round(result["continue_s"], 4))
    print("Peak allocated-memory delta (MiB):", round(result["peak_delta_mib"], 2))

print("\n--- SNAP vs FULL ---")
print("KV storage reduction (%):",
      round((1 - snap["kv_mib"] / baseline["kv_mib"]) * 100, 2))
print("Both found identifier:",
      baseline["exact"] and snap["exact"])


In [ ]:

question = "What is the assigned project identifier?"

cases = [
    ("Start",       0.05, "RG-101-A"),
    ("Early-Mid",   0.25, "RG-205-B"),
    ("Exact-Mid",   0.50, "RG-500-M"),
    ("Late-Mid",    0.75, "RG-750-L"),
    ("Recent",      0.92, "RG-920-Z"),
]

results = []

for label, depth, expected_id in cases:
    test_ctx, meta = build_context_at_depth(
        tokenizer, expected_id, depth
    )

    q_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    def snap_policy(cache):
        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )

    full = eval_case(
        model, tokenizer, device, test_ctx,
        question, expected_id, None, "absolute", n_new=16
    )

    compressed = eval_case(
        model, tokenizer, device, test_ctx,
        question, expected_id, snap_policy, "absolute", n_new=16
    )

    results.append({
        "position": label,
        "depth": depth,
        "tokens": full["L"],
        "full_found": full["exact"],
        "snap_found": compressed["exact"],
        "snap_id_kept": compressed["id_kept"],
        "full_kv_mib": full["kv_mib"],
        "snap_kv_mib": compressed["kv_mib"],
        "reduction_pct": (
            100 * (1 - compressed["kv_mib"] / full["kv_mib"])
        ),
        "full_answer": full["answer"],
        "snap_answer": compressed["answer"],
    })

for row in results:
    print("\n", row)


In [ ]:

import inspect

print(inspect.signature(build_context_at_depth))
print(inspect.getsource(build_context_at_depth))


In [ ]:

question = "What is the assigned project identifier?"

cases = [
    ("Start",       0.05, "RG-101-A"),
    ("Early-Mid",   0.25, "RG-205-B"),
    ("Exact-Mid",   0.50, "RG-500-M"),
    ("Late-Mid",    0.75, "RG-750-L"),
    ("Recent",      0.92, "RG-920-Z"),
]

results = []

for label, target_depth, expected_id in cases:
    # Function returns: context, measured depth, token length
    test_ctx, measured_depth, token_length = build_context_at_depth(
        tokenizer, expected_id, target_depth
    )

    q_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    def snap_policy(cache):
        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )

    full = eval_case(
        model, tokenizer, device, test_ctx,
        question, expected_id, None, "absolute", n_new=16
    )

    compressed = eval_case(
        model, tokenizer, device, test_ctx,
        question, expected_id, snap_policy, "absolute", n_new=16
    )

    results.append({
        "position": label,
        "target_depth": target_depth,
        "measured_depth": round(measured_depth, 3),
        "tokens": full["L"],
        "full_found": full["exact"],
        "snap_found": compressed["exact"],
        "snap_id_kept": compressed["id_kept"],
        "full_kv_mib": round(full["kv_mib"], 4),
        "snap_kv_mib": round(compressed["kv_mib"], 4),
        "reduction_pct": round(
            100 * (1 - compressed["kv_mib"] / full["kv_mib"]), 2
        ),
        "full_answer": full["answer"],
        "snap_answer": compressed["answer"],
    })

for row in results:
    print("\n", row)


In [ ]:

print("Long-context fixture:")
print("Characters:", len(long_text))
print("Tokens:", len(tokenizer(long_text).input_ids))

for identifier in [
    "RG-101-A", "RG-205-B", "RG-500-M",
    "RG-750-L", "RG-920-Z"
]:
    print(identifier, identifier in long_text)


In [ ]:

LONG_TARGET = 4000
question = "What is the assigned project identifier?"

cases = [
    ("Start",       0.05, "RG-101-A"),
    ("Early-Mid",   0.25, "RG-205-B"),
    ("Exact-Mid",   0.50, "RG-500-M"),
    ("Late-Mid",    0.75, "RG-750-L"),
    ("Recent",      0.92, "RG-920-Z"),
]

long_cases = []

for label, target_depth, expected_id in cases:
    test_ctx, measured_depth, token_length = build_context_at_depth(
        tokenizer, expected_id, target_depth, n_tokens=LONG_TARGET
    )

    long_cases.append({
        "position": label,
        "target_depth": target_depth,
        "measured_depth": measured_depth,
        "expected_id": expected_id,
        "context": test_ctx,
        "tokens": token_length,
    })

    print(
        f"{label:10} | tokens={token_length:4} "
        f"| depth={measured_depth:.3f} "
        f"| fact_present={expected_id in test_ctx}"
    )

assert len(long_cases) == 5
assert all(c["expected_id"] in c["context"] for c in long_cases)


In [ ]:

long_results = []

for case in long_cases:
    label = case["position"]
    test_ctx = case["context"]
    expected_id = case["expected_id"]

    q_ids = tokenizer(
        "\nQuestion: What is the assigned project identifier?\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    def snap_policy(cache):
        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )

    print(f"\nRunning {label}: {case['tokens']} tokens")

    full = eval_case(
        model, tokenizer, device, test_ctx,
        "What is the assigned project identifier?",
        expected_id, None, "absolute", n_new=16
    )

    snap = eval_case(
        model, tokenizer, device, test_ctx,
        "What is the assigned project identifier?",
        expected_id, snap_policy, "absolute", n_new=16
    )

    row = {
        "position": label,
        "tokens": full["L"],
        "measured_depth": round(case["measured_depth"], 3),
        "full_found": full["exact"],
        "snap_found": snap["exact"],
        "snap_id_kept": snap["id_kept"],
        "full_kv_mib": full["kv_mib"],
        "snap_kv_mib": snap["kv_mib"],
        "reduction_pct": 100 * (
            1 - snap["kv_mib"] / full["kv_mib"]
        ),
        "full_answer": full["answer"],
        "snap_answer": snap["answer"],
        "full_peak_mib": full["peak_delta_mib"],
        "snap_peak_mib": snap["peak_delta_mib"],
        "snap_selector_s": snap["selector_s"],
        "snap_compress_s": snap["compress_s"],
    }

    long_results.append(row)

    print("FULL:", full["exact"], repr(full["answer"]))
    print("SNAP:", snap["exact"], repr(snap["answer"]))
    print("KV reduction:", f"{row['reduction_pct']:.2f}%")
    print("Identifier tokens retained:", snap["id_kept"])
    print("SNAP selector time:", f"{snap['selector_s']:.3f}s")

print("\nFinished:", len(long_results), "cases")


In [ ]:

import statistics

print("Cases:", len(long_results))
print("FULL retrieval:", sum(r["full_found"] for r in long_results),
      "/", len(long_results))
print("SNAP retrieval:", sum(r["snap_found"] for r in long_results),
      "/", len(long_results))
print("All identifier tokens retained:",
      all(r["snap_id_kept"] == 1.0 for r in long_results))
print("Mean KV-storage reduction:",
      round(statistics.mean(r["reduction_pct"] for r in long_results), 2), "%")
print("Mean SNAP selector time:",
      round(statistics.mean(r["snap_selector_s"] for r in long_results), 4), "s")
print("Mean SNAP compression/copy time:",
      round(statistics.mean(r["snap_compress_s"] for r in long_results), 4), "s")
print("Mean FULL KV storage:",
      round(statistics.mean(r["full_kv_mib"] for r in long_results), 4), "MiB")
print("Mean SNAP KV storage:",
      round(statistics.mean(r["snap_kv_mib"] for r in long_results), 4), "MiB")


In [ ]:

robust_cases = [
    {
        "question": "What is the registered project code?",
        "fact": "IMPORTANT RECORD: The registered project code is VELVET-4826.",
        "expected": "VELVET-4826",
    },
    {
        "question": "Which facility is listed as the primary site?",
        "fact": "IMPORTANT RECORD: The primary facility is NORTH-HARBOR-17.",
        "expected": "NORTH-HARBOR-17",
    },
    {
        "question": "What is the emergency reference number?",
        "fact": "IMPORTANT RECORD: The emergency reference number is QX-70419.",
        "expected": "QX-70419",
    },
    {
        "question": "Which department owns the project?",
        "fact": "IMPORTANT RECORD: The owning department is ORBITAL-OPS.",
        "expected": "ORBITAL-OPS",
    },
    {
        "question": "What is the scheduled launch code?",
        "fact": "IMPORTANT RECORD: The scheduled launch code is LAUNCH-6382-Z.",
        "expected": "LAUNCH-6382-Z",
    },
]

print("New robustness cases:", len(robust_cases))
for case in robust_cases:
    print(case["question"], "|", case["expected"])


In [ ]:

ROBUST_TARGET_TOKENS = 4000

robust_long_cases = []

for i, case in enumerate(robust_cases):
    target_depth = [0.05, 0.25, 0.50, 0.75, 0.92][i]

    test_ctx, measured_depth, token_length = build_context_at_depth(
        tokenizer,
        case["expected"],
        target_depth,
        n_tokens=ROBUST_TARGET_TOKENS
    )

    # Ensure the context contains the exact fact needed by the question.
    assert case["fact"] in test_ctx, (
        f"Expected fact missing for {case['expected']}"
    )

    robust_long_cases.append({
        **case,
        "position": ["Start", "Early-Mid", "Exact-Mid", "Late-Mid", "Recent"][i],
        "target_depth": target_depth,
        "measured_depth": measured_depth,
        "tokens": token_length,
        "context": test_ctx,
    })

    print(
        f"{robust_long_cases[-1]['position']:10} | "
        f"tokens={token_length:4} | "
        f"depth={measured_depth:.3f} | "
        f"fact_present={case['fact'] in test_ctx}"
    )

print("\nRobustness contexts ready:", len(robust_long_cases))


In [ ]:

ROBUST_TARGET_TOKENS = 4000

robust_long_cases = []

positions = [
    ("Start", 0.05),
    ("Early-Mid", 0.25),
    ("Exact-Mid", 0.50),
    ("Late-Mid", 0.75),
    ("Recent", 0.92),
]

for case, (label, target_depth) in zip(robust_cases, positions):
    test_ctx, measured_depth, token_length = build_context_at_depth(
        tokenizer,
        case["expected"],
        target_depth,
        n_tokens=ROBUST_TARGET_TOKENS
    )

    # Verify the fact that the builder actually inserts.
    generated_fact = (
        f"IMPORTANT RECORD: The assigned project identifier is "
        f"{case['expected']}."
    )
    assert generated_fact in test_ctx, (
        f"Generated fact missing for {case['expected']}"
    )

    robust_long_cases.append({
        **case,
        "position": label,
        "target_depth": target_depth,
        "measured_depth": measured_depth,
        "tokens": token_length,
        "context": test_ctx,
    })

    print(
        f"{label:10} | tokens={token_length:4} "
        f"| depth={measured_depth:.3f} "
        f"| fact_present={generated_fact in test_ctx}"
    )

print("\nRobustness contexts ready:", len(robust_long_cases))


In [ ]:

ROBUST_TARGET_TOKENS = 4000

robust_long_cases = []

for case, (label, target_depth) in zip(
    robust_cases,
    [
        ("Start", 0.05),
        ("Early-Mid", 0.25),
        ("Exact-Mid", 0.50),
        ("Late-Mid", 0.75),
        ("Recent", 0.92),
    ]
):
    fact = case["fact"]
    ft = max(1, len(tokenizer(FILLER).input_ids))
    best = None

    # Search for padding that places the fact at the requested depth.
    for a in range(ROBUST_TARGET_TOKENS // ft + 1):
        for b in range(12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((ROBUST_TARGET_TOKENS - used) / ft))
            test_ctx = before + fact + " " + FILLER * c

            toks, L = id_token_span(tokenizer, test_ctx, case["expected"])
            depth = (toks[0] + toks[-1]) / 2 / L
            key = (abs(depth - target_depth),
                   abs(L - ROBUST_TARGET_TOKENS))

            if best is None or key < best[0]:
                best = (key, test_ctx, depth, L)

    _, test_ctx, measured_depth, token_length = best

    assert fact in test_ctx
    robust_long_cases.append({
        **case,
        "position": label,
        "target_depth": target_depth,
        "measured_depth": measured_depth,
        "tokens": token_length,
        "context": test_ctx,
    })

    print(
        f"{label:10} | tokens={token_length:4} "
        f"| depth={measured_depth:.3f} "
        f"| correct fact={fact in test_ctx}"
    )

print("\nSemantically matched contexts:", len(robust_long_cases))


In [ ]:

robust_results = []

for i, case in enumerate(semantically_matched_cases):
    question = case["question"]
    expected = case["expected"]
    ctx = case["context"]

    print(f"\n[{i+1}/5] Testing: {question}")

    # FULL cache baseline
    full = eval_case(
        model, tokenizer, device,
        ctx, question, expected,
        policy_fn=None,
        mode="absolute",
        n_new=16
    )

    # SNAP compressed cache
    q_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    def snap_policy(cache, q_ids=q_ids):
        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )

    snap = eval_case(
        model, tokenizer, device,
        ctx, question, expected,
        policy_fn=snap_policy,
        mode="absolute",
        n_new=16
    )

    reduction = (
        100 * (1 - snap["kv_mib"] / full["kv_mib"])
        if full["kv_mib"] else 0
    )

    result = {
        "case": i + 1,
        "tokens": full["L"],
        "full_found": full["exact"],
        "snap_found": snap["exact"],
        "id_kept": snap["id_kept"],
        "full_kv_mib": full["kv_mib"],
        "snap_kv_mib": snap["kv_mib"],
        "reduction_pct": reduction,
        "selector_s": snap["selector_s"],
        "full_answer": full["answer"],
        "snap_answer": snap["answer"],
    }
    robust_results.append(result)

    print(f"  FULL found expected text: {full['exact']}")
    print(f"  SNAP found expected text: {snap['exact']}")
    print(f"  Identifier tokens retained: {snap['id_kept']:.2f}")
    print(f"  KV storage reduction: {reduction:.2f}%")
    print(f"  SNAP selector time: {snap['selector_s']:.3f}s")

print("\nRobustness benchmark complete:", len(robust_results), "cases")


In [ ]:

print("Available context-list variables:")

for name, value in list(globals().items()):
    if isinstance(value, list) and value:
        first = value[0]
        if isinstance(first, dict) and (
            "context" in first or "fact" in first
        ):
            print(f"\nVariable: {name}")
            print("Number of items:", len(value))
            print("First item's keys:", list(first.keys()))


In [ ]:

import gc
import pandas as pd
import torch

robust_results = []

for i, case in enumerate(robust_long_cases):
    question = case["question"]
    expected = case["expected"]
    ctx = case["context"]

    print(f"\n[{i+1}/5] {case['position']} | {case['tokens']} tokens")

    # FULL baseline
    full = eval_case(
        model, tokenizer, device,
        ctx, question, expected,
        policy_fn=None,
        mode="absolute",
        n_new=16
    )

    full_found = full["exact"]
    full_kv = full["kv_mib"]
    full_answer = full["answer"]
    full_tokens = full["L"]
    del full
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # SNAP compressed cache
    q_ids = tokenizer(
        "\nQuestion: " + question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    snap = eval_case(
        model, tokenizer, device,
        ctx, question, expected,
        policy_fn=lambda cache, q=q_ids: select_snap_span_indices(
            cache, model, q,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        ),
        mode="absolute",
        n_new=16
    )

    reduction = 100 * (1 - snap["kv_mib"] / full_kv) if full_kv else 0

    robust_results.append({
        "position": case["position"],
        "tokens": full_tokens,
        "full_found": full_found,
        "snap_found": snap["exact"],
        "id_kept": snap["id_kept"],
        "full_kv_mib": full_kv,
        "snap_kv_mib": snap["kv_mib"],
        "reduction_pct": reduction,
        "selector_s": snap["selector_s"],
        "full_answer": full_answer,
        "snap_answer": snap["answer"],
    })

    print(f"  FULL found: {full_found} | SNAP found: {snap['exact']}")
    print(f"  Identifier retained: {snap['id_kept']:.0%}")
    print(f"  KV storage reduction: {reduction:.2f}%")
    print(f"  Selector time: {snap['selector_s']:.3f}s")

    del snap, q_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n=== ROBUSTNESS RESULTS ===")
display(pd.DataFrame(robust_results)[[
    "position", "tokens", "full_found", "snap_found",
    "id_kept", "reduction_pct", "selector_s"
]])

print("\nCases completed:", len(robust_results), "/ 5")


In [ ]:

import pandas as pd
import numpy as np

# Check that all five cases are present
assert "robust_results" in globals(), "robust_results not found."
assert len(robust_results) == 5, "Expected 5 robustness results."

df_robust = pd.DataFrame(robust_results)

# Aggregate the recorded metrics
summary = {
    "Cases evaluated": len(df_robust),
    "FULL retrieval successes": int(df_robust["full_found"].sum()),
    "SNAP retrieval successes": int(df_robust["snap_found"].sum()),
    "Identifier retention (%)": round(df_robust["id_kept"].mean() * 100, 2),
    "Mean KV storage reduction (%)": round(df_robust["reduction_pct"].mean(), 2),
    "Mean selector time (seconds)": round(df_robust["selector_s"].mean(), 4),
    "Mean context length (tokens)": round(df_robust["tokens"].mean(), 1),
}

print("=== ROBUSTNESS SUMMARY ===")
for metric, value in summary.items():
    print(f"{metric}: {value}")

print("\n=== PER-CASE RESULTS ===")
display(df_robust[[
    "position", "tokens", "full_found", "snap_found",
    "id_kept", "full_kv_mib", "snap_kv_mib",
    "reduction_pct", "selector_s"
]].round(4))

# Find existing benchmark result variables without rerunning anything
print("\n=== EXISTING BENCHMARK RESULT LISTS ===")
for name, value in list(globals().items()):
    if isinstance(value, list) and value and isinstance(value[0], dict):
        keys = set(value[0].keys())
        if {"full_found", "snap_found"} <= keys or {
            "full_kv_mib", "snap_kv_mib"
        } <= keys:
            print(f"{name}: {len(value)} rows; keys={sorted(keys)}")


In [ ]:

import pandas as pd
import numpy as np

experiment_lists = {
    "Short-context": results,
    "Long-context": long_results,
    "Robustness": robust_results,
}

all_rows = []

for experiment, rows in experiment_lists.items():
    for row in rows:
        all_rows.append({
            "experiment": experiment,
            "position": row.get("position", "Unknown"),
            "tokens": row.get("tokens", np.nan),
            "full_found": row.get("full_found", False),
            "snap_found": row.get("snap_found", False),
            "id_kept": row.get(
                "snap_id_kept", row.get("id_kept", np.nan)
            ),
            "full_kv_mib": row.get("full_kv_mib", np.nan),
            "snap_kv_mib": row.get("snap_kv_mib", np.nan),
            "reduction_pct": row.get("reduction_pct", np.nan),
            "selector_s": row.get(
                "snap_selector_s", row.get("selector_s", np.nan)
            ),
        })

df_all = pd.DataFrame(all_rows)

def summarize(group):
    return pd.Series({
        "Cases": len(group),
        "FULL successes": int(group["full_found"].sum()),
        "SNAP successes": int(group["snap_found"].sum()),
        "Mean tokens": round(group["tokens"].mean(), 1),
        "Mean KV reduction (%)": round(group["reduction_pct"].mean(), 2),
        "Mean identifier retention (%)": round(
            group["id_kept"].mean() * 100, 2
        ),
        "Mean selector time (s)": round(group["selector_s"].mean(), 4),
        "Mean FULL KV (MiB)": round(group["full_kv_mib"].mean(), 3),
        "Mean SNAP KV (MiB)": round(group["snap_kv_mib"].mean(), 3),
    })

print("=== SUMMARY BY EXPERIMENT ===")
display(df_all.groupby("experiment").apply(
    summarize, include_groups=False
).round(3))

print("\n=== OVERALL SUMMARY ===")
display(summarize(df_all).to_frame("Value"))

print("\n=== ALL RECORDED CASES ===")
display(df_all.round(4))

print("\nTotal cases:", len(df_all))
print("FULL retrieval successes:", int(df_all["full_found"].sum()), "/", len(df_all))
print("SNAP retrieval successes:", int(df_all["snap_found"].sum()), "/", len(df_all))


In [ ]:

import json
import pandas as pd

# Uses df_all from the previous aggregation cell.
df_all.to_csv("recallguard_kv_all_results.csv", index=False)

summary_by_experiment = (
    df_all.groupby("experiment")
    .agg(
        cases=("position", "count"),
        full_successes=("full_found", "sum"),
        snap_successes=("snap_found", "sum"),
        mean_tokens=("tokens", "mean"),
        mean_kv_reduction_pct=("reduction_pct", "mean"),
        mean_identifier_retention=("id_kept", "mean"),
        mean_selector_time_s=("selector_s", "mean"),
        mean_full_kv_mib=("full_kv_mib", "mean"),
        mean_snap_kv_mib=("snap_kv_mib", "mean"),
    )
    .reset_index()
)

summary_by_experiment.to_csv(
    "recallguard_kv_experiment_summary.csv", index=False
)

with open("recallguard_kv_summary.json", "w") as f:
    json.dump({
        "total_cases": int(len(df_all)),
        "full_successes": int(df_all["full_found"].sum()),
        "snap_successes": int(df_all["snap_found"].sum()),
        "mean_casewise_kv_reduction_pct": float(
            df_all["reduction_pct"].mean()
        ),
        "experiments": summary_by_experiment.to_dict(orient="records"),
    }, f, indent=2)

print("Saved:")
print("- recallguard_kv_all_results.csv")
print("- recallguard_kv_experiment_summary.csv")
print("- recallguard_kv_summary.json")


In [ ]:

import os
import json
import pandas as pd

files_to_check = [
    "recallguard_kv_all_results.csv",
    "recallguard_kv_experiment_summary.csv",
    "recallguard_kv_summary.json",
]

for filename in files_to_check:
    assert os.path.exists(filename), f"Missing file: {filename}"
    print(f"OK: {filename} ({os.path.getsize(filename):,} bytes)")

all_results = pd.read_csv("recallguard_kv_all_results.csv")
experiment_summary = pd.read_csv("recallguard_kv_experiment_summary.csv")

with open("recallguard_kv_summary.json", "r") as f:
    saved_summary = json.load(f)

assert len(all_results) == 15, "Expected 15 recorded cases."
assert saved_summary["total_cases"] == 15
assert saved_summary["full_successes"] == 15
assert saved_summary["snap_successes"] == 15

print("\nVerification passed.")
print("Recorded cases:", len(all_results))
print("Experiments:", experiment_summary["experiment"].tolist())
print("FULL successes:", saved_summary["full_successes"])
print("SNAP successes:", saved_summary["snap_successes"])


In [ ]:

import inspect

for name in [
    "eval_case",
    "select_snap_span_indices",
    "_compress_by_indices",
    "build_context_at_depth",
]:
    obj = globals().get(name)

    print(f"\n--- {name} ---")

    if obj is None:
        print("Not found in current runtime")
    else:
        try:
            print("Signature:", inspect.signature(obj))
        except (TypeError, ValueError):
            print("Signature unavailable")

        print(inspect.getdoc(obj) or "No docstring")

print("\n--- Available experiment results ---")
for name in ["results", "long_results", "robust_results"]:
    obj = globals().get(name)
    if obj is None:
        print(f"{name}: not found")
    else:
        print(f"{name}: {len(obj)} rows")
        if len(obj) > 0:
            print("Columns/keys:", list(obj[0].keys()))



--- eval_case ---
Not found in current runtime

--- select_snap_span_indices ---
Not found in current runtime

--- _compress_by_indices ---
Not found in current runtime

--- build_context_at_depth ---
Not found in current runtime

--- Available experiment results ---
results: not found
long_results: not found
robust_results: not found


In [ ]:
# ===== RecallGuard-KV: RESTORE CELL (run once on a fresh runtime) =====
import os
RESULTS_DIR = '/content/recallguard_kv'
os.makedirs(RESULTS_DIR, exist_ok=True)
%pip -q install "transformers==5.18.0" accelerate

import copy, time, random, contextlib, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
print("Transformers:", transformers.__version__)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
device = next(model.parameters()).device
print("Loaded:", model_name, "on", device)

# ---------- DP span selection ----------
def _select_k_non_overlapping_spans_dp(span_scores, k_spans, span_size):
    n = len(span_scores)
    if k_spans == 0:
        return []
    if span_size < 1 or k_spans < 0:
        raise ValueError("bad args")
    if n < (k_spans - 1) * span_size + 1:
        raise ValueError(f"infeasible: {n} span starts cannot fit {k_spans} spans of {span_size}")
    NEG = float("-inf")
    best = [[NEG] * (n + 1) for _ in range(k_spans + 1)]
    for i in range(n + 1):
        best[0][i] = 0.0
    for j in range(1, k_spans + 1):
        for i in range(n - 1, -1, -1):
            take = span_scores[i] + best[j - 1][min(i + span_size, n)]
            best[j][i] = max(best[j][i + 1], take)
    out, i, j = [], 0, k_spans
    while j > 0:
        take = span_scores[i] + best[j - 1][min(i + span_size, n)]
        if take >= best[j][i + 1]:
            out.append(i); j -= 1; i = min(i + span_size, n)
        else:
            i += 1
    return out

# ---------- cache compression + continuation ----------
@torch.no_grad()
def _compress_by_indices(cache, idx):
    out = copy.deepcopy(cache)
    t = torch.tensor(idx, dtype=torch.long)
    for l in out.layers:
        if getattr(l, "keys", None) is None:
            continue
        ti = t.to(l.keys.device)
        l.keys = l.keys.index_select(2, ti).contiguous()
        l.values = l.values.index_select(2, ti).contiguous()
    return out

@torch.no_grad()
def _continue(model, tokenizer, cache, orig_len, q_ids, mode, n_new=12):
    dev = q_ids.device
    def step(ids, pos0):
        r = cache.layers[0].keys.shape[2]
        kw = dict(input_ids=ids, past_key_values=cache, use_cache=True,
                  attention_mask=torch.ones(1, r + ids.shape[1], dtype=torch.long, device=dev))
        if mode == "absolute":
            kw["position_ids"] = torch.arange(pos0, pos0 + ids.shape[1], device=dev).unsqueeze(0)
        return model(**kw)
    q = q_ids.shape[1]
    out = step(q_ids, orig_len)
    cache = out.past_key_values
    first = out.logits[0, -1].float().cpu()
    logits, toks = first, []
    for s in range(n_new):
        t = logits.argmax().item()
        if t == tokenizer.eos_token_id:
            break
        toks.append(t)
        out = step(torch.tensor([[t]], device=dev), orig_len + q + s)
        cache = out.past_key_values
        logits = out.logits[0, -1].float().cpu()
    return first, tokenizer.decode(toks, skip_special_tokens=True)

# ---------- SNAP selector ----------
@contextlib.contextmanager
def eager_attn(model):
    old = model.config._attn_implementation
    try:
        try: model.set_attn_implementation("eager")
        except Exception: model.config._attn_implementation = "eager"
        yield
    finally:
        try: model.set_attn_implementation(old)
        except Exception: model.config._attn_implementation = old

@torch.no_grad()
def select_snap_span_indices(cache, model, q_ids, start_keep=8, num_mid_spans=3,
                             span_size=8, recent_keep=16, pool=5):
    L = cache.layers[0].keys.shape[2]
    budget = start_keep + num_mid_spans * span_size + recent_keep
    if L <= budget:
        return list(range(L))
    lo, hi = start_keep, L - recent_keep
    q, dev = q_ids.shape[1], q_ids.device
    work = copy.deepcopy(cache)
    with eager_attn(model):
        out = model(input_ids=q_ids, past_key_values=work, use_cache=True,
                    attention_mask=torch.ones(1, L + q, dtype=torch.long, device=dev),
                    position_ids=torch.arange(L, L + q, device=dev).unsqueeze(0),
                    output_attentions=True)
    atts = out.attentions
    if atts is None or atts[0] is None:
        raise RuntimeError("No attentions returned: eager attention switch failed.")
    n = len(atts)
    score = sum(a[0, :, :, lo:hi].float().sum(dim=(0, 1)) for a in atts[n // 3:])
    score = F.avg_pool1d(score.view(1, 1, -1), kernel_size=pool, stride=1,
                         padding=pool // 2).view(-1)
    span_scores = F.avg_pool1d(score.view(1, 1, -1), kernel_size=span_size, stride=1).view(-1)
    offs = _select_k_non_overlapping_spans_dp(span_scores.cpu().tolist(), num_mid_spans, span_size)
    idx = (list(range(start_keep))
           + [lo + o + t for o in offs for t in range(span_size)]
           + list(range(L - recent_keep, L)))
    assert len(idx) == budget == len(set(idx)) and idx == sorted(idx)
    return idx

# ---------- contexts ----------
FILLER = "The quarterly summary notes that routine operations continued as planned. "

def id_token_span(tokenizer, ctx, ident):
    enc = tokenizer(ctx, return_offsets_mapping=True)
    c0 = ctx.index(ident); c1 = c0 + len(ident)
    toks = [i for i, (s, e) in enumerate(enc["offset_mapping"]) if e > s and s < c1 and e > c0]
    return toks, len(enc["input_ids"])

def build_context_at_depth(tokenizer, ident, target, n_tokens=240):
    fact = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    ft = len(tokenizer(FILLER).input_ids)
    best = None
    for a in range(0, n_tokens // ft + 1):
        for b in range(0, 12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((n_tokens - used) / ft))
            ctx = before + fact + " " + FILLER * c
            toks, L = id_token_span(tokenizer, ctx, ident)
            depth = (toks[0] + toks[-1]) / 2 / L
            key = (abs(depth - target), abs(L - n_tokens))
            if best is None or key < best[0]:
                best = (key, ctx, depth, L)
    return best[1], best[2], best[3]

WORDS = "alpha harbor copper lantern meadow quartz ridge summit tundra violet willow zenith".split()

def _sent(rng):
    return (f"Report {rng.randint(100,999)}: unit {rng.choice(WORDS)} logged value "
            f"{rng.randint(10,9999)} at site {rng.choice(WORDS)}-{rng.randint(1,99)}.")

def build_hard_context(tokenizer, ident, target, seed, n_sent=14, n_decoys=3):
    rng = random.Random(seed)
    sents = [_sent(rng) for _ in range(n_sent)]
    fpos = min(n_sent - 1, int(target * n_sent))
    others = [i for i in range(n_sent) if i != fpos]
    for i in rng.sample(others, n_decoys):
        d = f"RG-{rng.randint(100,999)}-{rng.choice('ABCDEFGHJKLMNPQRSTUVWXYZ')}"
        sents[i] = f"DECOY RECORD: The archived project identifier is {d}."
    sents[fpos] = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    return " ".join(sents)

# ---------- evaluation ----------
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def kv_bytes(cache):
    return sum(l.keys.numel() * l.keys.element_size() + l.values.numel() * l.values.element_size()
               for l in cache.layers if getattr(l, "keys", None) is not None)

@torch.no_grad()
def eval_case(model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16):
    toks, L = id_token_span(tokenizer, ctx, ident)
    ids = tokenizer(ctx, return_tensors="pt").input_ids.to(device)
    q_ids = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                      return_tensors="pt").input_ids.to(device)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        base_mem = torch.cuda.memory_allocated()
    _sync(); t = time.perf_counter()
    cache = model(input_ids=ids, use_cache=True).past_key_values
    _sync(); prefill_s = time.perf_counter() - t
    t = time.perf_counter()
    idx = list(range(L)) if policy_fn is None else policy_fn(cache)
    _sync(); selector_s = time.perf_counter() - t
    t = time.perf_counter()
    work = _compress_by_indices(cache, idx)
    _sync(); compress_s = time.perf_counter() - t
    kv_mib_pre = kv_bytes(work) / 2**20          # NEW: measured BEFORE question/generation
    n_pos_pre = work.layers[0].keys.shape[2]     # NEW: retained positions before generation
    t = time.perf_counter()
    _, ans = _continue(model, tokenizer, work, L, q_ids, mode, n_new=n_new)
    _sync(); continue_s = time.perf_counter() - t
    kept = set(idx)
    peak = (torch.cuda.max_memory_allocated() - base_mem) / 2**20 if torch.cuda.is_available() else float("nan")
    return dict(
        L=L, depth=(toks[0] + toks[-1]) / 2 / L, id_kept=sum(i in kept for i in toks) / len(toks),
        exact=ident in ans, answer=ans,
        kv_mib_pre=kv_mib_pre, n_pos_pre=n_pos_pre,
        kv_mib=kv_bytes(work) / 2**20,           # original field: measured AFTER generation
        prefill_s=prefill_s, selector_s=selector_s, compress_s=compress_s, continue_s=continue_s,
        peak_delta_mib=peak, idx=idx, id_toks=toks,
    )

# ---------- shared setup ----------
question = "What is the assigned project identifier?"
cases = [(0.05, "RG-101-A"), (0.25, "RG-205-B"), (0.50, "RG-500-M"), (0.75, "RG-750-L"), (0.92, "RG-920-Z")]
CURRENT_Q_IDS = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                          return_tensors="pt").input_ids.to(device)
policies = {
    "full": None,
    "snap_span": lambda c: select_snap_span_indices(c, model, CURRENT_Q_IDS),
}

# ---------- smoke test (1 context, 2 policies) ----------
ctx, depth, L = build_context_at_depth(tokenizer, "RG-500-M", 0.5)
_ = eval_case(model, tokenizer, device, ctx, question, "RG-500-M", None, "default")  # warm-up
print(f"\nContext: {L} tokens, depth {depth:.2f}")
print(f"{'policy':10} {'exact':>5} {'pos_pre':>7} {'KV_pre':>8} {'KV_post':>8} {'select_s':>8} | answer")
for pname, fn in policies.items():
    r = eval_case(model, tokenizer, device, ctx, question, "RG-500-M", fn,
                  "default" if fn is None else "absolute")
    print(f"{pname:10} {str(r['exact']):>5} {r['n_pos_pre']:7d} {r['kv_mib_pre']:8.4f} "
          f"{r['kv_mib']:8.4f} {r['selector_s']:8.3f} | {r['answer'][:30]!r}")

print("\nResults folder:", RESULTS_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.5 MB/s eta 0:00:00
Transformers: 5.18.0


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct on cuda:0

Context: 231 tokens, depth 0.50
policy     exact pos_pre   KV_pre  KV_post select_s | answer
full        True     231   2.7070   3.0352    0.000 | ' RG-500-M\nExplanation: The qua'
snap_span   True      48   0.5625   0.8906    0.065 | ' RG-500-M.  Question: What is '

Results folder: /content/recallguard_kv


In [ ]:
import re, pandas as pd

def recent_indices(L, budget=48):
    return list(range(L)) if L <= budget else list(range(L - budget, L))

def streaming_indices(L, sink=4, budget=48):
    if L <= budget:
        return list(range(L))
    idx = list(range(sink)) + list(range(L - (budget - sink), L))
    assert len(idx) == budget == len(set(idx)) and idx == sorted(idx)
    return idx

_snap_fn = policies["snap_span"]
policies = {
    "full": None,
    "recent48":    lambda c: recent_indices(c.layers[0].keys.shape[2]),
    "stream_4_44": lambda c: streaming_indices(c.layers[0].keys.shape[2], sink=4),
    "stream_8_40": lambda c: streaming_indices(c.layers[0].keys.shape[2], sink=8),
    "snap_span":   _snap_fn,
}

RG = re.compile(r"RG-\d+-[A-Z]")
rows = []
for seed in range(4):
    for target, ident in cases:
        ctx = build_hard_context(tokenizer, ident, target, seed)
        for pname, fn in policies.items():
            for mode in (("default",) if fn is None else ("default", "absolute")):
                r = eval_case(model, tokenizer, device, ctx, question, ident, fn, mode)
                rows.append(dict(
                    seed=seed, target=target, ident=ident, policy=pname, mode=mode,
                    L=r["L"], depth=round(r["depth"], 3), id_kept=r["id_kept"],
                    exact=r["exact"], strict=(RG.findall(r["answer"]) == [ident]),
                    n_pos_pre=r["n_pos_pre"], kv_mib_pre=r["kv_mib_pre"],
                    selector_s=r["selector_s"], prefill_s=r["prefill_s"],
                    compress_s=r["compress_s"], continue_s=r["continue_s"],
                    answer=r["answer"]))

df_hard = pd.DataFrame(rows)
df_hard.to_csv(f"{RESULTS_DIR}/hard_20case_baselines.csv", index=False)

summary = (df_hard.groupby(["policy", "mode"])
           .agg(n=("exact", "size"), exact=("exact", "sum"), strict=("strict", "sum"),
                id_kept=("id_kept", "mean"), pos=("n_pos_pre", "mean"),
                kv_mib_pre=("kv_mib_pre", "mean"), select_s=("selector_s", "mean"))
           .round(3))
print(summary)
print("\nExact successes by fact depth (max 4 per cell):")
print(df_hard.pivot_table(index=["policy", "mode"], columns="target",
                          values="exact", aggfunc="sum"))

                       n  exact  strict  id_kept    pos  kv_mib_pre  select_s
policy      mode                                                             
full        default   20     20      20    1.000  289.7       3.395     0.000
recent48    absolute  20      0       0    0.200   48.0       0.562     0.000
            default   20      2       2    0.200   48.0       0.562     0.000
snap_span   absolute  20     16      16    0.867   48.0       0.562     0.048
            default   20      3       3    0.867   48.0       0.562     0.048
stream_4_44 absolute  20      4       4    0.200   48.0       0.562     0.000
            default   20      0       0    0.200   48.0       0.562     0.000
stream_8_40 absolute  20      4       4    0.200   48.0       0.562     0.000
            default   20      0       0    0.200   48.0       0.562     0.000

Exact successes by fact depth (max 4 per cell):
target                0.05  0.25  0.50  0.75  0.92
policy      mode                          

In [ ]:
def snap_fn(k):
    return lambda c: select_snap_span_indices(c, model, CURRENT_Q_IDS, num_mid_spans=k)

budgets = {32: 1, 48: 3, 96: 9, 192: 21}   # 8 start + 8k + 16 recent
rows = []
for seed in range(4):
    for target, ident in cases:
        ctx = build_hard_context(tokenizer, ident, target, seed, n_sent=56, n_decoys=3)
        r = eval_case(model, tokenizer, device, ctx, question, ident, None, "default")
        rows.append(dict(seed=seed, target=target, policy="full", budget=r["L"], L=r["L"],
                         id_kept=1.0, exact=r["exact"], strict=(RG.findall(r["answer"]) == [ident]),
                         kv_mib_pre=r["kv_mib_pre"], selector_s=0.0, continue_s=r["continue_s"]))
        for B, k in budgets.items():
            pols = {
                "recent":  lambda c, B=B: recent_indices(c.layers[0].keys.shape[2], B),
                "stream4": lambda c, B=B: streaming_indices(c.layers[0].keys.shape[2], 4, B),
                "snap":    snap_fn(k),
            }
            for pname, fn in pols.items():
                r = eval_case(model, tokenizer, device, ctx, question, ident, fn, "absolute")
                assert r["n_pos_pre"] == B, (pname, B, r["n_pos_pre"])
                rows.append(dict(seed=seed, target=target, policy=pname, budget=B, L=r["L"],
                                 id_kept=r["id_kept"], exact=r["exact"],
                                 strict=(RG.findall(r["answer"]) == [ident]),
                                 kv_mib_pre=r["kv_mib_pre"], selector_s=r["selector_s"],
                                 continue_s=r["continue_s"]))

df_sweep = pd.DataFrame(rows)
df_sweep.to_csv(f"{RESULTS_DIR}/budget_sweep.csv", index=False)
print("Mean L:", df_sweep.L.mean().round(0))
print(df_sweep.groupby(["policy", "budget"]).agg(
    n=("exact", "size"), exact=("exact", "sum"), strict=("strict", "sum"),
    id_kept=("id_kept", "mean"), kv_mib_pre=("kv_mib_pre", "mean"),
    select_s=("selector_s", "mean")).round(3))
print(df_sweep[df_sweep.policy != "full"].pivot_table(
    index=["policy", "budget"], columns="target", values="exact", aggfunc="sum"))

Mean L: 1229.0
                 n  exact  strict  id_kept  kv_mib_pre  select_s
policy  budget                                                  
full    1220     1      1       1    1.000      14.297     0.000
        1221     1      1       1    1.000      14.309     0.000
        1223     2      1       1    1.000      14.332     0.000
        1225     1      1       1    1.000      14.355     0.000
        1228     1      1       1    1.000      14.391     0.000
        1229     3      3       3    1.000      14.402     0.000
        1230     2      2       2    1.000      14.414     0.000
        1231     3      3       3    1.000      14.426     0.000
        1232     1      0       0    1.000      14.438     0.000
        1233     1      0       0    1.000      14.449     0.000
        1234     2      2       2    1.000      14.461     0.000
        1235     2      1       1    1.000      14.473     0.000
recent  32      20      0       0    0.000       0.375     0.000
        48

In [ ]:
NAMES = ["Meera Iyer","Karthik Rao","Anil Desai","Priya Nair","Rohan Gupta","Sana Khan",
         "Vikram Joshi","Divya Menon","Arjun Patel","Isha Verma","Nikhil Bose","Tara Singh"]
QTEXT = {"id": "What is the assigned project identifier?",
         "mgr": "Who is the assigned project manager?",
         "bud": "What is the approved project budget?"}

def build_multi_context(seed, n_sent=56):
    rng = random.Random(1000 + seed)
    sents = [_sent(rng) for _ in range(n_sent)]
    nm = rng.sample(NAMES, 2)
    am = rng.sample(range(10000, 99999), 2)
    ids = rng.sample([f"RG-{n}-{c}" for n in range(100, 1000, 37) for c in "ABCDEFGH"], 2)
    exp = {"id": ids[0], "mgr": nm[0], "bud": str(am[0])}
    dec = {"id": ids[1], "mgr": nm[1], "bud": str(am[1])}
    ftxt = {"id": f"IMPORTANT RECORD: The assigned project identifier is {exp['id']}.",
            "mgr": f"IMPORTANT RECORD: The assigned project manager is {exp['mgr']}.",
            "bud": f"IMPORTANT RECORD: The approved project budget is {exp['bud']} rupees."}
    dtxt = {"id": f"DECOY RECORD: The archived project identifier is {dec['id']}.",
            "mgr": f"DECOY RECORD: The former project manager was {dec['mgr']}.",
            "bud": f"DECOY RECORD: The rejected project budget was {dec['bud']} rupees."}
    order = ["id", "mgr", "bud"]
    order = order[seed % 3:] + order[:seed % 3]
    used = set()
    for k, d in zip(order, (0.1, 0.5, 0.9)):
        p = min(n_sent - 1, int(d * n_sent)); sents[p] = ftxt[k]; used.add(p)
    free = [i for i in range(n_sent) if i not in used]
    for k, p in zip(order, rng.sample(free, 3)):
        sents[p] = dtxt[k]
    return " ".join(sents), exp, dec

rows = []
for seed in range(6):
    ctx, exp, dec = build_multi_context(seed)
    for qk, qt in QTEXT.items():
        qids = tokenizer("\nQuestion: " + qt + "\nAnswer:", add_special_tokens=False,
                         return_tensors="pt").input_ids.to(device)
        pols = {"full": None}
        for B, k in {48: 3, 96: 9}.items():
            pols[f"recent{B}"] = lambda c, B=B: recent_indices(c.layers[0].keys.shape[2], B)
            pols[f"stream4_{B}"] = lambda c, B=B: streaming_indices(c.layers[0].keys.shape[2], 4, B)
            pols[f"snap{B}"] = lambda c, k=k, qids=qids: select_snap_span_indices(
                c, model, qids, num_mid_spans=k)
        for pname, fn in pols.items():
            r = eval_case(model, tokenizer, device, ctx, qt, exp[qk], fn,
                          "default" if fn is None else "absolute")
            rows.append(dict(seed=seed, q=qk, policy=pname, L=r["L"], depth=round(r["depth"], 2),
                             id_kept=r["id_kept"], exact=r["exact"],
                             strict=(exp[qk] in r["answer"] and dec[qk] not in r["answer"]),
                             kv_mib_pre=r["kv_mib_pre"], selector_s=r["selector_s"],
                             answer=r["answer"]))

df_multi = pd.DataFrame(rows)
df_multi.to_csv(f"{RESULTS_DIR}/multi_question.csv", index=False)
print("Mean L:", df_multi.L.mean().round(0))
print(df_multi.groupby("policy").agg(n=("exact", "size"), exact=("exact", "sum"),
      strict=("strict", "sum"), id_kept=("id_kept", "mean"),
      kv_mib_pre=("kv_mib_pre", "mean"), select_s=("selector_s", "mean")).round(3))
print(df_multi.pivot_table(index="policy", columns="q", values="strict", aggfunc="sum"))
print(df_multi[(df_multi.policy == "snap96") & (~df_multi.strict)][["seed", "q", "depth", "id_kept", "answer"]])

Mean L: 1213.0
             n  exact  strict  id_kept  kv_mib_pre  select_s
policy                                                      
full        18     16      16    1.000      14.215     0.000
recent48    18      0       0    0.000       0.562     0.000
recent96    18      0       0    0.000       1.125     0.000
snap48      18      5       5    0.540       0.562     0.040
snap96      18     12      12    0.911       1.125     0.042
stream4_48  18      0       0    0.000       0.562     0.000
stream4_96  18      0       0    0.000       1.125     0.000
q           bud  id  mgr
policy                  
full          6   4    6
recent48      0   0    0
recent96      0   0    0
snap48        0   0    5
snap96        1   5    6
stream4_48    0   0    0
stream4_96    0   0    0
     seed    q  depth  id_kept  \
20      0  bud   0.90      0.6   
41      1  bud   0.50      0.8   
83      3  bud   0.90      0.6   
90      4   id   0.91      1.0   
104     4  bud   0.51      0.8   
125    

In [ ]:
@torch.no_grad()
def h2o_scores(ctx_ids):
    with eager_attn(model):
        out = model(input_ids=ctx_ids, use_cache=False, output_attentions=True)
    s = sum(a[0].float().sum(dim=(0, 1)) for a in out.attentions)
    del out
    return s

@torch.no_grad()
def snap_tok_indices(cache, q_ids, B, start_keep=8, recent_keep=16, pool=5):
    L = cache.layers[0].keys.shape[2]
    lo, hi = start_keep, L - recent_keep
    q, dev = q_ids.shape[1], q_ids.device
    work = copy.deepcopy(cache)
    with eager_attn(model):
        out = model(input_ids=q_ids, past_key_values=work, use_cache=True,
                    attention_mask=torch.ones(1, L + q, dtype=torch.long, device=dev),
                    position_ids=torch.arange(L, L + q, device=dev).unsqueeze(0),
                    output_attentions=True)
    n = len(out.attentions)
    sc = sum(a[0, :, :, lo:hi].float().sum(dim=(0, 1)) for a in out.attentions[n // 3:])
    sc = F.avg_pool1d(sc.view(1, 1, -1), kernel_size=pool, stride=1, padding=pool // 2).view(-1)
    top = sc.topk(B - start_keep - recent_keep).indices.add(lo).tolist()
    idx = sorted(set(range(start_keep)) | set(top) | set(range(L - recent_keep, L)))
    assert len(idx) == B
    return idx

def h2o_indices(cache, scores, B, recent_keep=16):
    L = cache.layers[0].keys.shape[2]
    top = scores[: L - recent_keep].topk(B - recent_keep).indices.tolist()
    idx = sorted(set(top) | set(range(L - recent_keep, L)))
    assert len(idx) == B
    return idx

rows = []
for seed in range(4):
    for target, ident in cases:
        ctx = build_hard_context(tokenizer, ident, target, seed, n_sent=56, n_decoys=3)
        ctx_ids = tokenizer(ctx, return_tensors="pt").input_ids.to(device)
        t0 = time.perf_counter(); sc = h2o_scores(ctx_ids); _sync()
        h2o_extra_s = time.perf_counter() - t0
        for B in (48, 96):
            pols = {"h2o": lambda c, B=B: h2o_indices(c, sc, B),
                    "snap_tok": lambda c, B=B: snap_tok_indices(c, CURRENT_Q_IDS, B)}
            for pname, fn in pols.items():
                r = eval_case(model, tokenizer, device, ctx, question, ident, fn, "absolute")
                rows.append(dict(seed=seed, target=target, policy=pname, budget=B, L=r["L"],
                                 id_kept=r["id_kept"], exact=r["exact"],
                                 strict=(RG.findall(r["answer"]) == [ident]),
                                 kv_mib_pre=r["kv_mib_pre"],
                                 selector_s=r["selector_s"] + (h2o_extra_s if pname == "h2o" else 0)))

df_abl = pd.DataFrame(rows)
df_abl.to_csv(f"{RESULTS_DIR}/h2o_and_snap_tok.csv", index=False)
ref = df_sweep[(df_sweep.policy.isin(["snap", "recent", "stream4"])) & (df_sweep.budget.isin([48, 96]))]
comb = pd.concat([ref, df_abl])
print(comb.groupby(["policy", "budget"]).agg(n=("exact", "size"), exact=("exact", "sum"),
      strict=("strict", "sum"), id_kept=("id_kept", "mean"),
      select_s=("selector_s", "mean")).round(3))
print(df_abl.pivot_table(index=["policy", "budget"], columns="target", values="exact", aggfunc="sum"))

                  n  exact  strict  id_kept  select_s
policy   budget                                      
h2o      48      20      0       0    0.000     0.216
         96      20      0       0    0.058     0.211
recent   48      20      0       0    0.000     0.000
         96      20      1       1    0.192     0.000
snap     48      20     15      15    0.800     0.044
         96      20     18      18    1.000     0.042
snap_tok 48      20      1       1    0.625     0.048
         96      20     14      14    0.933     0.044
stream4  48      20      0       0    0.000     0.000
         96      20      1       1    0.092     0.000
target           0.05  0.25  0.50  0.75  0.92
policy   budget                              
h2o      48         0     0     0     0     0
         96         0     0     0     0     0
snap_tok 48         0     0     1     0     0
         96         3     2     4     2     3


In [ ]:
import gc
if "0.5B" not in model.config._name_or_path:
    del model; gc.collect(); torch.cuda.empty_cache()
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda")
    device = next(model.parameters()).device
    CURRENT_Q_IDS = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
                              return_tensors="pt").input_ids.to(device)
print("Model:", model.config._name_or_path)

_ctx, _, _ = build_context_at_depth(tokenizer, "RG-500-M", 0.5)
_ = eval_case(model, tokenizer, device, _ctx, question, "RG-500-M", None, "default")  # warm-up

rows = []
for n_sent in (56, 112, 224):
    for seed in range(3):
        for target, ident in cases:
            ctx = build_hard_context(tokenizer, ident, target, seed, n_sent=n_sent, n_decoys=3)
            jobs = [("full", 0, None, "default")]
            for B, k in ((48, 3), (96, 9)):
                jobs += [("recent", B, lambda c, B=B: recent_indices(c.layers[0].keys.shape[2], B), "absolute"),
                         ("stream4", B, lambda c, B=B: streaming_indices(c.layers[0].keys.shape[2], 4, B), "absolute"),
                         ("snap", B, lambda c, k=k: select_snap_span_indices(c, model, CURRENT_Q_IDS, num_mid_spans=k), "absolute")]
            for pname, B, fn, mode in jobs:
                r = eval_case(model, tokenizer, device, ctx, question, ident, fn, mode)
                if B: assert r["n_pos_pre"] == B
                rows.append(dict(n_sent=n_sent, seed=seed, target=target, policy=pname, budget=B,
                                 L=r["L"], id_kept=r["id_kept"], exact=r["exact"],
                                 strict=(RG.findall(r["answer"]) == [ident]),
                                 kv_mib_pre=r["kv_mib_pre"], selector_s=r["selector_s"],
                                 prefill_s=r["prefill_s"], continue_s=r["continue_s"],
                                 peak_delta_mib=r["peak_delta_mib"]))
            torch.cuda.empty_cache()

df_len = pd.DataFrame(rows)
df_len.to_csv(f"{RESULTS_DIR}/length_latency_memory.csv", index=False)
print(df_len.groupby("n_sent").L.mean().round(0))
print(df_len.groupby(["n_sent", "policy", "budget"]).agg(
    n=("exact", "size"), exact=("exact", "sum"), strict=("strict", "sum"),
    id_kept=("id_kept", "mean"), kv_mib=("kv_mib_pre", "mean"), select_s=("selector_s", "mean"),
    prefill_s=("prefill_s", "mean"), cont_s=("continue_s", "mean"),
    peak_mib=("peak_delta_mib", "mean")).round(3))

Model: Qwen/Qwen2.5-0.5B-Instruct
n_sent
56     1229.0
112    2477.0
224    4979.0
Name: L, dtype: float64
                        n  exact  strict  id_kept  kv_mib  select_s  \
n_sent policy  budget                                                 
56     full    0       15     11      11    1.000  14.405     0.000   
       recent  48      15      0       0    0.000   0.562     0.000   
               96      15      1       1    0.189   1.125     0.000   
       snap    48      15     10      10    0.733   0.562     0.052   
               96      15     13      13    1.000   1.125     0.044   
       stream4 48      15      0       0    0.000   0.562     0.000   
               96      15      0       0    0.056   1.125     0.000   
112    full    0       15     12      12    1.000  29.030     0.000   
       recent  48      15      0       0    0.000   0.562     0.000   
               96      15      0       0    0.000   1.125     0.000   
       snap    48      15     10      10 

In [ ]:
import math
def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(c-h, 2), round(c+h, 2)

VARIANTS = {96: {"base_s8": (8, 9, 8, 16), "s12": (8, 6, 12, 16), "s16": (8, 4, 16, 24)},
            48: {"base_s8": (8, 3, 8, 16), "s12": (8, 2, 12, 16), "s16": (8, 1, 16, 24)}}

rows = []
for seed in range(10):
    ctx, exp, dec = build_multi_context(seed)
    for qk, qt in QTEXT.items():
        qids = tokenizer("\nQuestion: " + qt + "\nAnswer:", add_special_tokens=False,
                         return_tensors="pt").input_ids.to(device)
        jobs = [("full", 0, None, "default")]
        for B, vs in VARIANTS.items():
            for vn, (sk, k, sp, rk) in vs.items():
                jobs.append((f"snap_{vn}", B,
                    lambda c, sk=sk, k=k, sp=sp, rk=rk, qids=qids: select_snap_span_indices(
                        c, model, qids, start_keep=sk, num_mid_spans=k, span_size=sp, recent_keep=rk),
                    "absolute"))
        for pname, B, fn, mode in jobs:
            r = eval_case(model, tokenizer, device, ctx, qt, exp[qk], fn, mode)
            if B: assert r["n_pos_pre"] == B, (pname, B, r["n_pos_pre"])
            rows.append(dict(seed=seed, q=qk, policy=pname, budget=B, L=r["L"],
                             id_kept=r["id_kept"], exact=r["exact"],
                             strict=(exp[qk] in r["answer"] and dec[qk] not in r["answer"]),
                             kv_mib_pre=r["kv_mib_pre"], selector_s=r["selector_s"], answer=r["answer"]))

df_mod = pd.DataFrame(rows)
df_mod.to_csv(f"{RESULTS_DIR}/snap_variants_multiq.csv", index=False)
g = df_mod.groupby(["policy", "budget"]).agg(
    n=("strict", "size"), strict=("strict", "sum"), exact=("exact", "sum"),
    id_kept=("id_kept", "mean"), kv_mib=("kv_mib_pre", "mean"), select_s=("selector_s", "mean")).round(3)
g["strict_95CI"] = [wilson(int(s), int(n)) for s, n in zip(g.strict, g.n)]
print(g)
print(df_mod.pivot_table(index=["policy", "budget"], columns="q", values="strict", aggfunc="sum"))

                      n  strict  exact  id_kept  kv_mib  select_s  \
policy       budget                                                 
full         0       30      27     27    1.000  14.191     0.000   
snap_base_s8 48      30       9      9    0.541   0.562     0.049   
             96      30      20     20    0.900   1.125     0.050   
snap_s12     48      30      15     15    0.721   0.562     0.050   
             96      30      22     22    0.908   1.125     0.047   
snap_s16     48      30      28     28    0.933   0.562     0.050   
             96      30      30     30    1.000   1.125     0.050   

                      strict_95CI  
policy       budget                
full         0       (0.74, 0.97)  
snap_base_s8 48      (0.17, 0.48)  
             96      (0.49, 0.81)  
snap_s12     48      (0.33, 0.67)  
             96      (0.56, 0.86)  
snap_s16     48      (0.79, 0.98)  
             96       (0.89, 1.0)  
q                    bud  id  mgr
policy       budget 

In [ ]:
import inspect, random, bisect, math, os
import pandas as pd, torch

# ---------- 0. detect the span-width kwarg of your selector ----------
sig = inspect.signature(select_snap_span_indices)
print("selector signature:", sig)
_c = [p for p in sig.parameters
      if p not in ("cache", "model", "q_ids", "num_mid_spans")
      and any(s in p.lower() for s in ("span", "width", "len", "size"))]
print("width kwarg candidates:", _c)
assert _c, "No width kwarg found: paste inspect.getsource(select_snap_span_indices) and I'll adapt."
SPAN_KW = _c[0]

# ---------- 1. varied held-out generator ----------
FIRST = "Maya Arjun Elena Tomas Priya Jonas Lucia Kenji Amara Viktor Sana Mateo".split()
LAST = "Okafor Lindqvist Raman Duarte Nakamura Haddad Petrov Moreau Banerjee Costa Weiss Iyer".split()
PROJ = "Falcon Heron Osprey Kestrel Merlin Condor Harrier Swift Plover Tern Egret Lapwing".split()
WORDS = ("quarterly inventory vendor schedule compliance audit warehouse shipment training facility "
         "maintenance forecast regional supplier contract review staffing logistics procurement safety "
         "renewal inspection transport network upgrade policy meeting summary workflow capacity delivery "
         "handover archive calendar region operations portfolio").split()

T_ID = ["{P} code: {X}.",
        "The official identifier for Project {P} is {X}.",
        "After the quarterly compliance review concluded, the registry assigned Project {P} the official identifier {X}, effective immediately."]
T_MG = ["{P} manager: {X}.",
        "Project {P} is managed by {X}.",
        "Following the leadership reorganization, {X} was appointed as the manager responsible for Project {P} and its deliverables."]
T_BD = ["{P} budget: {X}.",
        "The approved budget for Project {P} is {X}.",
        "After several rounds of negotiation with the finance committee, the approved budget for Project {P} was set at {X} for the full year."]
QTXT = {"id": "What is the official identifier of Project {P}?",
        "mgr": "Who is the manager of Project {P}?",
        "bud": "What is the approved budget of Project {P}?"}

def make_case(seed, n_sent):
    rng = random.Random(seed)
    projs = rng.sample(PROJ, 4)
    used = {"id": set(), "mgr": set(), "bud": set()}
    def uniq(kind, fn):
        while True:
            v = fn()
            if v not in used[kind]:
                used[kind].add(v); return v
    rid = lambda: f"RG-{rng.randint(100,999)}-{rng.choice('ABCDEFGHJKMNPQRSTUVWXYZ')}"
    person = lambda: f"{rng.choice(FIRST)} {rng.choice(LAST)}"
    def money():
        n = rng.randrange(20000, 9000000, 250)
        s = f"${n:,}"
        return s + rng.choice([".25", ".50", ".75"]) if rng.random() < 0.4 else s
    facts, ans, decoys = [], {}, {"id": [], "mgr": [], "bud": []}
    for pi, P in enumerate(projs):
        for kind, T, gen in (("id", T_ID, rid), ("mgr", T_MG, person), ("bud", T_BD, money)):
            v = uniq(kind, gen)
            facts.append(T[rng.randint(0, 2)].format(P=P, X=v))
            if pi == 0: ans[kind] = v
            else: decoys[kind].append(v)
    pos = rng.sample(range(2, n_sent - 2), len(facts))
    sents = ["Report " + " ".join(rng.choice(WORDS) for _ in range(rng.randint(6, 26))) + "."
             for _ in range(n_sent)]
    for p, f in zip(pos, facts): sents[p] = f
    qa = [(k, QTXT[k].format(P=projs[0]), ans[k]) for k in ("id", "mgr", "bud")]
    return dict(ctx=" ".join(sents), qa=qa, decoys=decoys)

# ---------- 2. selectors ----------
def _as_list(x): return torch.as_tensor(x).flatten().tolist()
def _like(orig, lst):
    return torch.tensor(lst, dtype=orig.dtype, device=orig.device) if torch.is_tensor(orig) else lst

def snap_fn(B, w, q_ids):
    k = max(1, (B - 24) // w)           # 8 start + 16 recent + k spans of width w
    kw = {} if w == 8 else {SPAN_KW: w}
    return lambda c: select_snap_span_indices(c, model, q_ids, num_mid_spans=k, **kw)

def boundary_fn(B, q_ids, ctx):
    ids = tokenizer(ctx, add_special_tokens=False).input_ids
    toks = [tokenizer.decode([t]) for t in ids]
    ends = [i for i, t in enumerate(toks)
            if (t.rstrip().endswith((".", "!", "?")) or "\n" in t)
            and not (i + 1 < len(toks) and toks[i + 1].strip()[:1].isdigit())]
    starts = [0] + [e + 1 for e in ends]
    def expand(idx, L):
        mid = sorted(i for i in set(idx) if 8 <= i < L - 16)
        runs = []
        for i in mid:
            if runs and i == runs[-1][1] + 1: runs[-1][1] = i
            else: runs.append([i, i])
        keep = set(idx)
        for a, b in runs:
            s = starts[bisect.bisect_right(starts, a) - 1]
            j = bisect.bisect_left(ends, b)
            e = ends[j] if j < len(ends) else L - 1
            keep.update(range(max(s, 8), min(e, L - 17) + 1))
        return sorted(keep)
    def fn(c):
        L = c.layers[0].keys.shape[2]
        assert L == len(ids), f"cache {L} != tokens {len(ids)}"
        for k in range(max(1, (B - 24) // 12), 0, -1):   # most spans that fit after expansion
            raw = select_snap_span_indices(c, model, q_ids, num_mid_spans=k)
            new = expand(_as_list(raw), L)
            if len(new) <= B: return _like(raw, new)
        return raw                                         # fallback: unexpanded k=1
    return fn

# ---------- 3. preflight ----------
c0 = make_case(1000, 56)
n_tok = len(tokenizer(c0["ctx"], add_special_tokens=False).input_ids)
r0 = eval_case(model, tokenizer, device, c0["ctx"], c0["qa"][0][1], c0["qa"][0][2], None, "default")
print("ctx tokens:", n_tok, "| eval_case L:", r0["L"], "| answer:", repr(r0["answer"]))
BND_OK = (r0["L"] == n_tok)
if not BND_OK: print("WARNING: L != token count, skipping snap_bnd. Paste eval_case source and I'll fix.")

# ---------- 4. main loop (resumable, saves per context) ----------
OUT = f"{RESULTS_DIR}/heldout_varied.csv"
rows = pd.read_csv(OUT).to_dict("records") if os.path.exists(OUT) else []
done = {(r["n_sent"], r["seed"], r["qtype"], r["policy"], r["budget"]) for r in rows}

for n_sent in (56, 112):
    for seed in range(1000, 1012):
        case = make_case(seed, n_sent)
        for qtype, q, truth in case["qa"]:
            q_ids = tokenizer("\nQuestion: " + q + "\nAnswer:", add_special_tokens=False,
                              return_tensors="pt").input_ids.to(device)
            jobs = [("full", 0, None, "default")]
            for B in (48, 96):
                jobs += [("snap_w8", B, snap_fn(B, 8, q_ids), "absolute"),
                         ("snap_w16", B, snap_fn(B, 16, q_ids), "absolute")]
                if BND_OK: jobs.append(("snap_bnd", B, boundary_fn(B, q_ids, case["ctx"]), "absolute"))
            for pname, B, fn, mode in jobs:
                if (n_sent, seed, qtype, pname, B) in done: continue
                r = eval_case(model, tokenizer, device, case["ctx"], q, truth, fn, mode)
                strict = bool(r["exact"]) and not any(d in r["answer"] for d in case["decoys"][qtype])
                rows.append(dict(n_sent=n_sent, seed=seed, qtype=qtype, policy=pname, budget=B, L=r["L"],
                                 exact=int(r["exact"]), strict=int(strict), id_kept=r["id_kept"],
                                 n_pos_pre=r["n_pos_pre"], kv_mib_pre=r["kv_mib_pre"],
                                 selector_s=r["selector_s"], prefill_s=r["prefill_s"],
                                 continue_s=r["continue_s"], peak_delta_mib=r["peak_delta_mib"],
                                 answer=r["answer"][:60], truth=truth))
        pd.DataFrame(rows).to_csv(OUT, index=False)
        torch.cuda.empty_cache()
        print(f"done n_sent={n_sent} seed={seed}")

# ---------- 5. summaries ----------
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n)
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-h)/d, (c+h)/d)

def summarize(df, by):
    g = df.groupby(by).agg(n=("strict", "size"), strict=("strict", "sum"), exact=("exact", "sum"),
                           id_kept=("id_kept", "mean"), pos=("n_pos_pre", "mean"),
                           kv_mib=("kv_mib_pre", "mean"), sel_s=("selector_s", "mean"),
                           peak_mib=("peak_delta_mib", "mean"))
    g["lo"] = [wilson(a, b)[0] for a, b in zip(g.strict, g.n)]
    g["hi"] = [wilson(a, b)[1] for a, b in zip(g.strict, g.n)]
    return g.round(3)

df = pd.DataFrame(rows)
print(summarize(df, ["n_sent", "policy", "budget"]))
print(summarize(df, ["policy", "budget", "qtype"])[["n", "strict", "lo", "hi", "pos"]])
bad = df[(df.strict == 0) & df.policy.isin(["snap_w16", "snap_bnd"]) & (df.budget == 96)]
print(bad[["n_sent", "seed", "qtype", "policy", "answer", "truth", "id_kept"]].head(12))

selector signature: (cache, model, q_ids, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16, pool=5)
width kwarg candidates: ['span_size']
ctx tokens: 1067 | eval_case L: 1067 | answer: ' RG-461-C\n\nThis answer is correct based on the information provided'
done n_sent=56 seed=1000
done n_sent=56 seed=1001
done n_sent=56 seed=1002
done n_sent=56 seed=1003
done n_sent=56 seed=1004
done n_sent=56 seed=1005
done n_sent=56 seed=1006
done n_sent=56 seed=1007
done n_sent=56 seed=1008
done n_sent=56 seed=1009
done n_sent=56 seed=1010
done n_sent=56 seed=1011
done n_sent=112 seed=1000
done n_sent=112 seed=1001
done n_sent=112 seed=1002
done n_sent=112 seed=1003
done n_sent=112 seed=1004
done n_sent=112 seed=1005
done n_sent=112 seed=1006
done n_sent=112 seed=1007
done n_sent=112 seed=1008
done n_sent=112 seed=1009
done n_sent=112 seed=1010
done n_sent=112 seed=1011
                         n  strict  exact  id_kept       pos  kv_mib  sel_s  \
n_sent policy   budget                    

In [ ]:
import inspect, re
for fn in (eval_case, _continue):
    for i, line in enumerate(inspect.getsource(fn).splitlines()):
        if re.search(r"max_new_tokens|n_gen|max_new|range\(\s*1[0-9]\s*\)", line):
            print(fn.__name__, i, line.strip())

case = make_case(1000, 56)
qtype, q, truth = case["qa"][2]          # the budget question
r = eval_case(model, tokenizer, device, case["ctx"], q, truth, None, "default")
print("truth :", truth)
print("answer:", repr(r["answer"]))
print("answer tokens:", len(tokenizer(r["answer"], add_special_tokens=False).input_ids))

truth : $2,155,000.25
answer: ' The approved budget for Project Harrier is $2,155,0'
answer tokens: 16


In [ ]:
import inspect
print("eval_case  :", inspect.signature(eval_case))
print("_continue  :", inspect.signature(_continue))
print(inspect.getsource(_continue))

eval_case  : (model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16)
_continue  : (model, tokenizer, cache, orig_len, q_ids, mode, n_new=12)
@torch.no_grad()
def _continue(model, tokenizer, cache, orig_len, q_ids, mode, n_new=12):
    dev = q_ids.device
    def step(ids, pos0):
        r = cache.layers[0].keys.shape[2]
        kw = dict(input_ids=ids, past_key_values=cache, use_cache=True,
                  attention_mask=torch.ones(1, r + ids.shape[1], dtype=torch.long, device=dev))
        if mode == "absolute":
            kw["position_ids"] = torch.arange(pos0, pos0 + ids.shape[1], device=dev).unsqueeze(0)
        return model(**kw)
    q = q_ids.shape[1]
    out = step(q_ids, orig_len)
    cache = out.past_key_values
    first = out.logits[0, -1].float().cpu()
    logits, toks = first, []
    for s in range(n_new):
        t = logits.argmax().item()
        if t == tokenizer.eos_token_id:
            break
        toks.append(t)
        out = step(torch.tensor(

In [ ]:
import re, bisect, math, os
import pandas as pd, torch
assert "make_case" in globals() and "select_snap_span_indices" in globals(), \
    "Run the previous cell's definitions first (make_case, _as_list, _like, wilson, summarize)."

N_NEW = 40

# ---------- pad every selection to exactly B positions ----------
def pad_to_budget(idx, L, B):
    keep = set(_as_list(idx))
    lo, hi = 8, L - 16                      # middle zone; first 8 and last 16 are already kept
    runs = []
    for i in sorted(i for i in keep if lo <= i < hi):
        if runs and i == runs[-1][1] + 1: runs[-1][1] = i
        else: runs.append([i, i])
    while len(keep) < B and runs:
        grew = False
        for r in runs:
            for side in (-1, 1):
                if len(keep) >= B: break
                j = r[0] - 1 if side < 0 else r[1] + 1
                if lo <= j < hi and j not in keep:
                    keep.add(j); grew = True
                    if side < 0: r[0] = j
                    else: r[1] = j
        if not grew: break
    return sorted(keep)

def snap_fn(B, w, q_ids):
    k = max(1, (B - 24) // w)
    def fn(c):
        L = c.layers[0].keys.shape[2]
        raw = select_snap_span_indices(c, model, q_ids, num_mid_spans=k, span_size=w)
        return _like(raw, pad_to_budget(raw, L, B))
    return fn

def boundary_fn(B, q_ids, ctx):
    ids = tokenizer(ctx, add_special_tokens=False).input_ids
    toks = [tokenizer.decode([t]) for t in ids]
    ends = [i for i, t in enumerate(toks)
            if (t.rstrip().endswith((".", "!", "?")) or "\n" in t)
            and not (i + 1 < len(toks) and toks[i + 1].strip()[:1].isdigit())]
    starts = [0] + [e + 1 for e in ends]
    def expand(idx, L):
        runs = []
        for i in sorted(i for i in set(idx) if 8 <= i < L - 16):
            if runs and i == runs[-1][1] + 1: runs[-1][1] = i
            else: runs.append([i, i])
        keep = set(idx)
        for a, b in runs:
            s = starts[bisect.bisect_right(starts, a) - 1]
            j = bisect.bisect_left(ends, b)
            e = ends[j] if j < len(ends) else L - 1
            keep.update(range(max(s, 8), min(e, L - 17) + 1))
        return sorted(keep)
    def fn(c):
        L = c.layers[0].keys.shape[2]
        assert L == len(ids), f"cache {L} != tokens {len(ids)}"
        chosen = None
        for k in range((B - 24) // 8, 0, -1):          # most spans that still fit after stretching
            raw = select_snap_span_indices(c, model, q_ids, num_mid_spans=k)
            new = expand(_as_list(raw), L)
            if len(new) <= B: chosen = (raw, new); break
        if chosen is None:
            raw = select_snap_span_indices(c, model, q_ids, num_mid_spans=1)
            chosen = (raw, _as_list(raw))
        raw, new = chosen
        return _like(raw, pad_to_budget(new, L, B))
    return fn

# ---------- stricter scoring: first line only, exact value with boundaries ----------
def pat(v): return re.compile(r"(?<!\w)" + re.escape(v) + r"(?!\w)(?!\.\d)(?!,\d)")
def first_line(a): return a.strip().split("\n")[0]

# ---------- preflight: does FULL now finish the budget answer? ----------
c0 = make_case(1000, 56); _, q0, t0 = c0["qa"][2]
r0 = eval_case(model, tokenizer, device, c0["ctx"], q0, t0, None, "default", n_new=N_NEW)
print("truth :", t0); print("answer:", repr(r0["answer"]))
print("FULL gets it right:" , bool(pat(t0).search(first_line(r0["answer"]))))

# ---------- main loop (resumable) ----------
OUT = f"{RESULTS_DIR}/heldout_varied_v2.csv"
rows = pd.read_csv(OUT).to_dict("records") if os.path.exists(OUT) else []
done = {(r["n_sent"], r["seed"], r["qtype"], r["policy"], r["budget"]) for r in rows}

for n_sent in (56, 112):
    for seed in range(1000, 1012):
        case = make_case(seed, n_sent)
        for qtype, q, truth in case["qa"]:
            q_ids = tokenizer("\nQuestion: " + q + "\nAnswer:", add_special_tokens=False,
                              return_tensors="pt").input_ids.to(device)
            jobs = [("full", 0, None, "default")]
            for B in (48, 96):
                jobs += [("snap_w8", B, snap_fn(B, 8, q_ids), "absolute"),
                         ("snap_w16", B, snap_fn(B, 16, q_ids), "absolute"),
                         ("snap_bnd", B, boundary_fn(B, q_ids, case["ctx"]), "absolute")]
            for pname, B, fn, mode in jobs:
                if (n_sent, seed, qtype, pname, B) in done: continue
                r = eval_case(model, tokenizer, device, case["ctx"], q, truth, fn, mode, n_new=N_NEW)
                line = first_line(r["answer"])
                ok = bool(pat(truth).search(line))
                decoy_hit = any(pat(d).search(line) for d in case["decoys"][qtype])
                rows.append(dict(n_sent=n_sent, seed=seed, qtype=qtype, policy=pname, budget=B, L=r["L"],
                                 exact=int(ok), strict=int(ok and not decoy_hit), exact_eval=int(r["exact"]),
                                 id_kept=r["id_kept"], n_pos_pre=r["n_pos_pre"], kv_mib_pre=r["kv_mib_pre"],
                                 selector_s=r["selector_s"], prefill_s=r["prefill_s"],
                                 continue_s=r["continue_s"], peak_delta_mib=r["peak_delta_mib"],
                                 answer=line[:120], truth=truth))
        pd.DataFrame(rows).to_csv(OUT, index=False)
        torch.cuda.empty_cache()
        print(f"done n_sent={n_sent} seed={seed}")

# ---------- summaries ----------
df = pd.DataFrame(rows)
print(summarize(df, ["n_sent", "policy", "budget"]))
print(summarize(df, ["policy", "budget", "qtype"])[["n", "strict", "lo", "hi", "pos"]])
print("positions used vs budget:\n", df[df.budget > 0].groupby(["policy", "budget"]).n_pos_pre.mean().round(1))

truth : $2,155,000.25
answer: ' The approved budget for Project Harrier is $2,155,000.25.\n\nQuestion: What is the official identifier for Project Heron?\nAnswer: RG-14'
FULL gets it right: True
done n_sent=56 seed=1000
done n_sent=56 seed=1001
done n_sent=56 seed=1002
done n_sent=56 seed=1003
done n_sent=56 seed=1004
done n_sent=56 seed=1005
done n_sent=56 seed=1006
done n_sent=56 seed=1007
done n_sent=56 seed=1008
done n_sent=56 seed=1009
done n_sent=56 seed=1010
done n_sent=56 seed=1011
done n_sent=112 seed=1000
done n_sent=112 seed=1001
done n_sent=112 seed=1002
done n_sent=112 seed=1003
done n_sent=112 seed=1004
done n_sent=112 seed=1005
done n_sent=112 seed=1006
done n_sent=112 seed=1007
done n_sent=112 seed=1008
done n_sent=112 seed=1009
done n_sent=112 seed=1010
done n_sent=112 seed=1011
                         n  strict  exact  id_kept       pos  kv_mib  sel_s  \
n_sent policy   budget                                                        
56     full     0       36      27 

In [13]:
import os, inspect, math
import pandas as pd, torch

need = ["make_case", "pad_to_budget", "snap_fn", "boundary_fn", "pat", "first_line",
        "summarize", "wilson", "recent_indices", "streaming_indices"]
missing = [n for n in need if n not in globals()]
assert not missing, f"Rerun the earlier cells that define: {missing}"
print("H2O-like helpers:", [(n, str(inspect.signature(f))) for n, f in globals().items()
                            if callable(f) and "h2o" in n.lower()])

N_NEW = 40
SEEDS = range(2000, 2012)      # fresh seeds; widen to range(2000, 2024) later, it only runs the missing ones
LENS = (112, 224)
BUDGETS = (48, 96, 192)

FINAL = f"{RESULTS_DIR}/final_fresh_seeds.csv"
rows = pd.read_csv(FINAL).to_dict("records") if os.path.exists(FINAL) else []
done = {(r["n_sent"], r["seed"], r["qtype"], r["policy"], r["budget"]) for r in rows}

for n_sent in LENS:
    for seed in SEEDS:
        case = make_case(seed, n_sent)
        for qtype, q, truth in case["qa"]:
            q_ids = tokenizer("\nQuestion: " + q + "\nAnswer:", add_special_tokens=False,
                              return_tensors="pt").input_ids.to(device)
            jobs = [("full", 0, None, "default")]
            for B in BUDGETS:
                jobs += [("recent", B, lambda c, B=B: recent_indices(c.layers[0].keys.shape[2], B), "absolute"),
                         ("stream4", B, lambda c, B=B: streaming_indices(c.layers[0].keys.shape[2], 4, B), "absolute"),
                         ("snap_w16", B, snap_fn(B, 16, q_ids), "absolute"),
                         ("snap_bnd", B, boundary_fn(B, q_ids, case["ctx"]), "absolute")]
            for pname, B, fn, mode in jobs:
                if (n_sent, seed, qtype, pname, B) in done:
                    continue
                try:
                    r = eval_case(model, tokenizer, device, case["ctx"], q, truth, fn, mode, n_new=N_NEW)
                except torch.cuda.OutOfMemoryError:
                    print("OOM, skipped:", n_sent, seed, qtype, pname, B)
                    torch.cuda.empty_cache(); continue
                if B and r["n_pos_pre"] != B:
                    print("WARNING positions", r["n_pos_pre"], "!= budget", B, pname)
                line = first_line(r["answer"])
                ok = bool(pat(truth).search(line))
                decoy_hit = any(pat(d).search(line) for d in case["decoys"][qtype])
                rows.append(dict(n_sent=n_sent, seed=seed, qtype=qtype, policy=pname, budget=B, L=r["L"],
                                 exact=int(ok), strict=int(ok and not decoy_hit),
                                 id_kept=r["id_kept"], n_pos_pre=r["n_pos_pre"], kv_mib_pre=r["kv_mib_pre"],
                                 selector_s=r["selector_s"], prefill_s=r["prefill_s"],
                                 continue_s=r["continue_s"], peak_delta_mib=r["peak_delta_mib"],
                                 answer=line[:120], truth=truth))
        pd.DataFrame(rows).to_csv(FINAL, index=False)
        torch.cuda.empty_cache()
        print(f"done n_sent={n_sent} seed={seed}")

# ---------------- summaries ----------------
df = pd.DataFrame(rows)
full_kv = df[df.policy == "full"].groupby("n_sent").kv_mib_pre.mean()
df["kv_saved"] = 1 - df.kv_mib_pre / df.n_sent.map(full_kv)
full_rate = df[df.policy == "full"].strict.mean()

g = summarize(df, ["policy", "budget"])
g["pct_of_full"] = (100 * g.strict / g.n / full_rate).round(0)
g["kv_saved"] = df.groupby(["policy", "budget"]).kv_saved.mean().round(3)
print("\n=== MAIN TABLE (pooled) ===")
print(g[["n", "strict", "lo", "hi", "pct_of_full", "id_kept", "pos", "kv_saved", "sel_s"]])

print("\n=== BY QUESTION TYPE ===")
print(summarize(df, ["policy", "budget", "qtype"])[["n", "strict", "lo", "hi", "id_kept"]])

print("\n=== BY LENGTH ===")
print(summarize(df, ["n_sent", "policy", "budget"])[["n", "strict", "lo", "hi", "id_kept"]])

def paired(A, B_, budget):
    key = ["n_sent", "seed", "qtype"]
    a = df[(df.policy == A) & (df.budget == budget)].set_index(key).strict
    b = df[(df.policy == B_) & (df.budget == budget)].set_index(key).strict
    j = pd.concat([a, b], axis=1, keys=["a", "b"]).dropna()
    print(f"budget {budget}: only {A} right={int(((j.a==1)&(j.b==0)).sum())}, "
          f"only {B_} right={int(((j.a==0)&(j.b==1)).sum())}, n={len(j)}")

print("\n=== PAIRED (same cases) ===")
for B in BUDGETS:
    paired("snap_bnd", "snap_w16", B)
    paired("snap_bnd", "stream4", B)
    paired("snap_bnd", "recent", B)

print("\n=== LATENCY AND PEAK MEMORY (mean) ===")
print(df.groupby(["policy", "budget"])[["selector_s", "prefill_s", "continue_s", "peak_delta_mib"]].mean().round(3))

H2O-like helpers: [('h2o_scores', '(ctx_ids)'), ('h2o_indices', '(cache, scores, B, recent_keep=16)')]
done n_sent=112 seed=2000
done n_sent=112 seed=2001
done n_sent=112 seed=2002
done n_sent=112 seed=2003
done n_sent=112 seed=2004
done n_sent=112 seed=2005
done n_sent=112 seed=2006
done n_sent=112 seed=2007
done n_sent=112 seed=2008
done n_sent=112 seed=2009
done n_sent=112 seed=2010
done n_sent=112 seed=2011
done n_sent=224 seed=2000
done n_sent=224 seed=2001
done n_sent=224 seed=2002
done n_sent=224 seed=2003
done n_sent=224 seed=2004
done n_sent=224 seed=2005
done n_sent=224 seed=2006
done n_sent=224 seed=2007
done n_sent=224 seed=2008
done n_sent=224 seed=2009
done n_sent=224 seed=2010
done n_sent=224 seed=2011

=== MAIN TABLE (pooled) ===
                  n  strict     lo     hi  pct_of_full  id_kept       pos  \
policy   budget                                                             
full     0       72      44  0.496  0.715        100.0    1.000  3122.542   
recent   48  

In [14]:
import shutil, os
from google.colab import files
d = globals().get("RESULTS_DIR", "/content/recallguard_kv")
print(os.listdir(d))                      # check final_fresh_seeds.csv is in the list
shutil.make_archive("/content/recallguard_results", "zip", d)
files.download("/content/recallguard_results.zip")

['hard_20case_baselines.csv', 'h2o_and_snap_tok.csv', 'heldout_varied_v2.csv', 'length_latency_memory.csv', 'final_fresh_seeds.csv', 'heldout_varied.csv', 'budget_sweep.csv', 'snap_variants_multiq.csv', 'multi_question.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:

import os
import json
import subprocess
from datetime import datetime, timezone

BACKUP_DIR = "/content/recallguard_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

# Save environment information
with open(f"{BACKUP_DIR}/environment.txt", "w") as f:
    f.write(f"Timestamp UTC: {datetime.now(timezone.utc).isoformat()}\n")
    f.write(subprocess.getoutput("pip freeze"))
    f.write("\n\nGPU status:\n")
    f.write(subprocess.getoutput("nvidia-smi"))

# Save an inventory of functions and variables currently in RAM.
# This is a manifest, NOT a full copy of Python runtime memory.
inventory = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "functions": sorted(
        name for name, obj in globals().items()
        if callable(obj) and getattr(obj, "__module__", None) == "__main__"
    ),
    "important_variables": {
        name: str(type(globals()[name]))
        for name in [
            "model", "tokenizer", "device",
            "results", "all_results", "summary"
        ]
        if name in globals()
    }
}

with open(f"{BACKUP_DIR}/runtime_inventory.json", "w") as f:
    json.dump(inventory, f, indent=2)

print("Backup inventory saved to:", BACKUP_DIR)
print("Functions recorded:", len(inventory["functions"]))
print("Important variables:", inventory["important_variables"])
print("Next: download the backup folder or copy it to Google Drive.")


Backup inventory saved to: /content/recallguard_backup
Functions recorded: 31
Important variables: {'model': "<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>", 'tokenizer': "<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>", 'device': "<class 'torch.device'>", 'summary': "<class 'pandas.core.frame.DataFrame'>"}
Next: download the backup folder or copy it to Google Drive.


In [16]:

from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

destination = "/content/drive/MyDrive/RecallGuard-KV-backup"
os.makedirs(destination, exist_ok=True)

shutil.copytree(
    "/content/recallguard_backup",
    destination,
    dirs_exist_ok=True
)

print("Backup copied to:", destination)


MessageError: Error: credential propagation was unsuccessful

In [17]:
# RecallGuard-KV — Step 1 Environment Recovery Check
import sys
import importlib.metadata as md
import torch

print("=" * 48)
print("RECALLGUARD-KV | STEP 1 DIAGNOSTIC")
print("=" * 48)

# 1. Python and package versions
print("\n[1] ENVIRONMENT")
print("Python:", sys.version.split()[0])

for package in ["torch", "transformers", "accelerate"]:
    try:
        print(f"{package}: {md.version(package)}")
    except Exception:
        print(f"{package}: NOT INSTALLED")

# 2. GPU availability
print("\n[2] GPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch CUDA:", torch.version.cuda)
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 3),
        "GiB"
    )
else:
    print("GPU: NOT AVAILABLE")

# 3. Model and tokenizer currently loaded
print("\n[3] MODEL STATE")

for name in ["model", "tokenizer"]:
    obj = globals().get(name)
    print(f"{name}:", "LOADED" if obj is not None else "MISSING")

if globals().get("model") is not None:
    try:
        print("Model class:", type(model).__name__)
        print("Model device:", next(model.parameters()).device)
        print("Model layers:", getattr(model.config, "num_hidden_layers", "unknown"))
    except Exception as e:
        print("Model inspection error:", repr(e))

# 4. Step 1 backup artifacts
print("\n[4] BACKUP FILES")
from pathlib import Path

for path in [
    "/content/recallguard_backup",
    "/content/recallguard_results_backup.csv",
    "/content/recallguard_kv_all_results.csv",
    "/content/recallguard_kv_experiment_summary.csv",
    "/content/recallguard_kv_summary.json",
]:
    p = Path(path)
    print(f"{path}: {'EXISTS' if p.exists() else 'NOT FOUND'}")

print("\n" + "=" * 48)
print("DIAGNOSTIC COMPLETE — no experiments were run")
print("=" * 48)

RECALLGUARD-KV | STEP 1 DIAGNOSTIC

[1] ENVIRONMENT
Python: 3.13.15
torch: 2.11.0+cu130
transformers: 5.18.0
accelerate: 1.15.0

[2] GPU
CUDA available: True
GPU: Tesla T4
PyTorch CUDA: 13.0
GPU memory allocated: 0.937 GiB

[3] MODEL STATE
model: LOADED
tokenizer: LOADED
Model class: Qwen2ForCausalLM
Model device: cuda:0
Model layers: 24

[4] BACKUP FILES
/content/recallguard_backup: EXISTS
/content/recallguard_results_backup.csv: NOT FOUND
/content/recallguard_kv_all_results.csv: NOT FOUND
/content/recallguard_kv_experiment_summary.csv: NOT FOUND
/content/recallguard_kv_summary.json: NOT FOUND

DIAGNOSTIC COMPLETE — no experiments were run


In [18]:
from pathlib import Path

print("=== BACKUP DIRECTORY ===")
backup = Path("/content/recallguard_backup")

if backup.exists():
    for p in sorted(backup.rglob("*")):
        if p.is_file():
            print(f"FILE: {p} ({p.stat().st_size:,} bytes)")
        elif p.is_dir():
            print(f"DIR:  {p}")
else:
    print("Backup directory missing!")

print("\n=== GOOGLE DRIVE ===")
drive = Path("/content/drive/MyDrive")
print("Drive mounted:", drive.exists())

if drive.exists():
    target = drive / "RecallGuard-KV-backup"
    print("RecallGuard Drive backup exists:", target.exists())
    if target.exists():
        for p in sorted(target.rglob("*")):
            if p.is_file():
                print(f"DRIVE FILE: {p} ({p.stat().st_size:,} bytes)")

print("\n=== CURRENT NOTEBOOK FILES ===")
for p in sorted(Path("/content").glob("*.ipynb")):
    print(p)

=== BACKUP DIRECTORY ===
FILE: /content/recallguard_backup/environment.txt (15,911 bytes)
FILE: /content/recallguard_backup/runtime_inventory.json (987 bytes)

=== GOOGLE DRIVE ===
Drive mounted: False

=== CURRENT NOTEBOOK FILES ===


In [19]:
from pathlib import Path
import json

print("=== FUNCTIONS CURRENTLY LOADED ===")
required = [
    "_select_k_non_overlapping_spans_dp",
    "select_snap_span_indices",
    "_compress_by_indices",
    "build_context_at_depth",
    "eval_case",
]

for name in required:
    print(f"{name}: {'READY' if callable(globals().get(name)) else 'MISSING'}")

print("\n=== NOTEBOOKS IN RUNTIME ===")
notebooks = list(Path("/content").glob("*.ipynb"))

if notebooks:
    for nb in notebooks:
        print(f"\nNotebook: {nb}")
        try:
            data = json.loads(nb.read_text(encoding="utf-8"))
            for i, cell in enumerate(data.get("cells", [])):
                source = "".join(cell.get("source", []))
                hits = [name for name in required if f"def {name}(" in source]
                if hits:
                    print(f"Cell {i}: {', '.join(hits)}")
        except Exception as e:
            print("Could not inspect notebook:", repr(e))
else:
    print("No .ipynb file found in /content.")
    print("Your notebook may still be saved in Colab or in your ZIP.")

=== FUNCTIONS CURRENTLY LOADED ===
_select_k_non_overlapping_spans_dp: READY
select_snap_span_indices: READY
_compress_by_indices: READY
build_context_at_depth: READY
eval_case: READY

=== NOTEBOOKS IN RUNTIME ===
No .ipynb file found in /content.
Your notebook may still be saved in Colab or in your ZIP.


In [20]:
import inspect
import torch

required = [
    "_select_k_non_overlapping_spans_dp",
    "select_snap_span_indices",
    "_compress_by_indices",
    "build_context_at_depth",
    "eval_case",
]

print("=== STEP 2 VERIFICATION ===")

for name in required:
    fn = globals().get(name)
    if callable(fn):
        print(f"\n{name}")
        print("Signature:", inspect.signature(fn))
    else:
        print(f"\n{name}: MISSING")

print("\n=== MODEL CHECK ===")
print("Model loaded:", globals().get("model") is not None)
print("Tokenizer loaded:", globals().get("tokenizer") is not None)
print("CUDA available:", torch.cuda.is_available())

if globals().get("model") is not None:
    print("Model device:", next(model.parameters()).device)
    print("Model layers:", model.config.num_hidden_layers)

print("\nVerification complete. No experiment was run.")

=== STEP 2 VERIFICATION ===

_select_k_non_overlapping_spans_dp
Signature: (span_scores, k_spans, span_size)

select_snap_span_indices
Signature: (cache, model, q_ids, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16, pool=5)

_compress_by_indices
Signature: (cache, idx)

build_context_at_depth
Signature: (tokenizer, ident, target, n_tokens=240)

eval_case
Signature: (model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16)

=== MODEL CHECK ===
Model loaded: True
Tokenizer loaded: True
CUDA available: True
Model device: cuda:0
Model layers: 24

Verification complete. No experiment was run.


In [21]:
# RecallGuard-KV — Step 3A: Small smoke test
import torch
import traceback

try:
    test_ident = "ORCHID-7319"
    test_target = (
        "The assigned project identifier is ORCHID-7319. "
        "When asked for the assigned project identifier, return the identifier."
    )
    test_question = "What is the assigned project identifier?"

    # Build a small synthetic context
    test_ctx = build_context_at_depth(
        tokenizer,
        test_ident,
        test_target,
        n_tokens=240
    )

    print("Context tokens:", len(tokenizer.encode(test_ctx)))
    print("Context created:", bool(test_ctx))

    # Run the uncompressed baseline first
    baseline = eval_case(
        model=model,
        tokenizer=tokenizer,
        device="cuda",
        ctx=test_ctx,
        question=test_question,
        ident=test_ident,
        policy_fn=None,
        mode="absolute",
        n_new=16
    )

    print("\n=== BASELINE RESULT ===")
    print(baseline)

except Exception:
    print("\nSMOKE TEST FAILED")
    traceback.print_exc()


SMOKE TEST FAILED


Traceback (most recent call last):
  File "/tmp/ipykernel_649/2041763229.py", line 14, in <cell line: 0>
    test_ctx = build_context_at_depth(
        tokenizer,
    ...<2 lines>...
        n_tokens=240
    )
  File "/tmp/ipykernel_649/1573232962.py", line 146, in build_context_at_depth
    key = (abs(depth - target), abs(L - n_tokens))
               ~~~~~~^~~~~~~~
TypeError: unsupported operand type(s) for -: 'float' and 'str'


In [22]:
import inspect

print("=== build_context_at_depth source ===")
print(inspect.getsource(build_context_at_depth))

=== build_context_at_depth source ===
def build_context_at_depth(tokenizer, ident, target, n_tokens=240):
    fact = f"IMPORTANT RECORD: The assigned project identifier is {ident}."
    ft = len(tokenizer(FILLER).input_ids)
    best = None
    for a in range(0, n_tokens // ft + 1):
        for b in range(0, 12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((n_tokens - used) / ft))
            ctx = before + fact + " " + FILLER * c
            toks, L = id_token_span(tokenizer, ctx, ident)
            depth = (toks[0] + toks[-1]) / 2 / L
            key = (abs(depth - target), abs(L - n_tokens))
            if best is None or key < best[0]:
                best = (key, ctx, depth, L)
    return best[1], best[2], best[3]



In [23]:
import traceback

try:
    test_ident = "ORCHID-7319"
    test_question = "What is the assigned project identifier?"

    # Target depth 0.5 means the middle of the context
    test_ctx, actual_depth, actual_tokens = build_context_at_depth(
        tokenizer,
        test_ident,
        target=0.5,
        n_tokens=240
    )

    print("Context tokens:", actual_tokens)
    print("Actual fact depth:", round(actual_depth, 3))
    print("Context generated:", bool(test_ctx))

    baseline = eval_case(
        model=model,
        tokenizer=tokenizer,
        device="cuda",
        ctx=test_ctx,
        question=test_question,
        ident=test_ident,
        policy_fn=None,
        mode="absolute",
        n_new=16
    )

    print("\n=== BASELINE RESULT ===")
    print(baseline)

except Exception:
    print("\nSMOKE TEST FAILED")
    traceback.print_exc()

Context tokens: 233
Actual fact depth: 0.5
Context generated: True

=== BASELINE RESULT ===
{'L': 233, 'depth': 0.5, 'id_kept': 1.0, 'exact': True, 'answer': ' ORCHID-7319. The quarterly summary notes that routine operations', 'kv_mib_pre': 2.73046875, 'n_pos_pre': 233, 'kv_mib': 3.05859375, 'prefill_s': 0.05906537000009848, 'selector_s': 5.912100095883943e-05, 'compress_s': 0.008594504999564379, 'continue_s': 0.8555044949989679, 'peak_delta_mib': 71.140625, 'idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127,

In [24]:
import traceback

try:
    snap_result = eval_case(
        model=model,
        tokenizer=tokenizer,
        device="cuda",
        ctx=test_ctx,
        question=test_question,
        ident=test_ident,
        policy_fn=select_snap_span_indices,
        mode="absolute",
        n_new=16
    )

    print("=== SNAP SMOKE TEST ===")
    for key, value in snap_result.items():
        if key != "idx" and key != "id_toks":
            print(f"{key}: {value}")

    print("\n=== COMPARISON ===")
    print("Baseline retrieved identifier:", baseline["exact"])
    print("SNAP retrieved identifier:", snap_result["exact"])
    print("Baseline KV MiB:", baseline["kv_mib"])
    print("SNAP KV MiB:", snap_result["kv_mib"])

except Exception:
    print("SNAP SMOKE TEST FAILED")
    traceback.print_exc()

SNAP SMOKE TEST FAILED


Traceback (most recent call last):
  File "/tmp/ipykernel_649/164814913.py", line 4, in <cell line: 0>
    snap_result = eval_case(
        model=model,
    ...<7 lines>...
        n_new=16
    )
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/tmp/ipykernel_649/1573232962.py", line 190, in eval_case
    idx = list(range(L)) if policy_fn is None else policy_fn(cache)
                                                   ~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
TypeError: select_snap_span_indices() missing 2 required positional arguments: 'model' and 'q_ids'


In [25]:
import functools
import inspect

print("eval_case policy call:")
source = inspect.getsource(eval_case)
for line in source.splitlines():
    if "policy_fn" in line or "q_ids" in line:
        print(line)

print("\nSelector signature:")
print(inspect.signature(select_snap_span_indices))

eval_case policy call:
def eval_case(model, tokenizer, device, ctx, question, ident, policy_fn, mode, n_new=16):
    q_ids = tokenizer("\nQuestion: " + question + "\nAnswer:", add_special_tokens=False,
    idx = list(range(L)) if policy_fn is None else policy_fn(cache)
    _, ans = _continue(model, tokenizer, work, L, q_ids, mode, n_new=n_new)

Selector signature:
(cache, model, q_ids, start_keep=8, num_mid_spans=3, span_size=8, recent_keep=16, pool=5)


In [26]:
# Adapter: makes SNAP compatible with eval_case's policy_fn(cache) interface
def snap_policy_adapter(cache):
    q_ids = tokenizer(
        "\nQuestion: " + test_question + "\nAnswer:",
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to("cuda")

    return select_snap_span_indices(
        cache,
        model,
        q_ids,
        start_keep=8,
        num_mid_spans=3,
        span_size=8,
        recent_keep=16,
        pool=5
    )

print("SNAP adapter ready:", callable(snap_policy_adapter))

SNAP adapter ready: True


In [27]:
import traceback

try:
    snap_result = eval_case(
        model=model,
        tokenizer=tokenizer,
        device="cuda",
        ctx=test_ctx,
        question=test_question,
        ident=test_ident,
        policy_fn=snap_policy_adapter,
        mode="absolute",
        n_new=16
    )

    print("=== SNAP SMOKE TEST ===")
    for key, value in snap_result.items():
        if key not in ("idx", "id_toks"):
            print(f"{key}: {value}")

    print("\n=== COMPARISON ===")
    print("Baseline identifier found:", baseline["exact"])
    print("SNAP identifier found:", snap_result["exact"])
    print("Baseline KV MiB:", baseline["kv_mib"])
    print("SNAP KV MiB:", snap_result["kv_mib"])

except Exception:
    print("SNAP SMOKE TEST FAILED")
    traceback.print_exc()

=== SNAP SMOKE TEST ===
L: 233
depth: 0.5
id_kept: 1.0
exact: True
answer:  ORCHID-7319.  Question: What is the project
kv_mib_pre: 0.5625
n_pos_pre: 48
kv_mib: 0.890625
prefill_s: 0.05583997400026419
selector_s: 0.05481521699948644
compress_s: 0.008121677999952226
continue_s: 0.8543619299998682
peak_delta_mib: 71.140625

=== COMPARISON ===
Baseline identifier found: True
SNAP identifier found: True
Baseline KV MiB: 3.05859375
SNAP KV MiB: 0.890625


In [28]:
import shutil, os
from google.colab import files
d = globals().get("RESULTS_DIR", "/content/recallguard_kv")
print(os.listdir(d))                      # check final_fresh_seeds.csv is in the list
shutil.make_archive("/content/recallguard_results", "zip", d)
files.download("/content/recallguard_results.zip")

['hard_20case_baselines.csv', 'h2o_and_snap_tok.csv', 'heldout_varied_v2.csv', 'length_latency_memory.csv', 'final_fresh_seeds.csv', 'heldout_varied.csv', 'budget_sweep.csv', 'snap_variants_multiq.csv', 'multi_question.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import json
from pathlib import Path

backup = Path("/content/recallguard_backup")
backup.mkdir(parents=True, exist_ok=True)

smoke_summary = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "gpu": torch.cuda.get_device_name(0),
    "context_tokens": snap_result["L"],
    "fact_depth": snap_result["depth"],
    "baseline_exact": baseline["exact"],
    "snap_exact": snap_result["exact"],
    "baseline_kv_mib": baseline["kv_mib"],
    "snap_kv_mib": snap_result["kv_mib"],
    "snap_selector_seconds": snap_result["selector_s"],
    "snap_positions_before_continuation": snap_result["n_pos_pre"],
    "snap_identifier_retained": snap_result["id_kept"],
}

out = backup / "smoke_test_summary.json"
out.write_text(json.dumps(smoke_summary, indent=2))

print("Saved:", out)
print(out.read_text())

Saved: /content/recallguard_backup/smoke_test_summary.json
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "gpu": "Tesla T4",
  "context_tokens": 233,
  "fact_depth": 0.5,
  "baseline_exact": true,
  "snap_exact": true,
  "baseline_kv_mib": 3.05859375,
  "snap_kv_mib": 0.890625,
  "snap_selector_seconds": 0.05481521699948644,
  "snap_positions_before_continuation": 48,
  "snap_identifier_retained": 1.0
}


In [30]:
# RecallGuard-KV | Step 4A: Main long-context tests
import time
import json
import traceback
import pandas as pd
from pathlib import Path

backup = Path("/content/recallguard_backup")
backup.mkdir(parents=True, exist_ok=True)
csv_path = backup / "main_long_context_results.csv"

test_question = "What is the assigned project identifier?"

# Five positions across a long context
cases = [
    {"case": "Start",     "depth": 0.05, "ident": "ORCHID-7319"},
    {"case": "Early-Mid", "depth": 0.25, "ident": "COBALT-4826"},
    {"case": "Middle",    "depth": 0.50, "ident": "MAPLE-9054"},
    {"case": "Late-Mid",  "depth": 0.75, "ident": "AMBER-6173"},
    {"case": "Recent",    "depth": 0.95, "ident": "RAVEN-2841"},
]

def make_snap_policy(question):
    def policy(cache):
        q_ids = tokenizer(
            "\nQuestion: " + question + "\nAnswer:",
            add_special_tokens=False,
            return_tensors="pt"
        ).input_ids.to("cuda")

        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )
    return policy

# Keep any results already saved from an earlier attempt
if csv_path.exists():
    records = pd.read_csv(csv_path).to_dict("records")
    completed = {
        (r["case"], r["method"]) for r in records
    }
    print("Resuming saved results:", len(records), "rows")
else:
    records = []
    completed = set()

for case in cases:
    print(f"\n{'='*55}")
    print(f"CASE: {case['case']} | target depth={case['depth']}")
    print("="*55)

    try:
        ctx, actual_depth, actual_tokens = build_context_at_depth(
            tokenizer,
            case["ident"],
            target=case["depth"],
            n_tokens=3800
        )
        print("Actual tokens:", actual_tokens,
              "| actual depth:", round(actual_depth, 3))

        # Same context and question for both methods
        for method in ["FULL", "SNAP"]:
            if (case["case"], method) in completed:
                print(method, "already saved; skipping")
                continue

            policy = None if method == "FULL" else make_snap_policy(
                test_question
            )

            result = eval_case(
                model=model,
                tokenizer=tokenizer,
                device="cuda",
                ctx=ctx,
                question=test_question,
                ident=case["ident"],
                policy_fn=policy,
                mode="absolute",
                n_new=16
            )

            row = {
                "case": case["case"],
                "target_depth": case["depth"],
                "actual_depth": actual_depth,
                "ident": case["ident"],
                "method": method,
                "L": result["L"],
                "exact": bool(result["exact"]),
                "id_kept": result["id_kept"],
                "kv_mib": result["kv_mib"],
                "kv_mib_pre": result["kv_mib_pre"],
                "n_pos_pre": result["n_pos_pre"],
                "selector_s": result["selector_s"],
                "prefill_s": result["prefill_s"],
                "continue_s": result["continue_s"],
                "answer": result["answer"],
            }

            records.append(row)
            completed.add((case["case"], method))

            # Save after each individual run
            pd.DataFrame(records).to_csv(csv_path, index=False)

            print(
                f"{method}: retrieved={row['exact']}, "
                f"KV={row['kv_mib']:.3f} MiB, "
                f"positions={row['n_pos_pre']}, "
                f"selector={row['selector_s']:.3f}s"
            )

    except Exception:
        print("ERROR in case:", case["case"])
        traceback.print_exc()
        pd.DataFrame(records).to_csv(csv_path, index=False)
        break

print("\n=== SAVED RESULTS ===")
print("File:", csv_path)
print("Completed rows:", len(records))

if records:
    df = pd.DataFrame(records)
    print("\n", df[[
        "case", "method", "L", "actual_depth",
        "exact", "kv_mib", "n_pos_pre", "selector_s"
    ]].to_string(index=False))

    print("\nCSV saved:", csv_path.exists())


CASE: Start | target depth=0.05
Actual tokens: 3504 | actual depth: 0.05
FULL: retrieved=True, KV=41.391 MiB, positions=3504, selector=0.000s
SNAP: retrieved=True, KV=0.891 MiB, positions=48, selector=0.072s

CASE: Early-Mid | target depth=0.25
Actual tokens: 3558 | actual depth: 0.25
FULL: retrieved=True, KV=42.023 MiB, positions=3558, selector=0.000s
SNAP: retrieved=True, KV=0.891 MiB, positions=48, selector=0.046s

CASE: Middle | target depth=0.5
Actual tokens: 3641 | actual depth: 0.5
FULL: retrieved=True, KV=42.996 MiB, positions=3641, selector=0.000s
SNAP: retrieved=True, KV=0.891 MiB, positions=48, selector=0.047s

CASE: Late-Mid | target depth=0.75
Actual tokens: 3720 | actual depth: 0.75
FULL: retrieved=True, KV=43.922 MiB, positions=3720, selector=0.000s
SNAP: retrieved=True, KV=0.891 MiB, positions=48, selector=0.060s

CASE: Recent | target depth=0.95
Actual tokens: 3780 | actual depth: 0.926
FULL: retrieved=True, KV=44.625 MiB, positions=3780, selector=0.000s
SNAP: retriev

In [31]:
# RecallGuard-KV | Step 4B: Multi-task benchmark
import json, time, traceback
from pathlib import Path
import pandas as pd
import torch

backup = Path("/content/recallguard_backup")
backup.mkdir(parents=True, exist_ok=True)
csv_path = backup / "general_benchmark_results.csv"

# Five different synthetic retrieval tasks
tasks = [
    {
        "case": "Project",
        "fact": "The project identifier recorded in the engineering register is ORCHID-7319.",
        "answer": "ORCHID-7319",
        "question": "What project identifier is recorded in the engineering register?",
        "depth": 0.05,
    },
    {
        "case": "Archive",
        "fact": "The archive reference assigned to the lunar sample is COBALT-4826.",
        "answer": "COBALT-4826",
        "question": "What is the archive reference for the lunar sample?",
        "depth": 0.25,
    },
    {
        "case": "Laboratory",
        "fact": "The access code assigned to the materials laboratory is MAPLE-9054.",
        "answer": "MAPLE-9054",
        "question": "What access code is assigned to the materials laboratory?",
        "depth": 0.50,
    },
    {
        "case": "Shipment",
        "fact": "The tracking reference for the research shipment is AMBER-6173.",
        "answer": "AMBER-6173",
        "question": "What is the tracking reference for the research shipment?",
        "depth": 0.75,
    },
    {
        "case": "Satellite",
        "fact": "The telemetry tag registered for the satellite test is RAVEN-2841.",
        "answer": "RAVEN-2841",
        "question": "What telemetry tag was registered for the satellite test?",
        "depth": 0.95,
    },
]

def build_generic_context(fact, answer, target, n_tokens=3800):
    """Construct a synthetic long context with the fact at a target depth."""
    ft = max(1, len(tokenizer(FILLER).input_ids))
    best = None

    for a in range(n_tokens // ft + 1):
        for b in range(12):
            before = FILLER * a + "Note " * b
            used = len(tokenizer(before + fact).input_ids)
            c = max(0, round((n_tokens - used) / ft))
            ctx = before + fact + " " + FILLER * c

            try:
                toks, L = id_token_span(tokenizer, ctx, answer)
                if not toks:
                    continue
                depth = (toks[0] + toks[-1]) / 2 / L
                key = (abs(depth - target), abs(L - n_tokens))
                if best is None or key < best[0]:
                    best = (key, ctx, depth, L)
            except Exception:
                continue

    if best is None:
        raise ValueError(f"Could not locate answer span: {answer}")

    return best[1], best[2], best[3]


def make_question_policy(question):
    def policy(cache):
        q_ids = tokenizer(
            "\nQuestion: " + question + "\nAnswer:",
            add_special_tokens=False,
            return_tensors="pt"
        ).input_ids.to("cuda")

        return select_snap_span_indices(
            cache, model, q_ids,
            start_keep=8,
            num_mid_spans=3,
            span_size=8,
            recent_keep=16,
            pool=5
        )
    return policy


def recent_48_policy(cache):
    L = int(cache.get_seq_length())
    return list(range(max(0, L - 48), L))


def streaming_48_policy(cache):
    L = int(cache.get_seq_length())
    if L <= 48:
        return list(range(L))
    return list(range(32)) + list(range(L - 16, L))


policies = {
    "FULL": None,
    "Recent-48": recent_48_policy,
    "Streaming-48": streaming_48_policy,
    # SNAP is constructed separately per task because it uses the question.
    "SNAP": "snap",
}

if csv_path.exists():
    records = pd.read_csv(csv_path).to_dict("records")
    completed = {(r["case"], r["method"]) for r in records}
    print("Resuming:", len(records), "saved rows")
else:
    records, completed = [], set()

for task in tasks:
    print(f"\n{'='*60}\n{task['case']} | target depth {task['depth']}\n{'='*60}")

    try:
        ctx, actual_depth, L = build_generic_context(
            task["fact"], task["answer"], task["depth"], n_tokens=3800
        )
        print("Tokens:", L, "| actual depth:", round(actual_depth, 3))

        for method, policy_def in policies.items():
            if (task["case"], method) in completed:
                print(method, "already saved; skipping")
                continue

            policy = (
                None if policy_def is None else
                make_question_policy(task["question"]) if policy_def == "snap"
                else policy_def
            )

            result = eval_case(
                model=model,
                tokenizer=tokenizer,
                device="cuda",
                ctx=ctx,
                question=task["question"],
                ident=task["answer"],
                policy_fn=policy,
                mode="absolute",
                n_new=16
            )

            row = {
                "case": task["case"],
                "question": task["question"],
                "answer_expected": task["answer"],
                "target_depth": task["depth"],
                "actual_depth": actual_depth,
                "method": method,
                "L": result["L"],
                "retrieval_success": bool(result["exact"]),
                "answer_span_retained": result["id_kept"],
                "kv_mib": result["kv_mib"],
                "kv_mib_pre": result["kv_mib_pre"],
                "positions_retained": result["n_pos_pre"],
                "selector_s": result["selector_s"],
                "prefill_s": result["prefill_s"],
                "continue_s": result["continue_s"],
                "generated_answer": result["answer"],
            }

            records.append(row)
            completed.add((task["case"], method))
            pd.DataFrame(records).to_csv(csv_path, index=False)

            print(
                f"{method}: success={row['retrieval_success']}, "
                f"KV={row['kv_mib']:.3f} MiB, "
                f"positions={row['positions_retained']}, "
                f"selector={row['selector_s']:.3f}s"
            )

    except Exception:
        print("FAILED:", task["case"])
        traceback.print_exc()
        pd.DataFrame(records).to_csv(csv_path, index=False)
        break

print("\n=== BENCHMARK SUMMARY ===")
if records:
    df = pd.DataFrame(records)
    print(df.groupby("method").agg(
        cases=("case", "count"),
        successes=("retrieval_success", "sum"),
        mean_kv_mib=("kv_mib", "mean"),
        mean_positions=("positions_retained", "mean"),
        mean_selector_s=("selector_s", "mean"),
    ).to_string())
    print("\nSaved:", csv_path)
    print("Rows:", len(df))
else:
    print("No results recorded.")


Project | target depth 0.05
Tokens: 3504 | actual depth: 0.05
FULL: success=True, KV=41.426 MiB, positions=3504, selector=0.000s
Recent-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
Streaming-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
SNAP: success=True, KV=0.926 MiB, positions=48, selector=0.046s

Archive | target depth 0.25
Tokens: 3558 | actual depth: 0.25
FULL: success=True, KV=42.059 MiB, positions=3558, selector=0.000s
Recent-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
Streaming-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
SNAP: success=True, KV=0.926 MiB, positions=48, selector=0.046s

Laboratory | target depth 0.5
Tokens: 3641 | actual depth: 0.5
FULL: success=True, KV=43.031 MiB, positions=3641, selector=0.000s
Recent-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
Streaming-48: success=False, KV=0.926 MiB, positions=48, selector=0.000s
SNAP: success=False, KV=0.926 MiB, positions=48, sel

In [32]:

from pathlib import Path
import zipfile
from google.colab import files

# Collect saved benchmark results from the current Colab runtime
root = Path("/content")
backup = root / "recallguard_backup"
output_zip = root / "RecallGuard-KV-benchmark-results.zip"

# Find CSV and JSON results, excluding the ZIP we are creating
result_files = [
    p for p in root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".csv", ".json"}
    and "RecallGuard-KV-benchmark-results.zip" not in p.name
    and "site-packages" not in str(p)
]

print("Found result files:")
for p in result_files:
    print(" -", p)

if not result_files:
    print("\nNo CSV/JSON results found. Check your saved paths.")
else:
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for p in result_files:
            z.write(p, arcname=str(p.relative_to(root)))

    print(f"\nCreated: {output_zip}")
    print(f"Files included: {len(result_files)}")
    files.download(str(output_zip))


Found result files:
 - /content/.config/.last_update_check.json
 - /content/recallguard_kv/hard_20case_baselines.csv
 - /content/recallguard_kv/h2o_and_snap_tok.csv
 - /content/recallguard_kv/heldout_varied_v2.csv
 - /content/recallguard_kv/length_latency_memory.csv
 - /content/recallguard_kv/final_fresh_seeds.csv
 - /content/recallguard_kv/heldout_varied.csv
 - /content/recallguard_kv/budget_sweep.csv
 - /content/recallguard_kv/snap_variants_multiq.csv
 - /content/recallguard_kv/multi_question.csv
 - /content/recallguard_backup/main_long_context_results.csv
 - /content/recallguard_backup/general_benchmark_results.csv
 - /content/recallguard_backup/smoke_test_summary.json
 - /content/recallguard_backup/runtime_inventory.json
 - /content/sample_data/anscombe.json
 - /content/sample_data/mnist_train_small.csv
 - /content/sample_data/california_housing_test.csv
 - /content/sample_data/california_housing_train.csv
 - /content/sample_data/mnist_test.csv

Created: /content/RecallGuard-KV-ben

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>